## I am going to start modelling  eden dataset

In [ ]:
# ==============================================================
# MODELLING STEP 1, PART 1
# Controlled loading and structural validation
# ==============================================================

from IPython import display
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------
# 1. Define the existing Eden project folders
# --------------------------------------------------------------

PROJECT_ROOT = Path("/Users/ryansmac/Desktop/Meng Project")
EDEN_DATASETS_DIR = PROJECT_ROOT / "eden_datasets"

# Keep the existing folder spelling used during forecast preparation.
FORECAST_PREPARATION_DIR = EDEN_DATASETS_DIR / "forceast_preparation"

# New folder for the modelling phase.
MODELLING_DIR = EDEN_DATASETS_DIR / "modelling"


# --------------------------------------------------------------
# 2. Create the modelling folder structure
# --------------------------------------------------------------

MODELLING_SUBFOLDERS = [
    "01_pipeline_setup",
    "02_baselines",
    "03_model_selection",
    "04_final_test",
    "05_final_forecasts",
    "models",
    "results",
    "plots",
]

for folder_name in MODELLING_SUBFOLDERS:
    folder_path = MODELLING_DIR / folder_name
    folder_path.mkdir(parents=True, exist_ok=True)

SETUP_OUTPUT_DIR = MODELLING_DIR / "01_pipeline_setup"

print("Modelling folder created or confirmed:")
print(MODELLING_DIR)


# --------------------------------------------------------------
# 3. Define the files allowed during model-selection setup
# --------------------------------------------------------------

MODEL_SELECTION_FILES = {
    "MODEL_SELECTION_TRAINING":
        FORECAST_PREPARATION_DIR /
        "17_model_selection_training_dataset.csv",

    "MODEL_SELECTION_VALIDATION":
        FORECAST_PREPARATION_DIR /
        "17_model_selection_validation_dataset.csv",

    "FROZEN_FEATURE_CONTRACT":
        FORECAST_PREPARATION_DIR /
        "17_frozen_model_feature_contract.csv",

    "FORECASTING_PROTOCOL":
        FORECAST_PREPARATION_DIR /
        "17_forecasting_protocol_contract.csv",
}


# --------------------------------------------------------------
# 4. Register the protected final-test files
#
# These paths are recorded only so that their protected status can
# be documented. Their contents are NOT loaded in this notebook part.
# --------------------------------------------------------------

PROTECTED_FINAL_TEST_FILES = {
    "RESERVED_FINAL_TEST_TRAINING":
        FORECAST_PREPARATION_DIR /
        "17_reserved_final_test_training_dataset.csv",

    "RESERVED_FINAL_TEST_SCORING_FEATURES":
        FORECAST_PREPARATION_DIR /
        "17_reserved_final_test_scoring_features.csv",

    "RESERVED_FINAL_TEST_TARGET_VAULT":
        FORECAST_PREPARATION_DIR /
        "17_reserved_final_test_target_vault.csv",
}


# --------------------------------------------------------------
# 5. Confirm that all required files exist
# --------------------------------------------------------------

all_registered_files = {
    **MODEL_SELECTION_FILES,
    **PROTECTED_FINAL_TEST_FILES,
}

missing_files = [
    str(file_path)
    for file_path in all_registered_files.values()
    if not file_path.exists()
]

if missing_files:
    missing_text = "\n".join(missing_files)

    raise FileNotFoundError(
        "The following required files were not found:\n"
        f"{missing_text}\n\n"
        "Check PROJECT_ROOT and FORECAST_PREPARATION_DIR."
    )

print("\nAll required files were found.")


# --------------------------------------------------------------
# 6. Load only the permitted model-selection files
# --------------------------------------------------------------

training_df = pd.read_csv(
    MODEL_SELECTION_FILES["MODEL_SELECTION_TRAINING"],
    low_memory=False,
)

validation_df = pd.read_csv(
    MODEL_SELECTION_FILES["MODEL_SELECTION_VALIDATION"],
    low_memory=False,
)

feature_contract_df = pd.read_csv(
    MODEL_SELECTION_FILES["FROZEN_FEATURE_CONTRACT"],
    low_memory=False,
)

protocol_contract_df = pd.read_csv(
    MODEL_SELECTION_FILES["FORECASTING_PROTOCOL"],
    low_memory=False,
)


# --------------------------------------------------------------
# 7. Parse the actual date columns
# --------------------------------------------------------------

DATE_COLUMNS = [
    "Date",
    "ProductFirstObservedDate",
]

for dataset_name, dataset in {
    "training": training_df,
    "validation": validation_df,
}.items():

    for column in DATE_COLUMNS:

        if column not in dataset.columns:
            raise KeyError(
                f"{column!r} is missing from the {dataset_name} dataset."
            )

        dataset[column] = pd.to_datetime(
            dataset[column],
            errors="raise",
        )


# --------------------------------------------------------------
# 8. Robustly standardise boolean contract columns
# --------------------------------------------------------------

def convert_to_boolean(series: pd.Series) -> pd.Series:
    """
    Convert a boolean-like pandas Series into true boolean values.
    Raises an error if unexpected values are found.
    """

    if pd.api.types.is_bool_dtype(series):
        return series

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    boolean_mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    converted = cleaned.map(boolean_mapping)

    if converted.isna().any():
        unexpected_values = (
            series[converted.isna()]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unexpected boolean values found: "
            f"{unexpected_values}"
        )

    return converted.astype(bool)


CONTRACT_BOOLEAN_COLUMNS = [
    "DirectModelInputAllowed",
    "IsHistoricalDemandFeature",
    "IsIdentifier",
    "IsSupportColumn",
    "IsForecastTarget",
    "FrozenForModelTraining",
]

for column in CONTRACT_BOOLEAN_COLUMNS:

    if column not in feature_contract_df.columns:
        raise KeyError(
            f"{column!r} is missing from the feature contract."
        )

    feature_contract_df[column] = convert_to_boolean(
        feature_contract_df[column]
    )


# --------------------------------------------------------------
# 9. Read the predictor and target definitions from the contract
# --------------------------------------------------------------

approved_predictors = feature_contract_df.loc[
    feature_contract_df["DirectModelInputAllowed"],
    "Column",
].tolist()

historical_predictors = feature_contract_df.loc[
    feature_contract_df["DirectModelInputAllowed"]
    & feature_contract_df["IsHistoricalDemandFeature"],
    "Column",
].tolist()

current_date_predictors = feature_contract_df.loc[
    feature_contract_df["DirectModelInputAllowed"]
    & ~feature_contract_df["IsHistoricalDemandFeature"],
    "Column",
].tolist()

forecast_targets = feature_contract_df.loc[
    feature_contract_df["IsForecastTarget"],
    "Column",
].tolist()

protocol_mapping = dict(
    zip(
        protocol_contract_df["ProtocolParameter"],
        protocol_contract_df["Value"].astype(str),
    )
)


# --------------------------------------------------------------
# 10. Run the structural validation checks
# --------------------------------------------------------------

validation_checks = []


def add_validation_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one validation result to the audit table.
    """

    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


# Dataset dimensions
add_validation_check(
    "Training rows",
    len(training_df),
    48_538,
)

add_validation_check(
    "Training columns",
    training_df.shape[1],
    72,
)

add_validation_check(
    "Validation rows",
    len(validation_df),
    5_080,
)

add_validation_check(
    "Validation columns",
    validation_df.shape[1],
    72,
)


# Product and context coverage
add_validation_check(
    "Training products",
    training_df["CanonicalProductID"].nunique(),
    127,
)

add_validation_check(
    "Validation products",
    validation_df["CanonicalProductID"].nunique(),
    127,
)

add_validation_check(
    "Training split contexts",
    training_df["SplitContextID"].nunique(),
    254,
)

add_validation_check(
    "Validation split contexts",
    validation_df["SplitContextID"].nunique(),
    254,
)


# Column parity
add_validation_check(
    "Training and validation columns are identical",
    training_df.columns.tolist(),
    validation_df.columns.tolist(),
    passed=(
        training_df.columns.tolist()
        == validation_df.columns.tolist()
    ),
)


# Split roles
add_validation_check(
    "Training split role",
    sorted(training_df["SplitRole"].dropna().unique().tolist()),
    ["TRAIN"],
)

add_validation_check(
    "Validation split role",
    sorted(validation_df["SplitRole"].dropna().unique().tolist()),
    ["VALIDATION"],
)


# Validation folds
expected_validation_windows = [
    "STANDARD_BACKTEST_FOLD_1",
    "STANDARD_BACKTEST_FOLD_2",
]

add_validation_check(
    "Validation windows",
    sorted(
        validation_df["WindowID"]
        .dropna()
        .unique()
        .tolist()
    ),
    expected_validation_windows,
)


# Duplicate checks within modelling contexts
training_duplicate_count = training_df.duplicated(
    subset=[
        "SplitContextID",
        "CanonicalProductID",
        "Date",
    ]
).sum()

validation_duplicate_count = validation_df.duplicated(
    subset=[
        "SplitContextID",
        "CanonicalProductID",
        "Date",
    ]
).sum()

add_validation_check(
    "Training duplicate context-product-date rows",
    int(training_duplicate_count),
    0,
)

add_validation_check(
    "Validation duplicate context-product-date rows",
    int(validation_duplicate_count),
    0,
)


# Target checks
add_validation_check(
    "Missing training target values",
    int(training_df["TotalDemand"].isna().sum()),
    0,
)

add_validation_check(
    "Missing validation target values",
    int(validation_df["TotalDemand"].isna().sum()),
    0,
)


# Frozen feature-contract checks
add_validation_check(
    "Frozen direct predictors",
    len(approved_predictors),
    53,
)

add_validation_check(
    "Frozen historical predictors",
    len(historical_predictors),
    31,
)

add_validation_check(
    "Frozen current-date predictors",
    len(current_date_predictors),
    22,
)

add_validation_check(
    "Frozen forecast target",
    forecast_targets,
    ["TotalDemand"],
)


# Forecasting-protocol checks
add_validation_check(
    "Protocol forecast target",
    protocol_mapping.get("ForecastTarget"),
    "TotalDemand",
)

add_validation_check(
    "Protocol forecast mode",
    protocol_mapping.get("OperationalForecastMode"),
    "ROLLING_ONE_OPERATING_DAY_AHEAD",
)

add_validation_check(
    "Random splitting prohibited",
    protocol_mapping.get("RandomSplittingAllowed"),
    "False",
)

add_validation_check(
    "Final test prohibited during model selection",
    protocol_mapping.get(
        "FinalTestMayBeUsedForModelSelection"
    ),
    "False",
)


# Convert the checks into a DataFrame
entry_validation_df = pd.DataFrame(validation_checks)


# --------------------------------------------------------------
# 11. Stop immediately if any validation check fails
# --------------------------------------------------------------

failed_checks_df = entry_validation_df.loc[
    ~entry_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print("\nFAILED VALIDATION CHECKS")
    print(failed_checks_df.to_string(index=False))

    raise AssertionError(
        "Step 1 Part 1 validation failed. "
        "Do not continue to preprocessing or modelling."
    )


# --------------------------------------------------------------
# 12. Create the dataset-loading summary
# --------------------------------------------------------------

loading_summary_df = pd.DataFrame(
    [
        {
            "Dataset": "MODEL_SELECTION_TRAINING",
            "Rows": len(training_df),
            "Columns": training_df.shape[1],
            "Products": training_df[
                "CanonicalProductID"
            ].nunique(),
            "SplitContexts": training_df[
                "SplitContextID"
            ].nunique(),
            "MinimumDate": training_df["Date"].min().date(),
            "MaximumDate": training_df["Date"].max().date(),
            "Loaded": True,
        },
        {
            "Dataset": "MODEL_SELECTION_VALIDATION",
            "Rows": len(validation_df),
            "Columns": validation_df.shape[1],
            "Products": validation_df[
                "CanonicalProductID"
            ].nunique(),
            "SplitContexts": validation_df[
                "SplitContextID"
            ].nunique(),
            "MinimumDate": validation_df["Date"].min().date(),
            "MaximumDate": validation_df["Date"].max().date(),
            "Loaded": True,
        },
        {
            "Dataset": "FROZEN_FEATURE_CONTRACT",
            "Rows": len(feature_contract_df),
            "Columns": feature_contract_df.shape[1],
            "Products": np.nan,
            "SplitContexts": np.nan,
            "MinimumDate": pd.NaT,
            "MaximumDate": pd.NaT,
            "Loaded": True,
        },
        {
            "Dataset": "FORECASTING_PROTOCOL",
            "Rows": len(protocol_contract_df),
            "Columns": protocol_contract_df.shape[1],
            "Products": np.nan,
            "SplitContexts": np.nan,
            "MinimumDate": pd.NaT,
            "MaximumDate": pd.NaT,
            "Loaded": True,
        },
    ]
)


# --------------------------------------------------------------
# 13. Create an access-control register
# --------------------------------------------------------------

input_access_records = []

for dataset_name, file_path in MODEL_SELECTION_FILES.items():
    input_access_records.append(
        {
            "Dataset": dataset_name,
            "FileName": file_path.name,
            "LoadedInStep1Part1": True,
            "AccessPolicy":
                "ALLOWED_FOR_MODEL_SELECTION_SETUP",
        }
    )

for dataset_name, file_path in PROTECTED_FINAL_TEST_FILES.items():
    input_access_records.append(
        {
            "Dataset": dataset_name,
            "FileName": file_path.name,
            "LoadedInStep1Part1": False,
            "AccessPolicy":
                "DO_NOT_LOAD_UNTIL_MODEL_SELECTION_IS_LOCKED",
        }
    )

input_access_register_df = pd.DataFrame(
    input_access_records
)


# --------------------------------------------------------------
# 14. Save the Part 1 audit outputs
# --------------------------------------------------------------

loading_summary_path = (
    SETUP_OUTPUT_DIR /
    "01_model_dataset_loading_summary.csv"
)

entry_validation_path = (
    SETUP_OUTPUT_DIR /
    "01_model_entry_validation_summary.csv"
)

input_access_path = (
    SETUP_OUTPUT_DIR /
    "01_model_input_access_register.csv"
)

loading_summary_df.to_csv(
    loading_summary_path,
    index=False,
)

entry_validation_df.to_csv(
    entry_validation_path,
    index=False,
)

input_access_register_df.to_csv(
    input_access_path,
    index=False,
)


# --------------------------------------------------------------
# 15. Display the final Part 1 results
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("MODELLING STEP 1 PART 1: PASSED")
print("=" * 70)

print(
    f"\nTraining dataset: "
    f"{training_df.shape[0]:,} rows × "
    f"{training_df.shape[1]} columns"
)

print(
    f"Validation dataset: "
    f"{validation_df.shape[0]:,} rows × "
    f"{validation_df.shape[1]} columns"
)

print(
    f"Training products: "
    f"{training_df['CanonicalProductID'].nunique()}"
)

print(
    f"Validation products: "
    f"{validation_df['CanonicalProductID'].nunique()}"
)

print(
    f"Split contexts: "
    f"{validation_df['SplitContextID'].nunique()}"
)

print(
    f"Approved predictors: "
    f"{len(approved_predictors)}"
)

print(
    f"Current-date predictors: "
    f"{len(current_date_predictors)}"
)

print(
    f"Historical predictors: "
    f"{len(historical_predictors)}"
)

print(
    f"Forecast target: "
    f"{forecast_targets[0]}"
)

print(
    f"Validation checks passed: "
    f"{entry_validation_df['Passed'].sum()} "
    f"of {len(entry_validation_df)}"
)

print("\nProtected final-test files loaded: 0")

print("\nSaved outputs:")
print(loading_summary_path)
print(entry_validation_path)
print(input_access_path)

print("\nLoading summary:")
display(loading_summary_df)

print("\nValidation summary:")
display(entry_validation_df)

Modelling folder created or confirmed:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling

All required files were found.

MODELLING STEP 1 PART 1: PASSED

Training dataset: 48,538 rows × 72 columns
Validation dataset: 5,080 rows × 72 columns
Training products: 127
Validation products: 127
Split contexts: 254
Approved predictors: 53
Current-date predictors: 22
Historical predictors: 31
Forecast target: TotalDemand
Validation checks passed: 24 of 24

Protected final-test files loaded: 0

Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/01_model_dataset_loading_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/01_model_entry_validation_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/01_model_input_access_register.csv

Loading summary:


,Dataset,Rows,Columns,Products,SplitContexts,MinimumDate,MaximumDate,Loaded
0,MODEL_SELECTION_TRAINING,48538,72,127.0,254.0,2025-04-01,2026-01-29,True
1,MODEL_SELECTION_VALIDATION,5080,72,127.0,254.0,2026-01-05,2026-02-27,True
2,FROZEN_FEATURE_CONTRACT,58,26,NaN,NaN,NaT,NaT,True
3,FORECASTING_PROTOCOL,20,2,NaN,NaN,NaT,NaT,True



Validation summary:


,Check,Expected,Actual,Passed
0,Training rows,48538,48538,True
1,Training columns,72,72,True
2,Validation rows,5080,5080,True
3,Validation columns,72,72,True
4,Training products,127,127,True
5,Validation products,127,127,True
6,Training split contexts,254,254,True
7,Validation split contexts,254,254,True
8,Training and validation columns are identical,"['Date', 'CanonicalProductID', 'OperatingDaySe...","['Date', 'CanonicalProductID', 'OperatingDaySe...",True
9,Training split role,['TRAIN'],['TRAIN'],True


In [3]:
# ==============================================================
# MODELLING STEP 1, PART 2
# Predictor contract, data-type and missingness audit
# ==============================================================

import numpy as np
import pandas as pd
from pandas.api.types import (
    is_bool_dtype,
    is_float_dtype,
    is_integer_dtype,
    is_numeric_dtype,
    is_object_dtype,
    is_string_dtype,
)


# --------------------------------------------------------------
# 1. Confirm that Part 1 variables are available
# --------------------------------------------------------------

required_objects = [
    "training_df",
    "validation_df",
    "feature_contract_df",
    "SETUP_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following Part 1 objects are not available:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 1 Part 1 before running this cell."
    )


# --------------------------------------------------------------
# 2. Robustly convert contract flag columns to boolean
# --------------------------------------------------------------

def convert_contract_flag(series: pd.Series) -> pd.Series:
    """
    Convert contract flag values such as True, False, 1 and 0
    into proper pandas boolean values.
    """

    if is_bool_dtype(series):
        return series.astype(bool)

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    converted = cleaned.map(mapping)

    if converted.isna().any():
        unexpected_values = (
            series.loc[converted.isna()]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unexpected contract flag values found: "
            f"{unexpected_values}"
        )

    return converted.astype(bool)


contract_flag_columns = [
    "DirectModelInputAllowed",
    "IsHistoricalDemandFeature",
    "IsIdentifier",
    "IsSupportColumn",
    "IsForecastTarget",
    "UsesCurrentRowTarget",
    "UsesFutureTarget",
    "FrozenForModelTraining",
]

for column in contract_flag_columns:

    if column not in feature_contract_df.columns:
        raise KeyError(
            f"Required feature-contract column is missing: {column}"
        )

    feature_contract_df[column] = convert_contract_flag(
        feature_contract_df[column]
    )


# --------------------------------------------------------------
# 3. Extract the frozen predictors in their approved order
# --------------------------------------------------------------

approved_contract_df = (
    feature_contract_df.loc[
        feature_contract_df["DirectModelInputAllowed"]
    ]
    .sort_values("ColumnOrder")
    .reset_index(drop=True)
)

approved_predictors = approved_contract_df["Column"].tolist()

historical_predictors = approved_contract_df.loc[
    approved_contract_df["IsHistoricalDemandFeature"],
    "Column",
].tolist()

current_date_predictors = approved_contract_df.loc[
    ~approved_contract_df["IsHistoricalDemandFeature"],
    "Column",
].tolist()

forecast_targets = feature_contract_df.loc[
    feature_contract_df["IsForecastTarget"],
    "Column",
].tolist()

if forecast_targets != ["TotalDemand"]:
    raise AssertionError(
        "The frozen forecast target is not exactly ['TotalDemand']."
    )

TARGET_COLUMN = "TotalDemand"


# --------------------------------------------------------------
# 4. Check predictor presence and order
# --------------------------------------------------------------

missing_from_training = [
    column
    for column in approved_predictors
    if column not in training_df.columns
]

missing_from_validation = [
    column
    for column in approved_predictors
    if column not in validation_df.columns
]

training_predictor_order = [
    column
    for column in training_df.columns
    if column in approved_predictors
]

validation_predictor_order = [
    column
    for column in validation_df.columns
    if column in approved_predictors
]


# --------------------------------------------------------------
# 5. Define data-type compatibility functions
# --------------------------------------------------------------

def raw_type_group(series: pd.Series) -> str:
    """
    Assign a broad raw data-type group.
    This is an audit classification only.
    """

    if is_bool_dtype(series):
        return "BOOLEAN"

    if is_integer_dtype(series):
        return "INTEGER_NUMERIC"

    if is_float_dtype(series):
        return "FLOAT_NUMERIC"

    if is_object_dtype(series) or is_string_dtype(series):
        return "TEXT_OR_CATEGORY"

    return "OTHER"


def dtype_matches_contract(
    series: pd.Series,
    contract_dtype: str,
) -> bool:
    """
    Check whether an actual pandas dtype is compatible with the
    dtype recorded in the frozen feature contract.
    """

    expected = str(contract_dtype).strip().lower()

    if expected.startswith("int"):
        return bool(is_integer_dtype(series))

    if expected.startswith("float"):
        return bool(is_float_dtype(series))

    if expected in {"object", "string"}:
        return bool(
            is_object_dtype(series)
            or is_string_dtype(series)
        )

    if expected in {"bool", "boolean"}:
        return bool(is_bool_dtype(series))

    if "datetime" in expected:
        return bool(
            pd.api.types.is_datetime64_any_dtype(series)
        )

    return str(series.dtype).lower() == expected


# --------------------------------------------------------------
# 6. Create the frozen predictor register
# --------------------------------------------------------------

predictor_register_records = []

for _, contract_row in approved_contract_df.iterrows():

    column = contract_row["Column"]

    predictor_register_records.append(
        {
            "Predictor": column,
            "ColumnOrder": contract_row["ColumnOrder"],
            "ContractSection": contract_row["ContractSection"],
            "FeatureFamily": contract_row["FeatureFamily"],
            "IsHistoricalDemandFeature":
                contract_row["IsHistoricalDemandFeature"],
            "ExpectedDataType": contract_row["DataType"],
            "TrainingDataType": str(training_df[column].dtype),
            "ValidationDataType": str(
                validation_df[column].dtype
            ),
            "RawTypeGroup": raw_type_group(
                training_df[column]
            ),
            "MissingValuePolicy":
                contract_row["MissingValuePolicy"],
            "PredictionTimeStatus":
                contract_row["PredictionTimeStatus"],
            "UsesCurrentRowTarget":
                contract_row["UsesCurrentRowTarget"],
            "UsesFutureTarget":
                contract_row["UsesFutureTarget"],
            "FrozenForModelTraining":
                contract_row["FrozenForModelTraining"],
        }
    )

predictor_register_df = pd.DataFrame(
    predictor_register_records
)


# --------------------------------------------------------------
# 7. Create the predictor data-type audit
# --------------------------------------------------------------

dtype_audit_records = []

for _, contract_row in approved_contract_df.iterrows():

    column = contract_row["Column"]

    training_contract_match = dtype_matches_contract(
        training_df[column],
        contract_row["DataType"],
    )

    validation_contract_match = dtype_matches_contract(
        validation_df[column],
        contract_row["DataType"],
    )

    train_validation_match = (
        str(training_df[column].dtype)
        == str(validation_df[column].dtype)
    )

    dtype_audit_records.append(
        {
            "Predictor": column,
            "ExpectedDataType": contract_row["DataType"],
            "TrainingDataType": str(training_df[column].dtype),
            "ValidationDataType": str(
                validation_df[column].dtype
            ),
            "TrainingMatchesContract":
                training_contract_match,
            "ValidationMatchesContract":
                validation_contract_match,
            "TrainingValidationDtypeMatch":
                train_validation_match,
            "Passed": bool(
                training_contract_match
                and validation_contract_match
                and train_validation_match
            ),
        }
    )

dtype_audit_df = pd.DataFrame(dtype_audit_records)


# --------------------------------------------------------------
# 8. Create the predictor missingness audit
# --------------------------------------------------------------

missingness_records = []

for _, contract_row in approved_contract_df.iterrows():

    column = contract_row["Column"]

    training_missing_count = int(
        training_df[column].isna().sum()
    )

    validation_missing_count = int(
        validation_df[column].isna().sum()
    )

    missingness_records.append(
        {
            "Predictor": column,
            "FeatureFamily": contract_row["FeatureFamily"],
            "IsHistoricalDemandFeature":
                contract_row["IsHistoricalDemandFeature"],
            "MissingValuePolicy":
                contract_row["MissingValuePolicy"],
            "TrainingMissingCount":
                training_missing_count,
            "TrainingMissingPercentage":
                round(
                    training_missing_count
                    / len(training_df)
                    * 100,
                    4,
                ),
            "ValidationMissingCount":
                validation_missing_count,
            "ValidationMissingPercentage":
                round(
                    validation_missing_count
                    / len(validation_df)
                    * 100,
                    4,
                ),
        }
    )

missingness_audit_df = pd.DataFrame(
    missingness_records
)


# --------------------------------------------------------------
# 9. Validate historical-feature readiness fields
# --------------------------------------------------------------

training_calculated_history_count = (
    training_df[historical_predictors]
    .notna()
    .sum(axis=1)
)

validation_calculated_history_count = (
    validation_df[historical_predictors]
    .notna()
    .sum(axis=1)
)

training_history_count_mismatches = int(
    (
        training_calculated_history_count
        != training_df["HistoricalFeatureAvailableCount"]
    ).sum()
)

validation_history_count_mismatches = int(
    (
        validation_calculated_history_count
        != validation_df["HistoricalFeatureAvailableCount"]
    ).sum()
)

training_all_history_flag_mismatches = int(
    (
        (
            training_calculated_history_count
            == len(historical_predictors)
        )
        != training_df["AllHistoricalFeaturesAvailable"]
    ).sum()
)

validation_all_history_flag_mismatches = int(
    (
        (
            validation_calculated_history_count
            == len(historical_predictors)
        )
        != validation_df["AllHistoricalFeaturesAvailable"]
    ).sum()
)


# --------------------------------------------------------------
# 10. Calculate target-quality checks
# --------------------------------------------------------------

def count_fractional_values(series: pd.Series) -> int:
    """
    Count numeric values that are not whole numbers.
    """

    numeric = pd.to_numeric(series, errors="coerce")
    non_missing = numeric.dropna()

    return int(
        (~np.isclose(non_missing % 1, 0)).sum()
    )


training_target_missing = int(
    training_df[TARGET_COLUMN].isna().sum()
)

validation_target_missing = int(
    validation_df[TARGET_COLUMN].isna().sum()
)

training_negative_targets = int(
    (training_df[TARGET_COLUMN] < 0).sum()
)

validation_negative_targets = int(
    (validation_df[TARGET_COLUMN] < 0).sum()
)

training_fractional_targets = count_fractional_values(
    training_df[TARGET_COLUMN]
)

validation_fractional_targets = count_fractional_values(
    validation_df[TARGET_COLUMN]
)


# --------------------------------------------------------------
# 11. Build the Part 2 validation summary
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one check to the validation summary.
    """

    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Approved predictor count",
    len(approved_predictors),
    53,
)

add_check(
    "Current-date predictor count",
    len(current_date_predictors),
    22,
)

add_check(
    "Historical predictor count",
    len(historical_predictors),
    31,
)

add_check(
    "Predictors missing from training",
    len(missing_from_training),
    0,
)

add_check(
    "Predictors missing from validation",
    len(missing_from_validation),
    0,
)

# --------------------------------------------------------------
# Validate predictor membership rather than physical CSV order
# --------------------------------------------------------------

training_predictor_set_matches = (
    set(training_predictor_order)
    == set(approved_predictors)
)

validation_predictor_set_matches = (
    set(validation_predictor_order)
    == set(approved_predictors)
)

training_duplicate_predictor_names = (
    len(training_predictor_order)
    - len(set(training_predictor_order))
)

validation_duplicate_predictor_names = (
    len(validation_predictor_order)
    - len(set(validation_predictor_order))
)

add_check(
    "Training predictor set matches contract",
    training_predictor_set_matches,
    True,
)

add_check(
    "Validation predictor set matches contract",
    validation_predictor_set_matches,
    True,
)

add_check(
    "Duplicate approved predictors in training",
    training_duplicate_predictor_names,
    0,
)

add_check(
    "Duplicate approved predictors in validation",
    validation_duplicate_predictor_names,
    0,
)

add_check(
    "Training-contract dtype mismatches",
    int(
        (~dtype_audit_df["TrainingMatchesContract"]).sum()
    ),
    0,
)

add_check(
    "Validation-contract dtype mismatches",
    int(
        (~dtype_audit_df["ValidationMatchesContract"]).sum()
    ),
    0,
)

add_check(
    "Training-validation dtype mismatches",
    int(
        (
            ~dtype_audit_df[
                "TrainingValidationDtypeMatch"
            ]
        ).sum()
    ),
    0,
)

add_check(
    "Approved predictors using current-row target",
    int(
        approved_contract_df[
            "UsesCurrentRowTarget"
        ].sum()
    ),
    0,
)

add_check(
    "Approved predictors using future target",
    int(
        approved_contract_df[
            "UsesFutureTarget"
        ].sum()
    ),
    0,
)

add_check(
    "Target included as a predictor",
    TARGET_COLUMN in approved_predictors,
    False,
)

add_check(
    "Training history-count mismatches",
    training_history_count_mismatches,
    0,
)

add_check(
    "Validation history-count mismatches",
    validation_history_count_mismatches,
    0,
)

add_check(
    "Training all-history flag mismatches",
    training_all_history_flag_mismatches,
    0,
)

add_check(
    "Validation all-history flag mismatches",
    validation_all_history_flag_mismatches,
    0,
)

validation_historical_missing_cells = int(
    validation_df[historical_predictors]
    .isna()
    .sum()
    .sum()
)

add_check(
    "Validation historical-feature missing cells",
    validation_historical_missing_cells,
    0,
)

add_check(
    "Training target missing values",
    training_target_missing,
    0,
)

add_check(
    "Validation target missing values",
    validation_target_missing,
    0,
)

add_check(
    "Training negative target values",
    training_negative_targets,
    0,
)

add_check(
    "Validation negative target values",
    validation_negative_targets,
    0,
)

add_check(
    "Training fractional target values",
    training_fractional_targets,
    0,
)

add_check(
    "Validation fractional target values",
    validation_fractional_targets,
    0,
)


part2_validation_df = pd.DataFrame(
    validation_checks
)

# --------------------------------------------------------------
# Create predictor views in the exact frozen-contract order
# --------------------------------------------------------------

X_training_contract_order = training_df.loc[
    :,
    approved_predictors,
].copy()

X_validation_contract_order = validation_df.loc[
    :,
    approved_predictors,
].copy()

assert (
    X_training_contract_order.columns.tolist()
    == approved_predictors
)

assert (
    X_validation_contract_order.columns.tolist()
    == approved_predictors
)

print(
    "Training and validation predictor views were reordered "
    "to match the frozen feature contract."
)
# --------------------------------------------------------------
# 12. Stop if any required validation check fails
# --------------------------------------------------------------

failed_checks_df = part2_validation_df.loc[
    ~part2_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print("\nFAILED STEP 1 PART 2 CHECKS")
    display(failed_checks_df)

    raise AssertionError(
        "Modelling Step 1 Part 2 failed. "
        "Do not continue to preprocessing."
    )


# --------------------------------------------------------------
# 13. Create raw predictor-type summary
# --------------------------------------------------------------

raw_type_summary_df = (
    predictor_register_df
    .groupby("RawTypeGroup", as_index=False)
    .agg(
        PredictorCount=("Predictor", "count")
    )
    .sort_values(
        "PredictorCount",
        ascending=False,
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 14. Create historical readiness summary
# --------------------------------------------------------------

history_readiness_summary_df = pd.DataFrame(
    [
        {
            "Dataset": "MODEL_SELECTION_TRAINING",
            "Rows": len(training_df),
            "HistoricalPredictors":
                len(historical_predictors),
            "RowsWithAllHistoricalPredictors":
                int(
                    training_df[
                        historical_predictors
                    ].notna().all(axis=1).sum()
                ),
            "RowsWithMissingHistoricalPredictors":
                int(
                    training_df[
                        historical_predictors
                    ].isna().any(axis=1).sum()
                ),
            "MissingHistoricalCells":
                int(
                    training_df[
                        historical_predictors
                    ].isna().sum().sum()
                ),
        },
        {
            "Dataset": "MODEL_SELECTION_VALIDATION",
            "Rows": len(validation_df),
            "HistoricalPredictors":
                len(historical_predictors),
            "RowsWithAllHistoricalPredictors":
                int(
                    validation_df[
                        historical_predictors
                    ].notna().all(axis=1).sum()
                ),
            "RowsWithMissingHistoricalPredictors":
                int(
                    validation_df[
                        historical_predictors
                    ].isna().any(axis=1).sum()
                ),
            "MissingHistoricalCells":
                validation_historical_missing_cells,
        },
    ]
)


# --------------------------------------------------------------
# 15. Save Part 2 outputs
# --------------------------------------------------------------

predictor_register_path = (
    SETUP_OUTPUT_DIR /
    "02_frozen_predictor_register.csv"
)

dtype_audit_path = (
    SETUP_OUTPUT_DIR /
    "02_predictor_dtype_audit.csv"
)

missingness_audit_path = (
    SETUP_OUTPUT_DIR /
    "02_predictor_missingness_audit.csv"
)

raw_type_summary_path = (
    SETUP_OUTPUT_DIR /
    "02_raw_predictor_type_summary.csv"
)

history_readiness_path = (
    SETUP_OUTPUT_DIR /
    "02_historical_feature_readiness_summary.csv"
)

part2_validation_path = (
    SETUP_OUTPUT_DIR /
    "02_predictor_structure_validation_summary.csv"
)

predictor_register_df.to_csv(
    predictor_register_path,
    index=False,
)

dtype_audit_df.to_csv(
    dtype_audit_path,
    index=False,
)

missingness_audit_df.to_csv(
    missingness_audit_path,
    index=False,
)

raw_type_summary_df.to_csv(
    raw_type_summary_path,
    index=False,
)

history_readiness_summary_df.to_csv(
    history_readiness_path,
    index=False,
)

part2_validation_df.to_csv(
    part2_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 16. Display results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 1 PART 2: PASSED")
print("=" * 72)

print(f"\nApproved predictors: {len(approved_predictors)}")
print(
    f"Current-date predictors: "
    f"{len(current_date_predictors)}"
)
print(
    f"Historical predictors: "
    f"{len(historical_predictors)}"
)

print(
    "\nTraining rows with missing historical predictors: "
    f"{training_df[historical_predictors].isna().any(axis=1).sum():,}"
)

print(
    "Validation rows with missing historical predictors: "
    f"{validation_df[historical_predictors].isna().any(axis=1).sum():,}"
)

print(
    "Validation historical missing cells: "
    f"{validation_historical_missing_cells:,}"
)

print(
    "\nTraining target missing values: "
    f"{training_target_missing:,}"
)

print(
    "Validation target missing values: "
    f"{validation_target_missing:,}"
)

print(
    "Negative target values: "
    f"{training_negative_targets + validation_negative_targets:,}"
)

print(
    "Fractional target values: "
    f"{training_fractional_targets + validation_fractional_targets:,}"
)

print(
    f"\nValidation checks passed: "
    f"{part2_validation_df['Passed'].sum()} "
    f"of {len(part2_validation_df)}"
)

print("\nRaw predictor-type summary:")
display(raw_type_summary_df)

print("\nHistorical-feature readiness:")
display(history_readiness_summary_df)

print("\nPredictors containing missing values:")
display(
    missingness_audit_df.loc[
        (
            missingness_audit_df["TrainingMissingCount"] > 0
        )
        |
        (
            missingness_audit_df["ValidationMissingCount"] > 0
        )
    ].reset_index(drop=True)
)

print("\nPart 2 validation summary:")
display(part2_validation_df)

print("\nSaved outputs:")
print(predictor_register_path)
print(dtype_audit_path)
print(missingness_audit_path)
print(raw_type_summary_path)
print(history_readiness_path)
print(part2_validation_path)

Training and validation predictor views were reordered to match the frozen feature contract.

MODELLING STEP 1 PART 2: PASSED

Approved predictors: 53
Current-date predictors: 22
Historical predictors: 31

Training rows with missing historical predictors: 5,080
Validation rows with missing historical predictors: 0
Validation historical missing cells: 0

Training target missing values: 0
Validation target missing values: 0
Negative target values: 0
Fractional target values: 0

Validation checks passed: 26 of 26

Raw predictor-type summary:


,RawTypeGroup,PredictorCount
0,FLOAT_NUMERIC,32
1,INTEGER_NUMERIC,12
2,TEXT_OR_CATEGORY,6
3,BOOLEAN,3



Historical-feature readiness:


,Dataset,Rows,HistoricalPredictors,RowsWithAllHistoricalPredictors,RowsWithMissingHistoricalPredictors,MissingHistoricalCells
0,MODEL_SELECTION_TRAINING,48538,31,43458,5080,16764
1,MODEL_SELECTION_VALIDATION,5080,31,5080,0,0



Predictors containing missing values:


,Predictor,FeatureFamily,IsHistoricalDemandFeature,MissingValuePolicy,TrainingMissingCount,TrainingMissingPercentage,ValidationMissingCount,ValidationMissingPercentage
0,BeverageSeries,CONDITIONAL_PRODUCT_METADATA_FEATURE,False,PRESERVE_NOT_APPLICABLE_NA_UNTIL_MODEL_PIPELINE,46200,95.1832,4840,95.2756
1,BeverageType,CONDITIONAL_PRODUCT_METADATA_FEATURE,False,PRESERVE_NOT_APPLICABLE_NA_UNTIL_MODEL_PIPELINE,46200,95.1832,4840,95.2756
2,SupplierLabelsObserved,CONDITIONAL_PRODUCT_METADATA_FEATURE,False,PRESERVE_NOT_APPLICABLE_NA_UNTIL_MODEL_PIPELINE,46200,95.1832,4840,95.2756
3,TierProductFamily,CONDITIONAL_PRODUCT_METADATA_FEATURE,False,PRESERVE_NOT_APPLICABLE_NA_UNTIL_MODEL_PIPELINE,45932,94.6310,4800,94.4882
4,NominalPriceTier,CONDITIONAL_PRODUCT_METADATA_FEATURE,False,PRESERVE_NOT_APPLICABLE_NA_UNTIL_MODEL_PIPELINE,45932,94.6310,4800,94.4882
5,MenuGeneration,CONDITIONAL_PRODUCT_METADATA_FEATURE,False,PRESERVE_NOT_APPLICABLE_NA_UNTIL_MODEL_PIPELINE,45932,94.6310,4800,94.4882
6,TotalDemandLag_1,DEMAND_LAG,True,PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE,254,0.5233,0,0.0000
7,TotalDemandLag_2,DEMAND_LAG,True,PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE,508,1.0466,0,0.0000
8,TotalDemandLag_3,DEMAND_LAG,True,PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE,762,1.5699,0,0.0000
9,TotalDemandLag_5,DEMAND_LAG,True,PRESERVE_EARLY_HISTORY_NA_UNTIL_MODEL_PIPELINE,1270,2.6165,0,0.0000



Part 2 validation summary:


,Check,Expected,Actual,Passed
0,Approved predictor count,53,53,True
1,Current-date predictor count,22,22,True
2,Historical predictor count,31,31,True
3,Predictors missing from training,0,0,True
4,Predictors missing from validation,0,0,True
5,Training predictor set matches contract,True,True,True
6,Validation predictor set matches contract,True,True,True
7,Duplicate approved predictors in training,0,0,True
8,Duplicate approved predictors in validation,0,0,True
9,Training-contract dtype mismatches,0,0,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/02_frozen_predictor_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/02_predictor_dtype_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/02_predictor_missingness_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/02_raw_predictor_type_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/02_historical_feature_readiness_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/02_predictor_structure_validation_summary.csv


In [6]:
# ==============================================================
# MODELLING STEP 1, PART 3
# Semantic predictor-role assignment
# ==============================================================

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype


# --------------------------------------------------------------
# 1. Confirm required objects from Parts 1 and 2
# --------------------------------------------------------------

required_objects = [
    "training_df",
    "validation_df",
    "approved_contract_df",
    "approved_predictors",
    "SETUP_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 1 Parts 1 and 2 first."
    )


# --------------------------------------------------------------
# 2. Define predictors with special semantic roles
#
# These definitions do not change the frozen contract.
# They specify how approved predictors will be preprocessed.
# --------------------------------------------------------------

CATEGORICAL_PREDICTOR_SET = {
    # Stored as an integer code, but arithmetic differences between
    # group codes have no forecasting meaning.
    "SourceGroupCodes",

    # Text-based product metadata.
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "MenuGeneration",
}

BINARY_PREDICTOR_SET = {
    "IsMultiPLUCanonicalProduct",
    "IsWeekend",
    "IsConsecutiveCalendarDay",
}


# Preserve the exact frozen-contract order inside each role.
categorical_predictors = [
    column
    for column in approved_predictors
    if column in CATEGORICAL_PREDICTOR_SET
]

binary_predictors = [
    column
    for column in approved_predictors
    if column in BINARY_PREDICTOR_SET
]

numeric_predictors = [
    column
    for column in approved_predictors
    if (
        column not in CATEGORICAL_PREDICTOR_SET
        and column not in BINARY_PREDICTOR_SET
    )
]


# --------------------------------------------------------------
# 3. Define semantic-role explanations
# --------------------------------------------------------------

def semantic_role_reason(
    predictor,
    semantic_role,
    feature_family,
):
    """
    Return a concise explanation for the assigned semantic role.
    """

    if predictor == "SourceGroupCodes":
        return (
            "Integer group code representing category membership; "
            "numerical distance between codes has no meaning."
        )

    if predictor == "NominalPriceTier":
        return (
            "Ordered numeric price-level feature; missing values "
            "represent not-applicable product metadata and will "
            "receive a missingness indicator."
        )

    if semantic_role == "CATEGORICAL":
        return (
            "Product metadata consisting of labels or categories "
            "rather than continuous numerical measurements."
        )

    if semantic_role == "BINARY":
        return (
            "Boolean indicator with two possible states; it will "
            "be represented as 0 or 1."
        )

    if feature_family in {
        "DEMAND_LAG",
        "MEAN",
        "MEDIAN",
        "STANDARD_DEVIATION",
        "SUM",
        "ROLLING_ZERO_DEMAND_RATE",
        "ROLLING_POSITIVE_DEMAND_COUNT",
        "DAYS_SINCE_PREVIOUS_POSITIVE_DEMAND",
        "EXPANDING_PAST_MEAN",
        "EXPANDING_PAST_POSITIVE_RATE",
    }:
        return (
            "Past-only numerical demand measurement or historical "
            "summary used as a quantitative predictor."
        )

    return (
        "Known-ahead numerical count, sequence, calendar value or "
        "ordered product characteristic."
    )


# --------------------------------------------------------------
# 4. Define intended preprocessing policies
# --------------------------------------------------------------

role_policy = {
    "NUMERIC": {
        "MissingTreatment":
            "TRAINING_MEDIAN_WITH_MISSING_INDICATOR",
        "LinearModelTreatment":
            "MEDIAN_IMPUTATION_MISSING_INDICATOR_AND_STANDARD_SCALING",
        "TreeModelTreatment":
            "MEDIAN_IMPUTATION_WITH_MISSING_INDICATOR_NO_SCALING",
        "Encoding":
            "NONE",
    },

    "CATEGORICAL": {
        "MissingTreatment":
            "CONSTANT_NOT_APPLICABLE_OR_MISSING_CATEGORY",
        "LinearModelTreatment":
            "CONSTANT_IMPUTATION_AND_ONE_HOT_ENCODING",
        "TreeModelTreatment":
            "CONSTANT_IMPUTATION_AND_ONE_HOT_ENCODING",
        "Encoding":
            "ONE_HOT_HANDLE_UNKNOWN_IGNORE",
    },

    "BINARY": {
        "MissingTreatment":
            "NO_MISSING_EXPECTED",
        "LinearModelTreatment":
            "CONVERT_BOOLEAN_TO_ZERO_OR_ONE",
        "TreeModelTreatment":
            "CONVERT_BOOLEAN_TO_ZERO_OR_ONE",
        "Encoding":
            "BOOLEAN_TO_INTEGER",
    },
}


# --------------------------------------------------------------
# 5. Build the semantic predictor-role register
# --------------------------------------------------------------

role_records = []

for _, contract_row in approved_contract_df.iterrows():

    predictor = contract_row["Column"]

    if predictor in CATEGORICAL_PREDICTOR_SET:
        semantic_role = "CATEGORICAL"

    elif predictor in BINARY_PREDICTOR_SET:
        semantic_role = "BINARY"

    else:
        semantic_role = "NUMERIC"

    policy = role_policy[semantic_role]

    role_records.append(
        {
            "Predictor": predictor,
            "FrozenColumnOrder": contract_row["ColumnOrder"],
            "FeatureFamily": contract_row["FeatureFamily"],
            "ContractDataType": contract_row["DataType"],
            "SemanticRole": semantic_role,
            "RoleReason": semantic_role_reason(
                predictor=predictor,
                semantic_role=semantic_role,
                feature_family=contract_row["FeatureFamily"],
            ),
            "ContractMissingValuePolicy":
                contract_row["MissingValuePolicy"],
            "PipelineMissingTreatment":
                policy["MissingTreatment"],
            "LinearModelTreatment":
                policy["LinearModelTreatment"],
            "TreeModelTreatment":
                policy["TreeModelTreatment"],
            "EncodingPolicy":
                policy["Encoding"],
        }
    )

semantic_role_register_df = pd.DataFrame(role_records)


# --------------------------------------------------------------
# 6. Audit numerical predictors
# --------------------------------------------------------------

numeric_quality_records = []

for predictor in numeric_predictors:

    training_numeric = pd.to_numeric(
        training_df[predictor],
        errors="coerce",
    )

    validation_numeric = pd.to_numeric(
        validation_df[predictor],
        errors="coerce",
    )

    training_unexpected_non_numeric = int(
        (
            training_df[predictor].notna()
            & training_numeric.isna()
        ).sum()
    )

    validation_unexpected_non_numeric = int(
        (
            validation_df[predictor].notna()
            & validation_numeric.isna()
        ).sum()
    )

    training_infinite_count = int(
        np.isinf(
            training_numeric.dropna().to_numpy(dtype=float)
        ).sum()
    )

    validation_infinite_count = int(
        np.isinf(
            validation_numeric.dropna().to_numpy(dtype=float)
        ).sum()
    )

    numeric_quality_records.append(
        {
            "Predictor": predictor,
            "TrainingDataType":
                str(training_df[predictor].dtype),
            "ValidationDataType":
                str(validation_df[predictor].dtype),
            "TrainingMissingCount":
                int(training_df[predictor].isna().sum()),
            "ValidationMissingCount":
                int(validation_df[predictor].isna().sum()),
            "TrainingUnexpectedNonNumericCount":
                training_unexpected_non_numeric,
            "ValidationUnexpectedNonNumericCount":
                validation_unexpected_non_numeric,
            "TrainingInfiniteCount":
                training_infinite_count,
            "ValidationInfiniteCount":
                validation_infinite_count,
            "Passed": bool(
                training_unexpected_non_numeric == 0
                and validation_unexpected_non_numeric == 0
                and training_infinite_count == 0
                and validation_infinite_count == 0
            ),
        }
    )

numeric_quality_audit_df = pd.DataFrame(
    numeric_quality_records
)


# --------------------------------------------------------------
# 7. Audit categorical levels
# --------------------------------------------------------------

categorical_coverage_records = []

for predictor in categorical_predictors:

    training_levels = set(
        training_df[predictor]
        .dropna()
        .astype(str)
        .unique()
    )

    validation_levels = set(
        validation_df[predictor]
        .dropna()
        .astype(str)
        .unique()
    )

    unseen_validation_levels = sorted(
        validation_levels - training_levels
    )

    categorical_coverage_records.append(
        {
            "Predictor": predictor,
            "TrainingLevelCount": len(training_levels),
            "ValidationLevelCount": len(validation_levels),
            "TrainingMissingCount":
                int(training_df[predictor].isna().sum()),
            "ValidationMissingCount":
                int(validation_df[predictor].isna().sum()),
            "UnseenValidationLevelCount":
                len(unseen_validation_levels),
            "UnseenValidationLevels":
                " | ".join(unseen_validation_levels),
            "PipelineSafetyPolicy":
                "ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE",
        }
    )

categorical_level_audit_df = pd.DataFrame(
    categorical_coverage_records
)


# --------------------------------------------------------------
# 8. Audit binary predictors
# --------------------------------------------------------------

def invalid_binary_count(series):
    """
    Count non-missing values that cannot represent a binary value.
    """

    valid_values = {
        True,
        False,
        1,
        0,
        "True",
        "False",
        "true",
        "false",
        "1",
        "0",
    }

    non_missing_values = series.dropna()

    return int(
        (~non_missing_values.isin(valid_values)).sum()
    )


binary_quality_records = []

for predictor in binary_predictors:

    training_invalid = invalid_binary_count(
        training_df[predictor]
    )

    validation_invalid = invalid_binary_count(
        validation_df[predictor]
    )

    training_missing = int(
        training_df[predictor].isna().sum()
    )

    validation_missing = int(
        validation_df[predictor].isna().sum()
    )

    binary_quality_records.append(
        {
            "Predictor": predictor,
            "TrainingDataType":
                str(training_df[predictor].dtype),
            "ValidationDataType":
                str(validation_df[predictor].dtype),
            "TrainingUniqueValues":
                " | ".join(
                    sorted(
                        training_df[predictor]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                ),
            "ValidationUniqueValues":
                " | ".join(
                    sorted(
                        validation_df[predictor]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                ),
            "TrainingMissingCount": training_missing,
            "ValidationMissingCount": validation_missing,
            "TrainingInvalidBinaryCount": training_invalid,
            "ValidationInvalidBinaryCount": validation_invalid,
            "Passed": bool(
                training_missing == 0
                and validation_missing == 0
                and training_invalid == 0
                and validation_invalid == 0
            ),
        }
    )

binary_quality_audit_df = pd.DataFrame(
    binary_quality_records
)


# --------------------------------------------------------------
# 9. Create role summary
# --------------------------------------------------------------

role_summary_df = (
    semantic_role_register_df
    .groupby("SemanticRole", as_index=False)
    .agg(
        PredictorCount=("Predictor", "count")
    )
    .sort_values(
        "SemanticRole"
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 10. Validate role assignment
# --------------------------------------------------------------

assigned_predictors = (
    numeric_predictors
    + categorical_predictors
    + binary_predictors
)

assigned_predictor_set = set(assigned_predictors)
approved_predictor_set = set(approved_predictors)

overlap_count = (
    len(set(numeric_predictors) & set(categorical_predictors))
    + len(set(numeric_predictors) & set(binary_predictors))
    + len(set(categorical_predictors) & set(binary_predictors))
)

unassigned_predictors = sorted(
    approved_predictor_set - assigned_predictor_set
)

unexpected_predictors = sorted(
    assigned_predictor_set - approved_predictor_set
)

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Approved predictor count",
    len(approved_predictors),
    53,
)

add_check(
    "Numerical predictor count",
    len(numeric_predictors),
    43,
)

add_check(
    "Categorical predictor count",
    len(categorical_predictors),
    7,
)

add_check(
    "Binary predictor count",
    len(binary_predictors),
    3,
)

add_check(
    "Total assigned predictor count",
    len(assigned_predictors),
    53,
)

add_check(
    "Unique assigned predictor count",
    len(assigned_predictor_set),
    53,
)

add_check(
    "Cross-role overlap count",
    overlap_count,
    0,
)

add_check(
    "Unassigned approved predictors",
    len(unassigned_predictors),
    0,
)

add_check(
    "Unexpected assigned predictors",
    len(unexpected_predictors),
    0,
)

add_check(
    "Role-register rows",
    len(semantic_role_register_df),
    53,
)

add_check(
    "Duplicate role-register predictors",
    int(
        semantic_role_register_df[
            "Predictor"
        ].duplicated().sum()
    ),
    0,
)

add_check(
    "Numerical predictors failing quality audit",
    int(
        (~numeric_quality_audit_df["Passed"]).sum()
    ),
    0,
)

add_check(
    "Binary predictors failing quality audit",
    int(
        (~binary_quality_audit_df["Passed"]).sum()
    ),
    0,
)

add_check(
    "Forecast target assigned as predictor",
    "TotalDemand" in assigned_predictor_set,
    False,
)

part3_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 11. Stop if validation fails
# --------------------------------------------------------------

failed_checks_df = part3_validation_df.loc[
    ~part3_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print("\nFAILED STEP 1 PART 3 CHECKS")
    display(failed_checks_df)

    raise AssertionError(
        "Modelling Step 1 Part 3 failed. "
        "Do not construct the preprocessing transformers."
    )


# --------------------------------------------------------------
# 12. Save Part 3 outputs
# --------------------------------------------------------------

semantic_role_register_path = (
    SETUP_OUTPUT_DIR /
    "03_semantic_predictor_role_register.csv"
)

role_summary_path = (
    SETUP_OUTPUT_DIR /
    "03_preprocessing_role_summary.csv"
)

numeric_quality_path = (
    SETUP_OUTPUT_DIR /
    "03_numeric_predictor_quality_audit.csv"
)

categorical_level_path = (
    SETUP_OUTPUT_DIR /
    "03_categorical_level_coverage_audit.csv"
)

binary_quality_path = (
    SETUP_OUTPUT_DIR /
    "03_binary_predictor_quality_audit.csv"
)

part3_validation_path = (
    SETUP_OUTPUT_DIR /
    "03_semantic_role_validation_summary.csv"
)

semantic_role_register_df.to_csv(
    semantic_role_register_path,
    index=False,
)

role_summary_df.to_csv(
    role_summary_path,
    index=False,
)

numeric_quality_audit_df.to_csv(
    numeric_quality_path,
    index=False,
)

categorical_level_audit_df.to_csv(
    categorical_level_path,
    index=False,
)

binary_quality_audit_df.to_csv(
    binary_quality_path,
    index=False,
)

part3_validation_df.to_csv(
    part3_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 13. Display results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 1 PART 3: PASSED")
print("=" * 72)

print(f"\nNumerical predictors: {len(numeric_predictors)}")
print(f"Categorical predictors: {len(categorical_predictors)}")
print(f"Binary predictors: {len(binary_predictors)}")
print(f"Total assigned predictors: {len(assigned_predictors)}")

print(
    "\nNumerical quality failures: "
    f"{(~numeric_quality_audit_df['Passed']).sum()}"
)

print(
    "Binary quality failures: "
    f"{(~binary_quality_audit_df['Passed']).sum()}"
)

categorical_predictors_with_unseen_levels = int(
    (
        categorical_level_audit_df[
            "UnseenValidationLevelCount"
        ] > 0
    ).sum()
)

print(
    "Categorical predictors with unseen validation levels: "
    f"{categorical_predictors_with_unseen_levels}"
)

print(
    f"\nValidation checks passed: "
    f"{part3_validation_df['Passed'].sum()} "
    f"of {len(part3_validation_df)}"
)

print("\nPreprocessing-role summary:")
display(role_summary_df)

print("\nCategorical-level coverage:")
display(categorical_level_audit_df)

print("\nBinary predictor audit:")
display(binary_quality_audit_df)

print("\nPart 3 validation summary:")
display(part3_validation_df)

print("\nSaved outputs:")
print(semantic_role_register_path)
print(role_summary_path)
print(numeric_quality_path)
print(categorical_level_path)
print(binary_quality_path)
print(part3_validation_path)


MODELLING STEP 1 PART 3: PASSED

Numerical predictors: 43
Categorical predictors: 7
Binary predictors: 3
Total assigned predictors: 53

Numerical quality failures: 0
Binary quality failures: 0
Categorical predictors with unseen validation levels: 0

Validation checks passed: 14 of 14

Preprocessing-role summary:


,SemanticRole,PredictorCount
0,BINARY,3
1,CATEGORICAL,7
2,NUMERIC,43



Categorical-level coverage:


,Predictor,TrainingLevelCount,ValidationLevelCount,TrainingMissingCount,ValidationMissingCount,UnseenValidationLevelCount,UnseenValidationLevels,PipelineSafetyPolicy
0,SourceGroupCodes,9,9,0,0,0,,ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE
1,SourceGroupNames,9,9,0,0,0,,ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE
2,BeverageSeries,1,1,46200,4840,0,,ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE
3,BeverageType,6,6,46200,4840,0,,ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE
4,SupplierLabelsObserved,2,2,46200,4840,0,,ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE
5,TierProductFamily,3,3,45932,4800,0,,ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE
6,MenuGeneration,1,1,45932,4800,0,,ONE_HOT_ENCODER_HANDLE_UNKNOWN_IGNORE



Binary predictor audit:


,Predictor,TrainingDataType,ValidationDataType,TrainingUniqueValues,ValidationUniqueValues,TrainingMissingCount,ValidationMissingCount,TrainingInvalidBinaryCount,ValidationInvalidBinaryCount,Passed
0,IsMultiPLUCanonicalProduct,bool,bool,False | True,False | True,0,0,0,0,True
1,IsWeekend,bool,bool,False | True,False | True,0,0,0,0,True
2,IsConsecutiveCalendarDay,bool,bool,False | True,False | True,0,0,0,0,True



Part 3 validation summary:


,Check,Expected,Actual,Passed
0,Approved predictor count,53,53,True
1,Numerical predictor count,43,43,True
2,Categorical predictor count,7,7,True
3,Binary predictor count,3,3,True
4,Total assigned predictor count,53,53,True
5,Unique assigned predictor count,53,53,True
6,Cross-role overlap count,0,0,True
7,Unassigned approved predictors,0,0,True
8,Unexpected assigned predictors,0,0,True
9,Role-register rows,53,53,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/03_semantic_predictor_role_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/03_preprocessing_role_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/03_numeric_predictor_quality_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/03_categorical_level_coverage_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/03_binary_predictor_quality_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/03_semantic_role_validation_summary.csv


In [8]:
# ==============================================================
# MODELLING STEP 1, PART 4
# Build and validate preprocessing transformers
# ==============================================================

import inspect

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# --------------------------------------------------------------
# 1. Confirm that earlier modelling parts are available
# --------------------------------------------------------------

required_objects = [
    "training_df",
    "validation_df",
    "approved_predictors",
    "numeric_predictors",
    "categorical_predictors",
    "binary_predictors",
    "SETUP_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 1 Parts 1, 2 and 3 first."
    )


# --------------------------------------------------------------
# 2. Validate the predictor-role inputs
# --------------------------------------------------------------

all_role_predictors = (
    numeric_predictors
    + categorical_predictors
    + binary_predictors
)

if len(approved_predictors) != 53:
    raise AssertionError(
        "Expected 53 approved predictors, but found "
        f"{len(approved_predictors)}."
    )

if len(numeric_predictors) != 43:
    raise AssertionError(
        "Expected 43 numerical predictors, but found "
        f"{len(numeric_predictors)}."
    )

if len(categorical_predictors) != 7:
    raise AssertionError(
        "Expected 7 categorical predictors, but found "
        f"{len(categorical_predictors)}."
    )

if len(binary_predictors) != 3:
    raise AssertionError(
        "Expected 3 binary predictors, but found "
        f"{len(binary_predictors)}."
    )

if set(all_role_predictors) != set(approved_predictors):
    raise AssertionError(
        "The semantic predictor roles do not exactly match "
        "the frozen approved predictor set."
    )

if len(all_role_predictors) != len(set(all_role_predictors)):
    raise AssertionError(
        "At least one predictor has been assigned to more than "
        "one preprocessing role."
    )


# --------------------------------------------------------------
# 3. Build a version-compatible dense one-hot encoder
# --------------------------------------------------------------

def build_dense_one_hot_encoder():
    """
    Create a OneHotEncoder that returns a dense float array.

    Newer versions of scikit-learn use sparse_output=False.
    Older versions use sparse=False.
    """

    encoder_parameters = {
        "handle_unknown": "ignore",
        "dtype": np.float64,
    }

    one_hot_parameters = inspect.signature(
        OneHotEncoder
    ).parameters

    if "sparse_output" in one_hot_parameters:
        encoder_parameters["sparse_output"] = False
    else:
        encoder_parameters["sparse"] = False

    return OneHotEncoder(**encoder_parameters)


# --------------------------------------------------------------
# 4. Build preprocessing for linear models
# --------------------------------------------------------------

def build_linear_preprocessor():
    """
    Create a fresh, unfitted preprocessing transformer for:

    - Linear Regression
    - Ridge Regression
    - Other scale-sensitive linear models
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "median_imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            (
                "standard_scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "missing_category_imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=(
                        "__NOT_APPLICABLE_OR_MISSING__"
                    ),
                ),
            ),
            (
                "one_hot_encoder",
                build_dense_one_hot_encoder(),
            ),
        ]
    )

    linear_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_predictors,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_predictors,
            ),
            (
                "binary",
                "passthrough",
                binary_predictors,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=True,
    )

    return linear_preprocessor


# --------------------------------------------------------------
# 5. Build preprocessing for tree-based models
# --------------------------------------------------------------

def build_tree_preprocessor():
    """
    Create a fresh, unfitted preprocessing transformer for:

    - Random Forest
    - Gradient Boosting
    - Other tree-based models

    Numerical scaling is not used because tree-based models do
    not require predictors to have comparable scales.
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "median_imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "missing_category_imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=(
                        "__NOT_APPLICABLE_OR_MISSING__"
                    ),
                ),
            ),
            (
                "one_hot_encoder",
                build_dense_one_hot_encoder(),
            ),
        ]
    )

    tree_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_predictors,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_predictors,
            ),
            (
                "binary",
                "passthrough",
                binary_predictors,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=True,
    )

    return tree_preprocessor


# --------------------------------------------------------------
# 6. Record the preprocessing policy
# --------------------------------------------------------------

preprocessing_policy_df = pd.DataFrame(
    [
        {
            "PipelineType": "LINEAR_MODEL",
            "PredictorRole": "NUMERIC",
            "MissingTreatment": "TRAINING_MEDIAN",
            "MissingIndicatorAdded": True,
            "Scaling": "STANDARD_SCALER",
            "Encoding": "NONE",
        },
        {
            "PipelineType": "LINEAR_MODEL",
            "PredictorRole": "CATEGORICAL",
            "MissingTreatment":
                "__NOT_APPLICABLE_OR_MISSING__",
            "MissingIndicatorAdded": False,
            "Scaling": "NONE",
            "Encoding":
                "ONE_HOT_HANDLE_UNKNOWN_IGNORE",
        },
        {
            "PipelineType": "LINEAR_MODEL",
            "PredictorRole": "BINARY",
            "MissingTreatment": "NONE_REQUIRED",
            "MissingIndicatorAdded": False,
            "Scaling": "NONE",
            "Encoding": "PASSTHROUGH_AS_ZERO_OR_ONE",
        },
        {
            "PipelineType": "TREE_MODEL",
            "PredictorRole": "NUMERIC",
            "MissingTreatment": "TRAINING_MEDIAN",
            "MissingIndicatorAdded": True,
            "Scaling": "NONE",
            "Encoding": "NONE",
        },
        {
            "PipelineType": "TREE_MODEL",
            "PredictorRole": "CATEGORICAL",
            "MissingTreatment":
                "__NOT_APPLICABLE_OR_MISSING__",
            "MissingIndicatorAdded": False,
            "Scaling": "NONE",
            "Encoding":
                "ONE_HOT_HANDLE_UNKNOWN_IGNORE",
        },
        {
            "PipelineType": "TREE_MODEL",
            "PredictorRole": "BINARY",
            "MissingTreatment": "NONE_REQUIRED",
            "MissingIndicatorAdded": False,
            "Scaling": "NONE",
            "Encoding": "PASSTHROUGH_AS_ZERO_OR_ONE",
        },
    ]
)


# --------------------------------------------------------------
# 7. Define the preprocessing systems
# --------------------------------------------------------------

pipeline_builders = {
    "LINEAR_MODEL": build_linear_preprocessor,
    "TREE_MODEL": build_tree_preprocessor,
}


# --------------------------------------------------------------
# 8. Identify the chronological model-selection folds
# --------------------------------------------------------------

validation_windows = sorted(
    validation_df["WindowID"]
    .dropna()
    .unique()
    .tolist()
)

if len(validation_windows) != 2:
    raise AssertionError(
        "Expected two model-selection validation windows, "
        f"but found {len(validation_windows)}."
    )


# --------------------------------------------------------------
# 9. Fit and validate each transformer within each fold
# --------------------------------------------------------------

transformation_audit_records = []
generated_feature_records = []

for window_id in validation_windows:

    fold_training_df = training_df.loc[
        training_df["WindowID"] == window_id
    ].copy()

    fold_validation_df = validation_df.loc[
        validation_df["WindowID"] == window_id
    ].copy()

    if fold_training_df.empty:
        raise ValueError(
            f"No training rows were found for {window_id}."
        )

    if fold_validation_df.empty:
        raise ValueError(
            f"No validation rows were found for {window_id}."
        )

    training_max_date = pd.to_datetime(
        fold_training_df["Date"]
    ).max()

    validation_min_date = pd.to_datetime(
        fold_validation_df["Date"]
    ).min()

    chronological_order_passed = bool(
        training_max_date < validation_min_date
    )

    if not chronological_order_passed:
        raise AssertionError(
            "Chronological overlap was found in "
            f"{window_id}."
        )

    # Select predictors in the exact frozen-contract order.
    X_fold_training = fold_training_df.loc[
        :,
        approved_predictors,
    ].copy()

    X_fold_validation = fold_validation_df.loc[
        :,
        approved_predictors,
    ].copy()

    for pipeline_type, pipeline_builder in (
        pipeline_builders.items()
    ):

        # Create a fresh transformer for this fold.
        preprocessor = pipeline_builder()

        # Fit preprocessing only on training rows.
        transformed_training = (
            preprocessor.fit_transform(
                X_fold_training
            )
        )

        # Apply the training-fitted transformation to validation.
        transformed_validation = (
            preprocessor.transform(
                X_fold_validation
            )
        )

        transformed_training = np.asarray(
            transformed_training,
            dtype=np.float64,
        )

        transformed_validation = np.asarray(
            transformed_validation,
            dtype=np.float64,
        )

        generated_feature_names = (
            preprocessor
            .get_feature_names_out()
            .tolist()
        )

        training_missing_cells = int(
            np.isnan(transformed_training).sum()
        )

        validation_missing_cells = int(
            np.isnan(transformed_validation).sum()
        )

        training_infinite_cells = int(
            np.isinf(transformed_training).sum()
        )

        validation_infinite_cells = int(
            np.isinf(transformed_validation).sum()
        )

        output_feature_count_match = bool(
            transformed_training.shape[1]
            == transformed_validation.shape[1]
        )

        generated_feature_name_count_match = bool(
            len(generated_feature_names)
            == transformed_training.shape[1]
        )

        duplicate_generated_feature_names = int(
            len(generated_feature_names)
            - len(set(generated_feature_names))
        )

        transformation_passed = bool(
            transformed_training.shape[0]
            == len(fold_training_df)
            and transformed_validation.shape[0]
            == len(fold_validation_df)
            and output_feature_count_match
            and generated_feature_name_count_match
            and duplicate_generated_feature_names == 0
            and training_missing_cells == 0
            and validation_missing_cells == 0
            and training_infinite_cells == 0
            and validation_infinite_cells == 0
            and chronological_order_passed
        )

        transformation_audit_records.append(
            {
                "WindowID": window_id,
                "PipelineType": pipeline_type,
                "TrainingRows":
                    len(fold_training_df),
                "ValidationRows":
                    len(fold_validation_df),
                "TrainingEndDate":
                    training_max_date.date(),
                "ValidationStartDate":
                    validation_min_date.date(),
                "ChronologicalOrderPassed":
                    chronological_order_passed,
                "OriginalPredictorCount":
                    len(approved_predictors),
                "GeneratedFeatureCount":
                    transformed_training.shape[1],
                "TrainingOutputRows":
                    transformed_training.shape[0],
                "ValidationOutputRows":
                    transformed_validation.shape[0],
                "TrainingMissingCells":
                    training_missing_cells,
                "ValidationMissingCells":
                    validation_missing_cells,
                "TrainingInfiniteCells":
                    training_infinite_cells,
                "ValidationInfiniteCells":
                    validation_infinite_cells,
                "DuplicateGeneratedFeatureNames":
                    duplicate_generated_feature_names,
                "Passed":
                    transformation_passed,
            }
        )

        for feature_position, feature_name in enumerate(
            generated_feature_names,
            start=1,
        ):
            generated_feature_records.append(
                {
                    "WindowID": window_id,
                    "PipelineType": pipeline_type,
                    "GeneratedFeaturePosition":
                        feature_position,
                    "GeneratedFeatureName":
                        feature_name,
                }
            )


transformation_audit_df = pd.DataFrame(
    transformation_audit_records
)

generated_feature_register_df = pd.DataFrame(
    generated_feature_records
)


# --------------------------------------------------------------
# 10. Create generated-feature summaries
# --------------------------------------------------------------

feature_count_summary_df = (
    transformation_audit_df
    .groupby(
        [
            "WindowID",
            "PipelineType",
        ],
        as_index=False,
    )
    .agg(
        GeneratedFeatureCount=(
            "GeneratedFeatureCount",
            "first",
        ),
        TrainingMissingCells=(
            "TrainingMissingCells",
            "first",
        ),
        ValidationMissingCells=(
            "ValidationMissingCells",
            "first",
        ),
        TrainingInfiniteCells=(
            "TrainingInfiniteCells",
            "first",
        ),
        ValidationInfiniteCells=(
            "ValidationInfiniteCells",
            "first",
        ),
        Passed=(
            "Passed",
            "first",
        ),
    )
)

generated_feature_counts = sorted(
    transformation_audit_df[
        "GeneratedFeatureCount"
    ]
    .unique()
    .tolist()
)


# --------------------------------------------------------------
# 11. Build the Part 4 validation summary
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one check to the Part 4 validation summary.
    """

    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Approved input predictor count",
    len(approved_predictors),
    53,
)

add_check(
    "Validation fold count",
    len(validation_windows),
    2,
)

add_check(
    "Preprocessing pipeline types",
    sorted(pipeline_builders.keys()),
    [
        "LINEAR_MODEL",
        "TREE_MODEL",
    ],
)

add_check(
    "Transformation audit rows",
    len(transformation_audit_df),
    4,
)

add_check(
    "Failed transformation contexts",
    int(
        (
            ~transformation_audit_df["Passed"]
        ).sum()
    ),
    0,
)

add_check(
    "Contexts with chronological overlap",
    int(
        (
            ~transformation_audit_df[
                "ChronologicalOrderPassed"
            ]
        ).sum()
    ),
    0,
)

add_check(
    "Total transformed training missing cells",
    int(
        transformation_audit_df[
            "TrainingMissingCells"
        ].sum()
    ),
    0,
)

add_check(
    "Total transformed validation missing cells",
    int(
        transformation_audit_df[
            "ValidationMissingCells"
        ].sum()
    ),
    0,
)

add_check(
    "Total transformed training infinite cells",
    int(
        transformation_audit_df[
            "TrainingInfiniteCells"
        ].sum()
    ),
    0,
)

add_check(
    "Total transformed validation infinite cells",
    int(
        transformation_audit_df[
            "ValidationInfiniteCells"
        ].sum()
    ),
    0,
)

add_check(
    "Duplicate generated feature names",
    int(
        transformation_audit_df[
            "DuplicateGeneratedFeatureNames"
        ].sum()
    ),
    0,
)

add_check(
    "Consistent generated feature count",
    len(generated_feature_counts),
    1,
)

add_check(
    "Generated feature count",
    generated_feature_counts,
    [114],
)

part4_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 12. Stop if any validation check fails
# --------------------------------------------------------------

failed_checks_df = part4_validation_df.loc[
    ~part4_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print("\nFAILED STEP 1 PART 4 CHECKS")
    display(failed_checks_df)

    raise AssertionError(
        "Modelling Step 1 Part 4 failed. "
        "Do not begin model pipeline construction."
    )


# --------------------------------------------------------------
# 13. Create fresh reusable, unfitted transformer templates
# --------------------------------------------------------------

linear_preprocessor_template = (
    build_linear_preprocessor()
)

tree_preprocessor_template = (
    build_tree_preprocessor()
)


# --------------------------------------------------------------
# 14. Save the Part 4 audit outputs
# --------------------------------------------------------------

preprocessing_policy_path = (
    SETUP_OUTPUT_DIR
    / "04_preprocessing_policy.csv"
)

transformation_audit_path = (
    SETUP_OUTPUT_DIR
    / "04_fold_preprocessing_transformation_audit.csv"
)

generated_feature_register_path = (
    SETUP_OUTPUT_DIR
    / "04_generated_feature_register.csv"
)

feature_count_summary_path = (
    SETUP_OUTPUT_DIR
    / "04_generated_feature_count_summary.csv"
)

part4_validation_path = (
    SETUP_OUTPUT_DIR
    / "04_preprocessing_transformer_validation_summary.csv"
)

preprocessing_policy_df.to_csv(
    preprocessing_policy_path,
    index=False,
)

transformation_audit_df.to_csv(
    transformation_audit_path,
    index=False,
)

generated_feature_register_df.to_csv(
    generated_feature_register_path,
    index=False,
)

feature_count_summary_df.to_csv(
    feature_count_summary_path,
    index=False,
)

part4_validation_df.to_csv(
    part4_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 15. Calculate final display values
# --------------------------------------------------------------

failed_transformation_contexts = int(
    (
        ~transformation_audit_df["Passed"]
    ).sum()
)

total_missing_cells_after_preprocessing = int(
    transformation_audit_df[
        "TrainingMissingCells"
    ].sum()
    + transformation_audit_df[
        "ValidationMissingCells"
    ].sum()
)

total_infinite_cells_after_preprocessing = int(
    transformation_audit_df[
        "TrainingInfiniteCells"
    ].sum()
    + transformation_audit_df[
        "ValidationInfiniteCells"
    ].sum()
)

passed_check_count = int(
    part4_validation_df["Passed"].sum()
)

total_check_count = len(
    part4_validation_df
)

generated_model_feature_count = (
    generated_feature_counts[0]
)


# --------------------------------------------------------------
# 16. Display the final results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 1 PART 4: PASSED")
print("=" * 72)

print(
    "\nOriginal approved predictors: "
    f"{len(approved_predictors)}"
)

print(
    "Generated model features: "
    f"{generated_model_feature_count}"
)

print(
    "Validation folds checked: "
    f"{len(validation_windows)}"
)

print(
    "Preprocessing systems checked: "
    f"{len(pipeline_builders)}"
)

print(
    "Failed transformation contexts: "
    f"{failed_transformation_contexts}"
)

print(
    "Missing cells after preprocessing: "
    f"{total_missing_cells_after_preprocessing}"
)

print(
    "Infinite cells after preprocessing: "
    f"{total_infinite_cells_after_preprocessing}"
)

print(
    "\nValidation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print("\nTransformation audit:")
display(transformation_audit_df)

print("\nGenerated-feature count summary:")
display(feature_count_summary_df)

print("\nPart 4 validation summary:")
display(part4_validation_df)

print("\nSaved outputs:")
print(preprocessing_policy_path)
print(transformation_audit_path)
print(generated_feature_register_path)
print(feature_count_summary_path)
print(part4_validation_path)


MODELLING STEP 1 PART 4: PASSED

Original approved predictors: 53
Generated model features: 114
Validation folds checked: 2
Preprocessing systems checked: 2
Failed transformation contexts: 0
Missing cells after preprocessing: 0
Infinite cells after preprocessing: 0

Validation checks passed: 13 of 13

Transformation audit:


,WindowID,PipelineType,TrainingRows,ValidationRows,TrainingEndDate,ValidationStartDate,ChronologicalOrderPassed,OriginalPredictorCount,GeneratedFeatureCount,TrainingOutputRows,ValidationOutputRows,TrainingMissingCells,ValidationMissingCells,TrainingInfiniteCells,ValidationInfiniteCells,DuplicateGeneratedFeatureNames,Passed
0,STANDARD_BACKTEST_FOLD_1,LINEAR_MODEL,22999,2540,2025-12-23,2026-01-05,True,53,114,22999,2540,0,0,0,0,0,True
1,STANDARD_BACKTEST_FOLD_1,TREE_MODEL,22999,2540,2025-12-23,2026-01-05,True,53,114,22999,2540,0,0,0,0,0,True
2,STANDARD_BACKTEST_FOLD_2,LINEAR_MODEL,25539,2540,2026-01-29,2026-01-30,True,53,114,25539,2540,0,0,0,0,0,True
3,STANDARD_BACKTEST_FOLD_2,TREE_MODEL,25539,2540,2026-01-29,2026-01-30,True,53,114,25539,2540,0,0,0,0,0,True



Generated-feature count summary:


,WindowID,PipelineType,GeneratedFeatureCount,TrainingMissingCells,ValidationMissingCells,TrainingInfiniteCells,ValidationInfiniteCells,Passed
0,STANDARD_BACKTEST_FOLD_1,LINEAR_MODEL,114,0,0,0,0,True
1,STANDARD_BACKTEST_FOLD_1,TREE_MODEL,114,0,0,0,0,True
2,STANDARD_BACKTEST_FOLD_2,LINEAR_MODEL,114,0,0,0,0,True
3,STANDARD_BACKTEST_FOLD_2,TREE_MODEL,114,0,0,0,0,True



Part 4 validation summary:


,Check,Expected,Actual,Passed
0,Approved input predictor count,53,53,True
1,Validation fold count,2,2,True
2,Preprocessing pipeline types,"['LINEAR_MODEL', 'TREE_MODEL']","['LINEAR_MODEL', 'TREE_MODEL']",True
3,Transformation audit rows,4,4,True
4,Failed transformation contexts,0,0,True
5,Contexts with chronological overlap,0,0,True
6,Total transformed training missing cells,0,0,True
7,Total transformed validation missing cells,0,0,True
8,Total transformed training infinite cells,0,0,True
9,Total transformed validation infinite cells,0,0,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/04_preprocessing_policy.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/04_fold_preprocessing_transformation_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/04_generated_feature_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/04_generated_feature_count_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/04_preprocessing_transformer_validation_summary.csv


In [9]:
# ==============================================================
# MODELLING STEP 1, PART 5
# Training, prediction and evaluation framework
# ==============================================================

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# --------------------------------------------------------------
# 1. Confirm required objects from previous parts
# --------------------------------------------------------------

required_objects = [
    "training_df",
    "validation_df",
    "approved_predictors",
    "build_linear_preprocessor",
    "build_tree_preprocessor",
    "SETUP_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 1 Parts 1 to 4 first."
    )


# --------------------------------------------------------------
# 2. Define the locked target and required metadata
# --------------------------------------------------------------

TARGET_COLUMN = "TotalDemand"

PREDICTION_METADATA_COLUMNS = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "WindowID",
    "SplitContextID",
]

required_training_columns = (
    approved_predictors
    + [TARGET_COLUMN]
    + PREDICTION_METADATA_COLUMNS
)

required_validation_columns = (
    approved_predictors
    + [TARGET_COLUMN]
    + PREDICTION_METADATA_COLUMNS
)

missing_training_columns = [
    column
    for column in required_training_columns
    if column not in training_df.columns
]

missing_validation_columns = [
    column
    for column in required_validation_columns
    if column not in validation_df.columns
]

if missing_training_columns:
    raise KeyError(
        "Required columns missing from training data:\n"
        f"{missing_training_columns}"
    )

if missing_validation_columns:
    raise KeyError(
        "Required columns missing from validation data:\n"
        f"{missing_validation_columns}"
    )


# --------------------------------------------------------------
# 3. Ensure date columns are datetime
# --------------------------------------------------------------

training_df["Date"] = pd.to_datetime(
    training_df["Date"],
    errors="raise",
)

validation_df["Date"] = pd.to_datetime(
    validation_df["Date"],
    errors="raise",
)


# --------------------------------------------------------------
# 4. Build a complete model pipeline
# --------------------------------------------------------------

def build_model_pipeline(
    estimator,
    pipeline_type,
):
    """
    Combine a fresh preprocessing transformer with a fresh copy
    of a forecasting estimator.

    Parameters
    ----------
    estimator:
        Any compatible scikit-learn regression estimator.

    pipeline_type:
        Either 'LINEAR_MODEL' or 'TREE_MODEL'.

    Returns
    -------
    sklearn.pipeline.Pipeline
        An unfitted pipeline.
    """

    pipeline_type = str(
        pipeline_type
    ).strip().upper()

    if pipeline_type == "LINEAR_MODEL":
        preprocessor = build_linear_preprocessor()

    elif pipeline_type == "TREE_MODEL":
        preprocessor = build_tree_preprocessor()

    else:
        raise ValueError(
            "pipeline_type must be either "
            "'LINEAR_MODEL' or 'TREE_MODEL'."
        )

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                clone(estimator),
            ),
        ]
    )

    return model_pipeline


# --------------------------------------------------------------
# 5. Retrieve a pooled/global validation fold
# --------------------------------------------------------------

def get_global_fold_data(window_id):
    """
    Return all product rows belonging to one chronological
    model-selection fold.

    This is intended for pooled/global models trained across all
    127 products.
    """

    fold_training = training_df.loc[
        training_df["WindowID"] == window_id
    ].copy()

    fold_validation = validation_df.loc[
        validation_df["WindowID"] == window_id
    ].copy()

    if fold_training.empty:
        raise ValueError(
            f"No training rows found for {window_id}."
        )

    if fold_validation.empty:
        raise ValueError(
            f"No validation rows found for {window_id}."
        )

    training_end_date = fold_training["Date"].max()
    validation_start_date = fold_validation["Date"].min()

    if training_end_date >= validation_start_date:
        raise AssertionError(
            f"Chronological overlap found for {window_id}."
        )

    return {
        "window_id": window_id,
        "training_rows": fold_training,
        "validation_rows": fold_validation,
        "X_train": fold_training.loc[
            :,
            approved_predictors,
        ].copy(),
        "y_train": fold_training[
            TARGET_COLUMN
        ].copy(),
        "X_validation": fold_validation.loc[
            :,
            approved_predictors,
        ].copy(),
        "y_validation": fold_validation[
            TARGET_COLUMN
        ].copy(),
        "validation_metadata": fold_validation.loc[
            :,
            PREDICTION_METADATA_COLUMNS
            + [TARGET_COLUMN],
        ].copy(),
    }


# --------------------------------------------------------------
# 6. Retrieve an individual product-window context
# --------------------------------------------------------------

def get_product_context_data(split_context_id):
    """
    Return one product's training and validation observations for
    one approved chronological evaluation window.
    """

    context_training = training_df.loc[
        training_df["SplitContextID"]
        == split_context_id
    ].copy()

    context_validation = validation_df.loc[
        validation_df["SplitContextID"]
        == split_context_id
    ].copy()

    if context_training.empty:
        raise ValueError(
            "No training rows found for context:\n"
            f"{split_context_id}"
        )

    if context_validation.empty:
        raise ValueError(
            "No validation rows found for context:\n"
            f"{split_context_id}"
        )

    training_products = (
        context_training["CanonicalProductID"]
        .dropna()
        .unique()
        .tolist()
    )

    validation_products = (
        context_validation["CanonicalProductID"]
        .dropna()
        .unique()
        .tolist()
    )

    if len(training_products) != 1:
        raise AssertionError(
            "A product context must contain exactly one "
            "training product."
        )

    if len(validation_products) != 1:
        raise AssertionError(
            "A product context must contain exactly one "
            "validation product."
        )

    if training_products != validation_products:
        raise AssertionError(
            "Training and validation products do not match "
            f"for {split_context_id}."
        )

    if (
        context_training["Date"].max()
        >= context_validation["Date"].min()
    ):
        raise AssertionError(
            "Chronological overlap found for context:\n"
            f"{split_context_id}"
        )

    return {
        "split_context_id": split_context_id,
        "product_id": training_products[0],
        "training_rows": context_training,
        "validation_rows": context_validation,
        "X_train": context_training.loc[
            :,
            approved_predictors,
        ].copy(),
        "y_train": context_training[
            TARGET_COLUMN
        ].copy(),
        "X_validation": context_validation.loc[
            :,
            approved_predictors,
        ].copy(),
        "y_validation": context_validation[
            TARGET_COLUMN
        ].copy(),
        "validation_metadata": context_validation.loc[
            :,
            PREDICTION_METADATA_COLUMNS
            + [TARGET_COLUMN],
        ].copy(),
    }


# --------------------------------------------------------------
# 7. Prevent impossible negative-demand forecasts
# --------------------------------------------------------------

def clip_demand_predictions(raw_predictions):
    """
    Convert predictions to finite numeric values and clip negative
    demand predictions to zero.
    """

    predictions = np.asarray(
        raw_predictions,
        dtype=float,
    ).reshape(-1)

    if not np.isfinite(predictions).all():
        raise ValueError(
            "Predictions contain missing or infinite values."
        )

    return np.clip(
        predictions,
        a_min=0.0,
        a_max=None,
    )


# --------------------------------------------------------------
# 8. Create a standard prediction output
# --------------------------------------------------------------

def build_prediction_output(
    validation_metadata,
    raw_predictions,
    model_name,
    model_family,
):
    """
    Create the common prediction-table structure used by every
    baseline and advanced forecasting model.
    """

    metadata = validation_metadata.copy()

    required_columns = (
        PREDICTION_METADATA_COLUMNS
        + [TARGET_COLUMN]
    )

    missing_columns = [
        column
        for column in required_columns
        if column not in metadata.columns
    ]

    if missing_columns:
        raise KeyError(
            "Prediction metadata is missing columns:\n"
            f"{missing_columns}"
        )

    raw_predictions = np.asarray(
        raw_predictions,
        dtype=float,
    ).reshape(-1)

    if len(raw_predictions) != len(metadata):
        raise ValueError(
            "The number of predictions does not match the "
            "number of validation rows."
        )

    clipped_predictions = clip_demand_predictions(
        raw_predictions
    )

    prediction_output = metadata.loc[
        :,
        PREDICTION_METADATA_COLUMNS
        + [TARGET_COLUMN],
    ].copy()

    prediction_output = prediction_output.rename(
        columns={
            TARGET_COLUMN: "ActualDemand",
        }
    )

    prediction_output["RawPrediction"] = (
        raw_predictions
    )

    prediction_output["PredictedDemand"] = (
        clipped_predictions
    )

    prediction_output["PredictionClippedAtZero"] = (
        prediction_output["RawPrediction"] < 0
    )

    prediction_output["ModelName"] = str(
        model_name
    )

    prediction_output["ModelFamily"] = str(
        model_family
    )

    prediction_output["ForecastError"] = (
        prediction_output["PredictedDemand"]
        - prediction_output["ActualDemand"]
    )

    prediction_output["AbsoluteError"] = (
        prediction_output["ForecastError"].abs()
    )

    prediction_output["SquaredError"] = (
        prediction_output["ForecastError"] ** 2
    )

    return prediction_output


# --------------------------------------------------------------
# 9. Define zero-safe evaluation metrics
# --------------------------------------------------------------

def calculate_forecast_metrics(
    actual_values,
    predicted_values,
):
    """
    Calculate common forecast metrics.

    WAPE is recorded as NaN when total actual demand is zero,
    because division by zero would otherwise occur.
    """

    actual = np.asarray(
        actual_values,
        dtype=float,
    ).reshape(-1)

    predicted = np.asarray(
        predicted_values,
        dtype=float,
    ).reshape(-1)

    if len(actual) != len(predicted):
        raise ValueError(
            "Actual and predicted arrays must have equal length."
        )

    if len(actual) == 0:
        raise ValueError(
            "Metric calculation received no observations."
        )

    if not np.isfinite(actual).all():
        raise ValueError(
            "Actual values contain missing or infinite values."
        )

    if not np.isfinite(predicted).all():
        raise ValueError(
            "Predicted values contain missing or infinite values."
        )

    if (actual < 0).any():
        raise ValueError(
            "Actual demand cannot be negative."
        )

    if (predicted < 0).any():
        raise ValueError(
            "Predicted demand cannot be negative."
        )

    errors = predicted - actual
    absolute_errors = np.abs(errors)

    actual_total = float(
        actual.sum()
    )

    predicted_total = float(
        predicted.sum()
    )

    mae = float(
        mean_absolute_error(
            actual,
            predicted,
        )
    )

    rmse = float(
        np.sqrt(
            mean_squared_error(
                actual,
                predicted,
            )
        )
    )

    if actual_total > 0:
        wape = float(
            absolute_errors.sum()
            / actual_total
            * 100
        )
    else:
        wape = np.nan

    overforecast_units = float(
        np.maximum(
            errors,
            0,
        ).sum()
    )

    underforecast_units = float(
        np.maximum(
            -errors,
            0,
        ).sum()
    )

    return {
        "ObservationCount": int(len(actual)),
        "ActualDemandTotal": actual_total,
        "PredictedDemandTotal": predicted_total,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE_Percent": wape,
        "MeanBias": float(errors.mean()),
        "TotalBias": float(errors.sum()),
        "OverforecastUnits": overforecast_units,
        "UnderforecastUnits": underforecast_units,
        "ZeroActualObservationCount": int(
            (actual == 0).sum()
        ),
    }


# --------------------------------------------------------------
# 10. Audit all global folds
# --------------------------------------------------------------

global_fold_records = []

validation_windows = sorted(
    validation_df["WindowID"]
    .dropna()
    .unique()
    .tolist()
)

for window_id in validation_windows:

    fold_data = get_global_fold_data(
        window_id
    )

    fold_training = fold_data[
        "training_rows"
    ]

    fold_validation = fold_data[
        "validation_rows"
    ]

    global_fold_records.append(
        {
            "WindowID": window_id,
            "TrainingRows":
                len(fold_training),
            "ValidationRows":
                len(fold_validation),
            "TrainingProducts":
                fold_training[
                    "CanonicalProductID"
                ].nunique(),
            "ValidationProducts":
                fold_validation[
                    "CanonicalProductID"
                ].nunique(),
            "TrainingEndDate":
                fold_training["Date"].max().date(),
            "ValidationStartDate":
                fold_validation["Date"].min().date(),
            "ValidationEndDate":
                fold_validation["Date"].max().date(),
            "ChronologicalOrderPassed": bool(
                fold_training["Date"].max()
                < fold_validation["Date"].min()
            ),
        }
    )

global_fold_register_df = pd.DataFrame(
    global_fold_records
)


# --------------------------------------------------------------
# 11. Audit all 254 product-window contexts
# --------------------------------------------------------------

training_context_ids = set(
    training_df["SplitContextID"]
    .dropna()
    .unique()
)

validation_context_ids = set(
    validation_df["SplitContextID"]
    .dropna()
    .unique()
)

all_context_ids = sorted(
    training_context_ids
    | validation_context_ids
)

product_context_records = []

for split_context_id in all_context_ids:

    context_training = training_df.loc[
        training_df["SplitContextID"]
        == split_context_id
    ].copy()

    context_validation = validation_df.loc[
        validation_df["SplitContextID"]
        == split_context_id
    ].copy()

    training_present = not context_training.empty
    validation_present = not context_validation.empty

    training_product_count = int(
        context_training[
            "CanonicalProductID"
        ].nunique()
    )

    validation_product_count = int(
        context_validation[
            "CanonicalProductID"
        ].nunique()
    )

    training_window_count = int(
        context_training[
            "WindowID"
        ].nunique()
    )

    validation_window_count = int(
        context_validation[
            "WindowID"
        ].nunique()
    )

    same_product = False
    same_window = False
    chronological_order_passed = False

    product_id = None
    window_id = None

    if training_present and validation_present:

        training_products = sorted(
            context_training[
                "CanonicalProductID"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        validation_products = sorted(
            context_validation[
                "CanonicalProductID"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        training_windows = sorted(
            context_training[
                "WindowID"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        validation_windows_context = sorted(
            context_validation[
                "WindowID"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        same_product = (
            training_products
            == validation_products
        )

        same_window = (
            training_windows
            == validation_windows_context
        )

        chronological_order_passed = bool(
            context_training["Date"].max()
            < context_validation["Date"].min()
        )

        if len(training_products) == 1:
            product_id = training_products[0]

        if len(training_windows) == 1:
            window_id = training_windows[0]

    context_passed = bool(
        training_present
        and validation_present
        and training_product_count == 1
        and validation_product_count == 1
        and training_window_count == 1
        and validation_window_count == 1
        and same_product
        and same_window
        and chronological_order_passed
        and len(context_validation) == 20
    )

    product_context_records.append(
        {
            "SplitContextID": split_context_id,
            "CanonicalProductID": product_id,
            "WindowID": window_id,
            "TrainingPresent": training_present,
            "ValidationPresent": validation_present,
            "TrainingRows":
                len(context_training),
            "ValidationRows":
                len(context_validation),
            "TrainingProductCount":
                training_product_count,
            "ValidationProductCount":
                validation_product_count,
            "TrainingWindowCount":
                training_window_count,
            "ValidationWindowCount":
                validation_window_count,
            "SameProduct":
                same_product,
            "SameWindow":
                same_window,
            "ChronologicalOrderPassed":
                chronological_order_passed,
            "Passed":
                context_passed,
        }
    )

product_context_register_df = pd.DataFrame(
    product_context_records
)


# --------------------------------------------------------------
# 12. Construct model pipeline templates without fitting
# --------------------------------------------------------------

pipeline_template_records = []

template_estimators = [
    {
        "TemplateName": "RIDGE_TEMPLATE",
        "PipelineType": "LINEAR_MODEL",
        "Estimator": Ridge(
            alpha=1.0,
        ),
    },
    {
        "TemplateName": "RANDOM_FOREST_TEMPLATE",
        "PipelineType": "TREE_MODEL",
        "Estimator": RandomForestRegressor(
            n_estimators=10,
            random_state=42,
            n_jobs=-1,
        ),
    },
]

for template_definition in template_estimators:

    template_name = template_definition[
        "TemplateName"
    ]

    pipeline_type = template_definition[
        "PipelineType"
    ]

    estimator = template_definition[
        "Estimator"
    ]

    pipeline_created = True
    error_message = ""

    try:
        pipeline = build_model_pipeline(
            estimator=estimator,
            pipeline_type=pipeline_type,
        )

        pipeline_steps = list(
            pipeline.named_steps.keys()
        )

        expected_steps_present = (
            pipeline_steps
            == [
                "preprocessor",
                "model",
            ]
        )

    except Exception as error:
        pipeline_created = False
        expected_steps_present = False
        pipeline_steps = []
        error_message = str(error)

    pipeline_template_records.append(
        {
            "TemplateName": template_name,
            "PipelineType": pipeline_type,
            "EstimatorClass":
                estimator.__class__.__name__,
            "PipelineCreated":
                pipeline_created,
            "PipelineSteps":
                " | ".join(pipeline_steps),
            "ExpectedStepsPresent":
                expected_steps_present,
            "FittedOnEdenData": False,
            "ErrorMessage":
                error_message,
            "Passed": bool(
                pipeline_created
                and expected_steps_present
            ),
        }
    )

model_pipeline_template_register_df = pd.DataFrame(
    pipeline_template_records
)


# --------------------------------------------------------------
# 13. Test the output and metric functions with artificial values
# --------------------------------------------------------------

synthetic_metadata_df = pd.DataFrame(
    {
        "Date": pd.date_range(
            "2026-01-01",
            periods=4,
            freq="D",
        ),
        "CanonicalProductID": [
            "SYNTHETIC_PRODUCT"
        ] * 4,
        "CanonicalProductName": [
            "SYNTHETIC PRODUCT"
        ] * 4,
        "WindowID": [
            "SYNTHETIC_WINDOW"
        ] * 4,
        "SplitContextID": [
            "SYNTHETIC_CONTEXT"
        ] * 4,
        "TotalDemand": [
            0,
            2,
            4,
            0,
        ],
    }
)

synthetic_raw_predictions = np.array(
    [
        -1,
        3,
        2,
        0,
    ],
    dtype=float,
)

synthetic_prediction_output_df = (
    build_prediction_output(
        validation_metadata=synthetic_metadata_df,
        raw_predictions=synthetic_raw_predictions,
        model_name="SYNTHETIC_TEST",
        model_family="UTILITY_VALIDATION",
    )
)

synthetic_metrics = calculate_forecast_metrics(
    actual_values=(
        synthetic_prediction_output_df[
            "ActualDemand"
        ]
    ),
    predicted_values=(
        synthetic_prediction_output_df[
            "PredictedDemand"
        ]
    ),
)

expected_rmse = float(
    np.sqrt(1.25)
)

synthetic_metric_tests = {
    "MAE": np.isclose(
        synthetic_metrics["MAE"],
        0.75,
    ),
    "RMSE": np.isclose(
        synthetic_metrics["RMSE"],
        expected_rmse,
    ),
    "WAPE": np.isclose(
        synthetic_metrics["WAPE_Percent"],
        50.0,
    ),
    "MeanBias": np.isclose(
        synthetic_metrics["MeanBias"],
        -0.25,
    ),
    "TotalBias": np.isclose(
        synthetic_metrics["TotalBias"],
        -1.0,
    ),
    "OverforecastUnits": np.isclose(
        synthetic_metrics["OverforecastUnits"],
        1.0,
    ),
    "UnderforecastUnits": np.isclose(
        synthetic_metrics["UnderforecastUnits"],
        2.0,
    ),
}

synthetic_metric_test_df = pd.DataFrame(
    [
        {
            "MetricTest": metric_name,
            "Passed": bool(test_passed),
        }
        for metric_name, test_passed
        in synthetic_metric_tests.items()
    ]
)

all_zero_metrics = calculate_forecast_metrics(
    actual_values=[
        0,
        0,
        0,
    ],
    predicted_values=[
        0,
        1,
        0,
    ],
)

all_zero_wape_is_undefined = bool(
    np.isnan(
        all_zero_metrics["WAPE_Percent"]
    )
)


# --------------------------------------------------------------
# 14. Define the frozen metric policy
# --------------------------------------------------------------

metric_policy_df = pd.DataFrame(
    [
        {
            "Metric": "MAE",
            "PrimaryOrSupporting": "PRIMARY",
            "Interpretation":
                "Average absolute forecast error in demand units.",
            "ZeroSafe": True,
        },
        {
            "Metric": "RMSE",
            "PrimaryOrSupporting": "SUPPORTING",
            "Interpretation":
                "Penalises larger individual forecast errors.",
            "ZeroSafe": True,
        },
        {
            "Metric": "WAPE",
            "PrimaryOrSupporting": "PRIMARY",
            "Interpretation":
                "Total absolute error divided by total actual demand.",
            "ZeroSafe":
                "DEFINED_ONLY_WHEN_ACTUAL_TOTAL_IS_POSITIVE",
        },
        {
            "Metric": "MEAN_BIAS",
            "PrimaryOrSupporting": "SUPPORTING",
            "Interpretation":
                "Positive means overforecasting; negative means underforecasting.",
            "ZeroSafe": True,
        },
        {
            "Metric": "OVERFORECAST_UNITS",
            "PrimaryOrSupporting": "SUPPORTING",
            "Interpretation":
                "Total predicted units above actual demand.",
            "ZeroSafe": True,
        },
        {
            "Metric": "UNDERFORECAST_UNITS",
            "PrimaryOrSupporting": "SUPPORTING",
            "Interpretation":
                "Total actual units above predicted demand.",
            "ZeroSafe": True,
        },
    ]
)


# --------------------------------------------------------------
# 15. Define the standard prediction-output schema
# --------------------------------------------------------------

prediction_output_schema_df = pd.DataFrame(
    [
        {
            "Column": column,
            "ColumnOrder": position,
        }
        for position, column in enumerate(
            synthetic_prediction_output_df.columns,
            start=1,
        )
    ]
)


# --------------------------------------------------------------
# 16. Build the Part 5 validation summary
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Global validation fold count",
    len(global_fold_register_df),
    2,
)

add_check(
    "Global folds with chronological failure",
    int(
        (
            ~global_fold_register_df[
                "ChronologicalOrderPassed"
            ]
        ).sum()
    ),
    0,
)

add_check(
    "Products per global validation fold",
    sorted(
        global_fold_register_df[
            "ValidationProducts"
        ].unique().tolist()
    ),
    [127],
)

add_check(
    "Product-window context count",
    len(product_context_register_df),
    254,
)

add_check(
    "Failed product-window contexts",
    int(
        (
            ~product_context_register_df["Passed"]
        ).sum()
    ),
    0,
)

add_check(
    "Contexts without training rows",
    int(
        (
            ~product_context_register_df[
                "TrainingPresent"
            ]
        ).sum()
    ),
    0,
)

add_check(
    "Contexts without validation rows",
    int(
        (
            ~product_context_register_df[
                "ValidationPresent"
            ]
        ).sum()
    ),
    0,
)

add_check(
    "Contexts without 20 validation rows",
    int(
        (
            product_context_register_df[
                "ValidationRows"
            ] != 20
        ).sum()
    ),
    0,
)

add_check(
    "Model pipeline templates",
    len(model_pipeline_template_register_df),
    2,
)

add_check(
    "Failed model pipeline templates",
    int(
        (
            ~model_pipeline_template_register_df[
                "Passed"
            ]
        ).sum()
    ),
    0,
)

add_check(
    "Model templates fitted on Eden data",
    int(
        model_pipeline_template_register_df[
            "FittedOnEdenData"
        ].sum()
    ),
    0,
)

add_check(
    "Synthetic negative predictions before clipping",
    int(
        (
            synthetic_prediction_output_df[
                "RawPrediction"
            ] < 0
        ).sum()
    ),
    1,
)

add_check(
    "Synthetic negative predictions after clipping",
    int(
        (
            synthetic_prediction_output_df[
                "PredictedDemand"
            ] < 0
        ).sum()
    ),
    0,
)

add_check(
    "Synthetic metric test failures",
    int(
        (
            ~synthetic_metric_test_df["Passed"]
        ).sum()
    ),
    0,
)

add_check(
    "All-zero actual WAPE is undefined",
    all_zero_wape_is_undefined,
    True,
)

part5_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 17. Stop if validation fails
# --------------------------------------------------------------

failed_checks_df = part5_validation_df.loc[
    ~part5_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print("\nFAILED STEP 1 PART 5 CHECKS")
    display(failed_checks_df)

    raise AssertionError(
        "Modelling Step 1 Part 5 failed. "
        "Do not begin baseline model development."
    )


# --------------------------------------------------------------
# 18. Save Part 5 outputs
# --------------------------------------------------------------

global_fold_register_path = (
    SETUP_OUTPUT_DIR
    / "05_global_model_selection_fold_register.csv"
)

product_context_register_path = (
    SETUP_OUTPUT_DIR
    / "05_product_window_context_register.csv"
)

pipeline_template_register_path = (
    SETUP_OUTPUT_DIR
    / "05_model_pipeline_template_register.csv"
)

metric_policy_path = (
    SETUP_OUTPUT_DIR
    / "05_forecast_metric_policy.csv"
)

prediction_schema_path = (
    SETUP_OUTPUT_DIR
    / "05_prediction_output_schema.csv"
)

synthetic_metric_test_path = (
    SETUP_OUTPUT_DIR
    / "05_metric_function_test.csv"
)

part5_validation_path = (
    SETUP_OUTPUT_DIR
    / "05_training_prediction_evaluation_validation_summary.csv"
)

global_fold_register_df.to_csv(
    global_fold_register_path,
    index=False,
)

product_context_register_df.to_csv(
    product_context_register_path,
    index=False,
)

model_pipeline_template_register_df.to_csv(
    pipeline_template_register_path,
    index=False,
)

metric_policy_df.to_csv(
    metric_policy_path,
    index=False,
)

prediction_output_schema_df.to_csv(
    prediction_schema_path,
    index=False,
)

synthetic_metric_test_df.to_csv(
    synthetic_metric_test_path,
    index=False,
)

part5_validation_df.to_csv(
    part5_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 19. Display final results
# --------------------------------------------------------------

failed_context_count = int(
    (
        ~product_context_register_df["Passed"]
    ).sum()
)

failed_pipeline_template_count = int(
    (
        ~model_pipeline_template_register_df[
            "Passed"
        ]
    ).sum()
)

failed_metric_test_count = int(
    (
        ~synthetic_metric_test_df["Passed"]
    ).sum()
)

passed_check_count = int(
    part5_validation_df["Passed"].sum()
)

total_check_count = int(
    len(part5_validation_df)
)

print("\n" + "=" * 72)
print("MODELLING STEP 1 PART 5: PASSED")
print("=" * 72)

print(
    "\nGlobal validation folds: "
    f"{len(global_fold_register_df)}"
)

print(
    "Product-window contexts: "
    f"{len(product_context_register_df)}"
)

print(
    "Failed product-window contexts: "
    f"{failed_context_count}"
)

print(
    "Model pipeline templates created: "
    f"{len(model_pipeline_template_register_df)}"
)

print(
    "Failed pipeline templates: "
    f"{failed_pipeline_template_count}"
)

print(
    "Synthetic metric-test failures: "
    f"{failed_metric_test_count}"
)

print(
    "All-zero actual WAPE handled safely: "
    f"{all_zero_wape_is_undefined}"
)

print(
    "\nValidation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print("\nGlobal-fold register:")
display(global_fold_register_df)

print("\nModel-pipeline template register:")
display(model_pipeline_template_register_df)

print("\nSynthetic metric tests:")
display(synthetic_metric_test_df)

print("\nPart 5 validation summary:")
display(part5_validation_df)

print("\nSaved outputs:")
print(global_fold_register_path)
print(product_context_register_path)
print(pipeline_template_register_path)
print(metric_policy_path)
print(prediction_schema_path)
print(synthetic_metric_test_path)
print(part5_validation_path)


MODELLING STEP 1 PART 5: PASSED

Global validation folds: 2
Product-window contexts: 254
Failed product-window contexts: 0
Model pipeline templates created: 2
Failed pipeline templates: 0
Synthetic metric-test failures: 0
All-zero actual WAPE handled safely: True

Validation checks passed: 15 of 15

Global-fold register:


,WindowID,TrainingRows,ValidationRows,TrainingProducts,ValidationProducts,TrainingEndDate,ValidationStartDate,ValidationEndDate,ChronologicalOrderPassed
0,STANDARD_BACKTEST_FOLD_1,22999,2540,127,127,2025-12-23,2026-01-05,2026-01-29,True
1,STANDARD_BACKTEST_FOLD_2,25539,2540,127,127,2026-01-29,2026-01-30,2026-02-27,True



Model-pipeline template register:


,TemplateName,PipelineType,EstimatorClass,PipelineCreated,PipelineSteps,ExpectedStepsPresent,FittedOnEdenData,ErrorMessage,Passed
0,RIDGE_TEMPLATE,LINEAR_MODEL,Ridge,True,preprocessor | model,True,False,,True
1,RANDOM_FOREST_TEMPLATE,TREE_MODEL,RandomForestRegressor,True,preprocessor | model,True,False,,True



Synthetic metric tests:


,MetricTest,Passed
0,MAE,True
1,RMSE,True
2,WAPE,True
3,MeanBias,True
4,TotalBias,True
5,OverforecastUnits,True
6,UnderforecastUnits,True



Part 5 validation summary:


,Check,Expected,Actual,Passed
0,Global validation fold count,2,2,True
1,Global folds with chronological failure,0,0,True
2,Products per global validation fold,[127],[127],True
3,Product-window context count,254,254,True
4,Failed product-window contexts,0,0,True
5,Contexts without training rows,0,0,True
6,Contexts without validation rows,0,0,True
7,Contexts without 20 validation rows,0,0,True
8,Model pipeline templates,2,2,True
9,Failed model pipeline templates,0,0,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/05_global_model_selection_fold_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/05_product_window_context_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/05_model_pipeline_template_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/05_forecast_metric_policy.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/05_prediction_output_schema.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/05_metric_function_test.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/05_training_prediction_evaluation_validation_summary.csv


In [11]:
# ==============================================================
# MODELLING STEP 1, PART 6
# Final modelling-pipeline completion audit
# ==============================================================

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from IPython.display import display
import sklearn


# --------------------------------------------------------------
# 1. Confirm required objects from Parts 1 to 5
# --------------------------------------------------------------

required_objects = [
    "training_df",
    "validation_df",
    "feature_contract_df",
    "protocol_contract_df",
    "approved_predictors",
    "current_date_predictors",
    "historical_predictors",
    "numeric_predictors",
    "categorical_predictors",
    "binary_predictors",
    "build_linear_preprocessor",
    "build_tree_preprocessor",
    "build_model_pipeline",
    "get_global_fold_data",
    "get_product_context_data",
    "build_prediction_output",
    "calculate_forecast_metrics",
    "transformation_audit_df",
    "global_fold_register_df",
    "product_context_register_df",
    "model_pipeline_template_register_df",
    "SETUP_OUTPUT_DIR",
]

missing_required_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_required_objects:
    raise RuntimeError(
        "The following required objects are unavailable:\n"
        f"{missing_required_objects}\n\n"
        "Run Modelling Step 1 Parts 1 to 5 before this cell."
    )


# --------------------------------------------------------------
# 2. Define all expected Step 1 audit outputs
# --------------------------------------------------------------

expected_step1_output_files = [
    # Part 1
    "01_model_dataset_loading_summary.csv",
    "01_model_entry_validation_summary.csv",
    "01_model_input_access_register.csv",

    # Part 2
    "02_frozen_predictor_register.csv",
    "02_predictor_dtype_audit.csv",
    "02_predictor_missingness_audit.csv",
    "02_raw_predictor_type_summary.csv",
    "02_historical_feature_readiness_summary.csv",
    "02_predictor_structure_validation_summary.csv",

    # Part 3
    "03_semantic_predictor_role_register.csv",
    "03_preprocessing_role_summary.csv",
    "03_numeric_predictor_quality_audit.csv",
    "03_categorical_level_coverage_audit.csv",
    "03_binary_predictor_quality_audit.csv",
    "03_semantic_role_validation_summary.csv",

    # Part 4
    "04_preprocessing_policy.csv",
    "04_fold_preprocessing_transformation_audit.csv",
    "04_generated_feature_register.csv",
    "04_generated_feature_count_summary.csv",
    "04_preprocessing_transformer_validation_summary.csv",

    # Part 5
    "05_global_model_selection_fold_register.csv",
    "05_product_window_context_register.csv",
    "05_model_pipeline_template_register.csv",
    "05_forecast_metric_policy.csv",
    "05_prediction_output_schema.csv",
    "05_metric_function_test.csv",
    "05_training_prediction_evaluation_validation_summary.csv",
]


# --------------------------------------------------------------
# 3. Create the output-file existence audit
# --------------------------------------------------------------

output_file_audit_records = []

for file_name in expected_step1_output_files:

    file_path = Path(SETUP_OUTPUT_DIR) / file_name

    output_file_audit_records.append(
        {
            "FileName": file_name,
            "FilePath": str(file_path),
            "Exists": file_path.exists(),
            "FileSizeBytes": (
                file_path.stat().st_size
                if file_path.exists()
                else 0
            ),
            "NonEmpty": bool(
                file_path.exists()
                and file_path.stat().st_size > 0
            ),
        }
    )

output_file_audit_df = pd.DataFrame(
    output_file_audit_records
)

missing_output_file_count = int(
    (~output_file_audit_df["Exists"]).sum()
)

empty_output_file_count = int(
    (
        output_file_audit_df["Exists"]
        & ~output_file_audit_df["NonEmpty"]
    ).sum()
)


# --------------------------------------------------------------
# 4. Load the saved validation-summary files
# --------------------------------------------------------------

validation_summary_files = {
    "PART_1":
        "01_model_entry_validation_summary.csv",

    "PART_2":
        "02_predictor_structure_validation_summary.csv",

    "PART_3":
        "03_semantic_role_validation_summary.csv",

    "PART_4":
        "04_preprocessing_transformer_validation_summary.csv",

    "PART_5":
        "05_training_prediction_evaluation_validation_summary.csv",
}

part_validation_records = []

for part_name, file_name in validation_summary_files.items():

    file_path = Path(SETUP_OUTPUT_DIR) / file_name

    if not file_path.exists():
        part_validation_records.append(
            {
                "Part": part_name,
                "ValidationFile": file_name,
                "CheckCount": 0,
                "PassedCheckCount": 0,
                "FailedCheckCount": 1,
                "PartPassed": False,
            }
        )

        continue

    part_validation_file_df = pd.read_csv(
        file_path
    )

    if "Passed" not in part_validation_file_df.columns:
        raise KeyError(
            f"'Passed' column missing from {file_name}."
        )

    passed_values = (
        part_validation_file_df["Passed"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
    )

    if passed_values.isna().any():
        raise ValueError(
            "Unexpected Passed values found in "
            f"{file_name}."
        )

    passed_count = int(
        passed_values.sum()
    )

    total_count = int(
        len(passed_values)
    )

    failed_count = int(
        total_count - passed_count
    )

    part_validation_records.append(
        {
            "Part": part_name,
            "ValidationFile": file_name,
            "CheckCount": total_count,
            "PassedCheckCount": passed_count,
            "FailedCheckCount": failed_count,
            "PartPassed": bool(
                failed_count == 0
                and total_count > 0
            ),
        }
    )

part_validation_register_df = pd.DataFrame(
    part_validation_records
)


# --------------------------------------------------------------
# 5. Confirm final-test access protection
# --------------------------------------------------------------

input_access_register_path = (
    Path(SETUP_OUTPUT_DIR)
    / "01_model_input_access_register.csv"
)

input_access_register_df = pd.read_csv(
    input_access_register_path
)

protected_access_rows_df = input_access_register_df.loc[
    input_access_register_df["AccessPolicy"]
    == "DO_NOT_LOAD_UNTIL_MODEL_SELECTION_IS_LOCKED"
].copy()

protected_loaded_values = (
    protected_access_rows_df["LoadedInStep1Part1"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        }
    )
)

if protected_loaded_values.isna().any():
    raise ValueError(
        "Unexpected protected-file access values were found."
    )

protected_final_test_file_count = int(
    len(protected_access_rows_df)
)

protected_final_test_files_loaded = int(
    protected_loaded_values.sum()
)


# --------------------------------------------------------------
# 6. Confirm the generated feature structure
# --------------------------------------------------------------

generated_feature_count_values = sorted(
    transformation_audit_df[
        "GeneratedFeatureCount"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

if len(generated_feature_count_values) == 1:
    generated_feature_count = (
        generated_feature_count_values[0]
    )
else:
    generated_feature_count = None


# --------------------------------------------------------------
# 7. Confirm chronological structure
# --------------------------------------------------------------

validation_window_count = int(
    validation_df["WindowID"].nunique()
)

split_context_count = int(
    validation_df["SplitContextID"].nunique()
)

training_product_count = int(
    training_df["CanonicalProductID"].nunique()
)

validation_product_count = int(
    validation_df["CanonicalProductID"].nunique()
)

failed_global_fold_count = int(
    (
        ~global_fold_register_df[
            "ChronologicalOrderPassed"
        ]
    ).sum()
)

failed_product_context_count = int(
    (
        ~product_context_register_df["Passed"]
    ).sum()
)

failed_preprocessing_context_count = int(
    (
        ~transformation_audit_df["Passed"]
    ).sum()
)

failed_pipeline_template_count = int(
    (
        ~model_pipeline_template_register_df[
            "Passed"
        ]
    ).sum()
)


# --------------------------------------------------------------
# 8. Confirm function availability
# --------------------------------------------------------------

required_framework_functions = {
    "build_linear_preprocessor":
        build_linear_preprocessor,

    "build_tree_preprocessor":
        build_tree_preprocessor,

    "build_model_pipeline":
        build_model_pipeline,

    "get_global_fold_data":
        get_global_fold_data,

    "get_product_context_data":
        get_product_context_data,

    "build_prediction_output":
        build_prediction_output,

    "calculate_forecast_metrics":
        calculate_forecast_metrics,
}

function_register_records = []

for function_name, function_object in (
    required_framework_functions.items()
):

    function_register_records.append(
        {
            "FunctionName": function_name,
            "Callable": callable(function_object),
        }
    )

framework_function_register_df = pd.DataFrame(
    function_register_records
)

missing_or_invalid_function_count = int(
    (
        ~framework_function_register_df["Callable"]
    ).sum()
)


# --------------------------------------------------------------
# 9. Record modelling environment information
# --------------------------------------------------------------

environment_register_df = pd.DataFrame(
    [
        {
            "Component": "PythonRunTimestamp",
            "VersionOrValue":
                datetime.now().isoformat(
                    timespec="seconds"
                ),
        },
        {
            "Component": "pandas",
            "VersionOrValue": pd.__version__,
        },
        {
            "Component": "numpy",
            "VersionOrValue": np.__version__,
        },
        {
            "Component": "scikit-learn",
            "VersionOrValue": sklearn.__version__,
        },
        {
            "Component": "RandomStatePolicy",
            "VersionOrValue":
                "random_state=42 where applicable",
        },
        {
            "Component": "ForecastTarget",
            "VersionOrValue": "TotalDemand",
        },
        {
            "Component": "ForecastMode",
            "VersionOrValue":
                "ROLLING_ONE_OPERATING_DAY_AHEAD",
        },
    ]
)


# --------------------------------------------------------------
# 10. Build final Step 1 validation checks
# --------------------------------------------------------------

final_validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one final Step 1 validation check.
    """

    if passed is None:
        passed = actual_value == expected_value

    final_validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Step 1 validation parts",
    len(part_validation_register_df),
    5,
)

add_check(
    "Failed Step 1 parts",
    int(
        (
            ~part_validation_register_df[
                "PartPassed"
            ]
        ).sum()
    ),
    0,
)

add_check(
    "Expected audit-output files",
    len(expected_step1_output_files),
    27,
)

add_check(
    "Missing audit-output files",
    missing_output_file_count,
    0,
)

add_check(
    "Empty audit-output files",
    empty_output_file_count,
    0,
)

add_check(
    "Approved predictors",
    len(approved_predictors),
    53,
)

add_check(
    "Current-date predictors",
    len(current_date_predictors),
    22,
)

add_check(
    "Historical predictors",
    len(historical_predictors),
    31,
)

add_check(
    "Numerical predictors",
    len(numeric_predictors),
    43,
)

add_check(
    "Categorical predictors",
    len(categorical_predictors),
    7,
)

add_check(
    "Binary predictors",
    len(binary_predictors),
    3,
)

add_check(
    "Generated model features",
    generated_feature_count,
    114,
)

add_check(
    "Training rows",
    len(training_df),
    48_538,
)

add_check(
    "Validation rows",
    len(validation_df),
    5_080,
)

add_check(
    "Training products",
    training_product_count,
    127,
)

add_check(
    "Validation products",
    validation_product_count,
    127,
)

add_check(
    "Validation windows",
    validation_window_count,
    2,
)

add_check(
    "Product-window contexts",
    split_context_count,
    254,
)

add_check(
    "Failed global chronological folds",
    failed_global_fold_count,
    0,
)

add_check(
    "Failed product-window contexts",
    failed_product_context_count,
    0,
)

add_check(
    "Failed preprocessing contexts",
    failed_preprocessing_context_count,
    0,
)

add_check(
    "Failed pipeline templates",
    failed_pipeline_template_count,
    0,
)

add_check(
    "Missing or invalid framework functions",
    missing_or_invalid_function_count,
    0,
)

add_check(
    "Protected final-test files registered",
    protected_final_test_file_count,
    3,
)

add_check(
    "Protected final-test files loaded",
    protected_final_test_files_loaded,
    0,
)

add_check(
    "Forecast target",
    "TotalDemand",
    "TotalDemand",
)

step1_final_validation_df = pd.DataFrame(
    final_validation_checks
)


# --------------------------------------------------------------
# 11. Stop if any final validation fails
# --------------------------------------------------------------

failed_final_checks_df = (
    step1_final_validation_df.loc[
        ~step1_final_validation_df["Passed"]
    ].copy()
)

if not failed_final_checks_df.empty:

    print("\nFAILED MODELLING STEP 1 COMPLETION CHECKS")
    display(failed_final_checks_df)

    raise AssertionError(
        "Modelling Step 1 is not complete. "
        "Do not begin baseline forecasting."
    )


# --------------------------------------------------------------
# 12. Create the final Step 1 completion summary
# --------------------------------------------------------------

step1_completion_summary_df = pd.DataFrame(
    [
        {
            "Phase": "MODELLING_STEP_1",
            "Status": "COMPLETED_AND_VALIDATED",
            "TrainingRows": len(training_df),
            "ValidationRows": len(validation_df),
            "ModelSelectionProducts":
                validation_product_count,
            "ValidationWindows":
                validation_window_count,
            "ProductWindowContexts":
                split_context_count,
            "ApprovedPredictors":
                len(approved_predictors),
            "GeneratedModelFeatures":
                generated_feature_count,
            "NumericalPredictors":
                len(numeric_predictors),
            "CategoricalPredictors":
                len(categorical_predictors),
            "BinaryPredictors":
                len(binary_predictors),
            "PreprocessingSystems": 2,
            "ProtectedFinalTestFilesLoaded":
                protected_final_test_files_loaded,
            "NextPhase":
                "MODELLING_STEP_2_BASELINE_FORECASTS",
        }
    ]
)


# --------------------------------------------------------------
# 13. Save final Step 1 outputs
# --------------------------------------------------------------

output_file_audit_path = (
    Path(SETUP_OUTPUT_DIR)
    / "06_step1_output_file_audit.csv"
)

part_validation_register_path = (
    Path(SETUP_OUTPUT_DIR)
    / "06_step1_part_validation_register.csv"
)

framework_function_register_path = (
    Path(SETUP_OUTPUT_DIR)
    / "06_framework_function_register.csv"
)

environment_register_path = (
    Path(SETUP_OUTPUT_DIR)
    / "06_modelling_environment_register.csv"
)

step1_final_validation_path = (
    Path(SETUP_OUTPUT_DIR)
    / "06_modelling_step1_final_validation_summary.csv"
)

step1_completion_summary_path = (
    Path(SETUP_OUTPUT_DIR)
    / "06_modelling_step1_completion_summary.csv"
)

step1_handoff_path = (
    Path(SETUP_OUTPUT_DIR)
    / "MODELLING_STEP1_HANDOFF.md"
)

output_file_audit_df.to_csv(
    output_file_audit_path,
    index=False,
)

part_validation_register_df.to_csv(
    part_validation_register_path,
    index=False,
)

framework_function_register_df.to_csv(
    framework_function_register_path,
    index=False,
)

environment_register_df.to_csv(
    environment_register_path,
    index=False,
)

step1_final_validation_df.to_csv(
    step1_final_validation_path,
    index=False,
)

step1_completion_summary_df.to_csv(
    step1_completion_summary_path,
    index=False,
)


# --------------------------------------------------------------
# 14. Create the Step 1 handover note
# --------------------------------------------------------------

handoff_text = f"""
# Modelling Step 1 Handoff

## Status

Completed and validated.

## Model-selection datasets

- Training rows: {len(training_df):,}
- Validation rows: {len(validation_df):,}
- Products: {validation_product_count}
- Validation windows: {validation_window_count}
- Product-window contexts: {split_context_count}

## Predictor contract

- Approved predictors: {len(approved_predictors)}
- Current-date predictors: {len(current_date_predictors)}
- Historical predictors: {len(historical_predictors)}
- Numerical predictors: {len(numeric_predictors)}
- Categorical predictors: {len(categorical_predictors)}
- Binary predictors: {len(binary_predictors)}
- Generated model features: {generated_feature_count}

## Preprocessing

Two leakage-safe preprocessing systems were created:

1. Linear-model preprocessing
   - Training-median numerical imputation
   - Missing-value indicators
   - Standard scaling
   - Categorical constant imputation
   - One-hot encoding
   - Binary passthrough

2. Tree-model preprocessing
   - Training-median numerical imputation
   - Missing-value indicators
   - No numerical scaling
   - Categorical constant imputation
   - One-hot encoding
   - Binary passthrough

Each preprocessing system must be fitted only on the relevant
chronological training rows.

## Forecast evaluation framework

Reusable functions were created for:

- Global-fold retrieval
- Product-context retrieval
- Model-pipeline construction
- Non-negative prediction handling
- Standard prediction outputs
- MAE
- RMSE
- WAPE
- Forecast bias
- Overforecasting and underforecasting

## Final-test protection

Protected final-test files loaded during Step 1: 0.

The reserved final-test target vault must remain unopened until
model selection, preprocessing and hyperparameters are locked.

## Next phase

Modelling Step 2: baseline forecasting models.
""".strip()

step1_handoff_path.write_text(
    handoff_text,
    encoding="utf-8",
)


# --------------------------------------------------------------
# 15. Display final completion results
# --------------------------------------------------------------

passed_check_count = int(
    step1_final_validation_df["Passed"].sum()
)

total_check_count = int(
    len(step1_final_validation_df)
)

passed_part_count = int(
    part_validation_register_df[
        "PartPassed"
    ].sum()
)

total_part_count = int(
    len(part_validation_register_df)
)

print("\n" + "=" * 72)
print("MODELLING STEP 1: COMPLETED AND VALIDATED")
print("=" * 72)

print(
    "\nStep 1 parts passed: "
    f"{passed_part_count} of {total_part_count}"
)

print(
    "Final validation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print(
    "Approved predictors: "
    f"{len(approved_predictors)}"
)

print(
    "Generated model features: "
    f"{generated_feature_count}"
)

print(
    "Validation folds: "
    f"{validation_window_count}"
)

print(
    "Product-window contexts: "
    f"{split_context_count}"
)

print(
    "Protected final-test files loaded: "
    f"{protected_final_test_files_loaded}"
)

print(
    "\nNext phase: "
    "MODELLING STEP 2 — BASELINE FORECASTS"
)

print("\nStep 1 completion summary:")
display(step1_completion_summary_df)

print("\nStep-by-step validation register:")
display(part_validation_register_df)

print("\nFinal validation summary:")
display(step1_final_validation_df)

print("\nSaved outputs:")
print(output_file_audit_path)
print(part_validation_register_path)
print(framework_function_register_path)
print(environment_register_path)
print(step1_final_validation_path)
print(step1_completion_summary_path)
print(step1_handoff_path)


MODELLING STEP 1: COMPLETED AND VALIDATED

Step 1 parts passed: 5 of 5
Final validation checks passed: 26 of 26
Approved predictors: 53
Generated model features: 114
Validation folds: 2
Product-window contexts: 254
Protected final-test files loaded: 0

Next phase: MODELLING STEP 2 — BASELINE FORECASTS

Step 1 completion summary:


,Phase,Status,TrainingRows,ValidationRows,ModelSelectionProducts,ValidationWindows,ProductWindowContexts,ApprovedPredictors,GeneratedModelFeatures,NumericalPredictors,CategoricalPredictors,BinaryPredictors,PreprocessingSystems,ProtectedFinalTestFilesLoaded,NextPhase
0,MODELLING_STEP_1,COMPLETED_AND_VALIDATED,48538,5080,127,2,254,53,114,43,7,3,2,0,MODELLING_STEP_2_BASELINE_FORECASTS



Step-by-step validation register:


,Part,ValidationFile,CheckCount,PassedCheckCount,FailedCheckCount,PartPassed
0,PART_1,01_model_entry_validation_summary.csv,24,24,0,True
1,PART_2,02_predictor_structure_validation_summary.csv,26,26,0,True
2,PART_3,03_semantic_role_validation_summary.csv,14,14,0,True
3,PART_4,04_preprocessing_transformer_validation_summar...,13,13,0,True
4,PART_5,05_training_prediction_evaluation_validation_s...,15,15,0,True



Final validation summary:


,Check,Expected,Actual,Passed
0,Step 1 validation parts,5,5,True
1,Failed Step 1 parts,0,0,True
2,Expected audit-output files,27,27,True
3,Missing audit-output files,0,0,True
4,Empty audit-output files,0,0,True
5,Approved predictors,53,53,True
6,Current-date predictors,22,22,True
7,Historical predictors,31,31,True
8,Numerical predictors,43,43,True
9,Categorical predictors,7,7,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/06_step1_output_file_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/06_step1_part_validation_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/06_framework_function_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/06_modelling_environment_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/06_modelling_step1_final_validation_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/06_modelling_step1_completion_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/01_pipeline_setup/MODELLING_STEP1_HANDOFF.md


## Now Baseline forecasts


In [16]:
# ==============================================================
# MODELLING STEP 2, PART 1
# Define and validate the baseline forecasting framework
# ==============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------
# 1. Confirm the required Step 1 objects
# --------------------------------------------------------------

required_objects = [
    "training_df",
    "validation_df",
    "approved_predictors",
    "feature_contract_df",
    "clip_demand_predictions",
    "build_prediction_output",
    "calculate_forecast_metrics",
    "MODELLING_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required Step 1 objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 1 Parts 1 to 6 first."
    )


# --------------------------------------------------------------
# 2. Create or confirm the baseline output folder
# --------------------------------------------------------------

BASELINE_OUTPUT_DIR = (
    Path(MODELLING_DIR)
    / "02_baselines"
)

BASELINE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Baseline output folder:")
print(BASELINE_OUTPUT_DIR)


# --------------------------------------------------------------
# 3. Confirm the locked forecast target
# --------------------------------------------------------------

TARGET_COLUMN = "TotalDemand"

if TARGET_COLUMN not in training_df.columns:
    raise KeyError(
        f"{TARGET_COLUMN} is missing from the training dataset."
    )

if TARGET_COLUMN not in validation_df.columns:
    raise KeyError(
        f"{TARGET_COLUMN} is missing from the validation dataset."
    )

if TARGET_COLUMN in approved_predictors:
    raise AssertionError(
        "The forecast target must not appear in the approved "
        "predictor list."
    )


# --------------------------------------------------------------
# 4. Define a robust boolean conversion helper
# --------------------------------------------------------------

def convert_value_to_boolean(value):
    """
    Convert common boolean representations into True or False.
    """

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if pd.isna(value):
        raise ValueError(
            "A required boolean contract value is missing."
        )

    cleaned_value = str(value).strip().lower()

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    if cleaned_value not in mapping:
        raise ValueError(
            "Unexpected boolean contract value: "
            f"{value!r}"
        )

    return mapping[cleaned_value]


# --------------------------------------------------------------
# 5. Define the baseline forecasting methods
# --------------------------------------------------------------

baseline_method_register_df = pd.DataFrame(
    [
        {
            "BaselineName": "ZERO_DEMAND",
            "BaselineFamily": "CONSTANT",
            "SourceType": "CONSTANT",
            "SourceFeature": None,
            "ConstantValue": 0.0,
            "Description":
                "Predict zero units for every product-date row.",
            "PrimaryPurpose":
                "Conservative reference for sparse demand.",
        },
        {
            "BaselineName": "LAST_OBSERVED_DEMAND",
            "BaselineFamily": "NAIVE",
            "SourceType": "FEATURE",
            "SourceFeature": "TotalDemandLag_1",
            "ConstantValue": np.nan,
            "Description":
                "Use demand from the previous operating day.",
            "PrimaryPurpose":
                "Standard one-step naive reference.",
        },
        {
            "BaselineName": "HISTORICAL_MEAN",
            "BaselineFamily": "HISTORICAL_AVERAGE",
            "SourceType": "FEATURE",
            "SourceFeature": "ExpandingPastMeanDemand",
            "ConstantValue": np.nan,
            "Description":
                "Use the mean demand from all previous operating days.",
            "PrimaryPurpose":
                "Long-term average-demand reference.",
        },
        {
            "BaselineName": "MOVING_AVERAGE_5",
            "BaselineFamily": "MOVING_AVERAGE",
            "SourceType": "FEATURE",
            "SourceFeature": "PastDemandRollingMean_5",
            "ConstantValue": np.nan,
            "Description":
                "Use average demand over the previous five operating days.",
            "PrimaryPurpose":
                "Short-term smoothed-demand reference.",
        },
        {
            "BaselineName": "MOVING_AVERAGE_10",
            "BaselineFamily": "MOVING_AVERAGE",
            "SourceType": "FEATURE",
            "SourceFeature": "PastDemandRollingMean_10",
            "ConstantValue": np.nan,
            "Description":
                "Use average demand over the previous ten operating days.",
            "PrimaryPurpose":
                "Medium-term smoothed-demand reference.",
        },
        {
            "BaselineName": "MOVING_AVERAGE_20",
            "BaselineFamily": "MOVING_AVERAGE",
            "SourceType": "FEATURE",
            "SourceFeature": "PastDemandRollingMean_20",
            "ConstantValue": np.nan,
            "Description":
                "Use average demand over the previous twenty operating days.",
            "PrimaryPurpose":
                "Longer-term smoothed-demand reference.",
        },
        {
            "BaselineName": "NAIVE_5_OPERATING_DAYS",
            "BaselineFamily": "OPERATING_DAY_NAIVE",
            "SourceType": "FEATURE",
            "SourceFeature": "TotalDemandLag_5",
            "ConstantValue": np.nan,
            "Description":
                "Use demand from five Eden operating days earlier.",
            "PrimaryPurpose":
                "Comparable operating-cycle reference.",
        },
    ]
)


# --------------------------------------------------------------
# 6. Extract required feature-based baseline sources
# --------------------------------------------------------------

feature_baseline_sources = (
    baseline_method_register_df.loc[
        baseline_method_register_df["SourceType"] == "FEATURE",
        "SourceFeature",
    ]
    .dropna()
    .tolist()
)

constant_baseline_count = int(
    (
        baseline_method_register_df["SourceType"]
        == "CONSTANT"
    ).sum()
)

feature_baseline_count = int(
    (
        baseline_method_register_df["SourceType"]
        == "FEATURE"
    ).sum()
)


# --------------------------------------------------------------
# 7. Confirm all baseline source features exist
# --------------------------------------------------------------

missing_baseline_sources_training = [
    column
    for column in feature_baseline_sources
    if column not in training_df.columns
]

missing_baseline_sources_validation = [
    column
    for column in feature_baseline_sources
    if column not in validation_df.columns
]

baseline_sources_not_approved = [
    column
    for column in feature_baseline_sources
    if column not in approved_predictors
]


# --------------------------------------------------------------
# 8. Audit the frozen feature contract for each baseline source
# --------------------------------------------------------------

required_contract_columns = [
    "Column",
    "UsesCurrentRowTarget",
    "UsesFutureTarget",
]

missing_contract_columns = [
    column
    for column in required_contract_columns
    if column not in feature_contract_df.columns
]

if missing_contract_columns:
    raise KeyError(
        "The feature contract is missing required columns:\n"
        f"{missing_contract_columns}"
    )

contract_lookup_df = (
    feature_contract_df
    .drop_duplicates(subset=["Column"])
    .set_index("Column")
)

baseline_source_audit_records = []

for _, baseline_row in baseline_method_register_df.iterrows():

    baseline_name = baseline_row["BaselineName"]
    source_type = baseline_row["SourceType"]
    source_feature = baseline_row["SourceFeature"]

    if source_type == "CONSTANT":

        baseline_source_audit_records.append(
            {
                "BaselineName": baseline_name,
                "SourceType": source_type,
                "SourceFeature": None,
                "PresentInTraining": True,
                "PresentInValidation": True,
                "ApprovedPredictor": True,
                "UsesCurrentRowTarget": False,
                "UsesFutureTarget": False,
                "TrainingMissingCount": 0,
                "ValidationMissingCount": 0,
                "Passed": True,
            }
        )

        continue

    if source_feature not in contract_lookup_df.index:
        raise KeyError(
            "Baseline source feature is missing from the frozen "
            f"feature contract: {source_feature}"
        )

    contract_row = contract_lookup_df.loc[
        source_feature
    ]

    uses_current_target = convert_value_to_boolean(
        contract_row["UsesCurrentRowTarget"]
    )

    uses_future_target = convert_value_to_boolean(
        contract_row["UsesFutureTarget"]
    )

    present_in_training = (
        source_feature in training_df.columns
    )

    present_in_validation = (
        source_feature in validation_df.columns
    )

    approved_predictor = (
        source_feature in approved_predictors
    )

    training_missing_count = int(
        training_df[source_feature]
        .isna()
        .sum()
    )

    validation_missing_count = int(
        validation_df[source_feature]
        .isna()
        .sum()
    )

    source_passed = bool(
        present_in_training
        and present_in_validation
        and approved_predictor
        and not uses_current_target
        and not uses_future_target
        and validation_missing_count == 0
    )

    baseline_source_audit_records.append(
        {
            "BaselineName": baseline_name,
            "SourceType": source_type,
            "SourceFeature": source_feature,
            "PresentInTraining": present_in_training,
            "PresentInValidation": present_in_validation,
            "ApprovedPredictor": approved_predictor,
            "UsesCurrentRowTarget": uses_current_target,
            "UsesFutureTarget": uses_future_target,
            "TrainingMissingCount": training_missing_count,
            "ValidationMissingCount": validation_missing_count,
            "Passed": source_passed,
        }
    )

baseline_source_audit_df = pd.DataFrame(
    baseline_source_audit_records
)


# --------------------------------------------------------------
# 9. Create the reusable simple-baseline prediction function
# --------------------------------------------------------------

def generate_simple_baseline_predictions(
    scoring_df,
    baseline_name,
):
    """
    Generate predictions for one registered simple baseline.

    Parameters
    ----------
    scoring_df:
        DataFrame containing the required historical predictors.

    baseline_name:
        Name from baseline_method_register_df.

    Returns
    -------
    numpy.ndarray
        Finite, non-negative demand predictions.
    """

    matching_rows = baseline_method_register_df.loc[
        baseline_method_register_df["BaselineName"]
        == baseline_name
    ]

    if matching_rows.empty:
        raise KeyError(
            f"Unknown baseline method: {baseline_name}"
        )

    if len(matching_rows) != 1:
        raise AssertionError(
            "Duplicate baseline definition found: "
            f"{baseline_name}"
        )

    baseline_definition = matching_rows.iloc[0]

    source_type = baseline_definition[
        "SourceType"
    ]

    if source_type == "CONSTANT":

        constant_value = float(
            baseline_definition["ConstantValue"]
        )

        raw_predictions = np.full(
            shape=len(scoring_df),
            fill_value=constant_value,
            dtype=float,
        )

    elif source_type == "FEATURE":

        source_feature = baseline_definition[
            "SourceFeature"
        ]

        if source_feature not in scoring_df.columns:
            raise KeyError(
                "Required baseline source feature is missing: "
                f"{source_feature}"
            )

        raw_predictions = pd.to_numeric(
            scoring_df[source_feature],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

    else:
        raise ValueError(
            "Unsupported baseline source type: "
            f"{source_type}"
        )

    if not np.isfinite(raw_predictions).all():
        raise ValueError(
            f"{baseline_name} generated missing or infinite values."
        )

    final_predictions = clip_demand_predictions(
        raw_predictions
    )

    return final_predictions


# --------------------------------------------------------------
# 10. Test every baseline on all validation rows
#
# This checks prediction generation only.
# Performance metrics are calculated in the next part.
# --------------------------------------------------------------

baseline_generation_audit_records = []

for baseline_name in baseline_method_register_df[
    "BaselineName"
].tolist():

    predictions = generate_simple_baseline_predictions(
        scoring_df=validation_df,
        baseline_name=baseline_name,
    )

    prediction_count = int(
        len(predictions)
    )

    missing_prediction_count = int(
        np.isnan(predictions).sum()
    )

    infinite_prediction_count = int(
        np.isinf(predictions).sum()
    )

    negative_prediction_count = int(
        (predictions < 0).sum()
    )

    generation_passed = bool(
        prediction_count == len(validation_df)
        and missing_prediction_count == 0
        and infinite_prediction_count == 0
        and negative_prediction_count == 0
    )

    baseline_generation_audit_records.append(
        {
            "BaselineName": baseline_name,
            "ValidationRows": len(validation_df),
            "PredictionCount": prediction_count,
            "MissingPredictionCount":
                missing_prediction_count,
            "InfinitePredictionCount":
                infinite_prediction_count,
            "NegativePredictionCount":
                negative_prediction_count,
            "MinimumPrediction":
                float(predictions.min()),
            "MaximumPrediction":
                float(predictions.max()),
            "Passed": generation_passed,
        }
    )

baseline_generation_audit_df = pd.DataFrame(
    baseline_generation_audit_records
)


# --------------------------------------------------------------
# 11. Reconstruct and validate the five-operating-day lag
#
# Training and validation are combined only inside each existing
# split context. This is an audit and does not alter the dataset.
# --------------------------------------------------------------

required_lag5_columns = [
    "SplitContextID",
    "WindowID",
    "CanonicalProductID",
    "Date",
    "OperatingDaySequence",
    "DayOfWeekNumber",
    "TotalDemand",
    "TotalDemandLag_5",
]

missing_lag5_training_columns = [
    column
    for column in required_lag5_columns
    if column not in training_df.columns
]

missing_lag5_validation_columns = [
    column
    for column in required_lag5_columns
    if column not in validation_df.columns
]

if missing_lag5_training_columns:
    raise KeyError(
        "Lag-five audit columns missing from training:\n"
        f"{missing_lag5_training_columns}"
    )

if missing_lag5_validation_columns:
    raise KeyError(
        "Lag-five audit columns missing from validation:\n"
        f"{missing_lag5_validation_columns}"
    )

lag5_training_df = training_df[
    required_lag5_columns
].copy()

lag5_training_df[
    "FrameworkRole"
] = "TRAIN"

lag5_validation_df = validation_df[
    required_lag5_columns
].copy()

lag5_validation_df[
    "FrameworkRole"
] = "VALIDATION"

lag5_combined_df = pd.concat(
    [
        lag5_training_df,
        lag5_validation_df,
    ],
    ignore_index=True,
)

lag5_combined_df["Date"] = pd.to_datetime(
    lag5_combined_df["Date"],
    errors="raise",
)

lag5_combined_df = (
    lag5_combined_df
    .sort_values(
        [
            "SplitContextID",
            "OperatingDaySequence",
        ]
    )
    .reset_index(drop=True)
)

lag5_group = lag5_combined_df.groupby(
    "SplitContextID",
    sort=False,
)

lag5_combined_df[
    "ReconstructedLag5Demand"
] = lag5_group[
    "TotalDemand"
].shift(5)

lag5_combined_df[
    "Lag5SourceDate"
] = lag5_group[
    "Date"
].shift(5)

lag5_combined_df[
    "Lag5SourceDayOfWeekNumber"
] = lag5_group[
    "DayOfWeekNumber"
].shift(5)

lag5_validation_audit_df = (
    lag5_combined_df.loc[
        lag5_combined_df["FrameworkRole"]
        == "VALIDATION"
    ]
    .copy()
)

lag5_validation_audit_df[
    "Lag5CalendarGapDays"
] = (
    lag5_validation_audit_df["Date"]
    - lag5_validation_audit_df["Lag5SourceDate"]
).dt.days

lag5_validation_audit_df[
    "SameWeekdayAsLag5Source"
] = (
    lag5_validation_audit_df["DayOfWeekNumber"]
    == lag5_validation_audit_df[
        "Lag5SourceDayOfWeekNumber"
    ]
)

lag5_validation_audit_df[
    "Lag5DemandMatchesFrozenFeature"
] = np.isclose(
    lag5_validation_audit_df[
        "TotalDemandLag_5"
    ],
    lag5_validation_audit_df[
        "ReconstructedLag5Demand"
    ],
    equal_nan=True,
)

lag5_reconstruction_mismatch_count = int(
    (
        ~lag5_validation_audit_df[
            "Lag5DemandMatchesFrozenFeature"
        ]
    ).sum()
)

lag5_missing_source_date_count = int(
    lag5_validation_audit_df[
        "Lag5SourceDate"
    ]
    .isna()
    .sum()
)

lag5_same_weekday_count = int(
    lag5_validation_audit_df[
        "SameWeekdayAsLag5Source"
    ].sum()
)

lag5_same_weekday_percentage = float(
    lag5_same_weekday_count
    / len(lag5_validation_audit_df)
    * 100
)


# --------------------------------------------------------------
# 12. Summarise lag-five calendar alignment by window
# --------------------------------------------------------------

lag5_calendar_alignment_summary_df = (
    lag5_validation_audit_df
    .groupby(
        "WindowID",
        as_index=False,
    )
    .agg(
        ValidationRows=(
            "SplitContextID",
            "size",
        ),
        SameWeekdayRows=(
            "SameWeekdayAsLag5Source",
            "sum",
        ),
        MinimumCalendarGapDays=(
            "Lag5CalendarGapDays",
            "min",
        ),
        MedianCalendarGapDays=(
            "Lag5CalendarGapDays",
            "median",
        ),
        MaximumCalendarGapDays=(
            "Lag5CalendarGapDays",
            "max",
        ),
        Lag5ReconstructionMismatches=(
            "Lag5DemandMatchesFrozenFeature",
            lambda values: int(
                (~values).sum()
            ),
        ),
    )
)

lag5_calendar_alignment_summary_df[
    "SameWeekdayPercentage"
] = (
    lag5_calendar_alignment_summary_df[
        "SameWeekdayRows"
    ]
    / lag5_calendar_alignment_summary_df[
        "ValidationRows"
    ]
    * 100
).round(4)


# --------------------------------------------------------------
# 13. Build Step 2 Part 1 validation checks
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one validation result.
    """

    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Registered baseline methods",
    len(baseline_method_register_df),
    7,
)

add_check(
    "Constant baseline methods",
    constant_baseline_count,
    1,
)

add_check(
    "Feature-based baseline methods",
    feature_baseline_count,
    6,
)

add_check(
    "Baseline sources missing from training",
    len(missing_baseline_sources_training),
    0,
)

add_check(
    "Baseline sources missing from validation",
    len(missing_baseline_sources_validation),
    0,
)

add_check(
    "Baseline sources outside approved predictors",
    len(baseline_sources_not_approved),
    0,
)

add_check(
    "Baseline source-contract failures",
    int(
        (
            ~baseline_source_audit_df["Passed"]
        ).sum()
    ),
    0,
)

add_check(
    "Validation missing cells in baseline sources",
    int(
        validation_df[
            feature_baseline_sources
        ]
        .isna()
        .sum()
        .sum()
    ),
    0,
)

add_check(
    "Failed baseline prediction generators",
    int(
        (
            ~baseline_generation_audit_df["Passed"]
        ).sum()
    ),
    0,
)

add_check(
    "Total generated baseline prediction rows",
    int(
        baseline_generation_audit_df[
            "PredictionCount"
        ].sum()
    ),
    (
        len(validation_df)
        * len(baseline_method_register_df)
    ),
)

add_check(
    "Negative generated predictions",
    int(
        baseline_generation_audit_df[
            "NegativePredictionCount"
        ].sum()
    ),
    0,
)

add_check(
    "Lag-five validation rows audited",
    len(lag5_validation_audit_df),
    len(validation_df),
)

add_check(
    "Lag-five missing source dates",
    lag5_missing_source_date_count,
    0,
)

add_check(
    "Lag-five reconstruction mismatches",
    lag5_reconstruction_mismatch_count,
    0,
)

part1_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 14. Stop if any required validation fails
# --------------------------------------------------------------

failed_checks_df = part1_validation_df.loc[
    ~part1_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print(
        "\nFAILED MODELLING STEP 2 PART 1 CHECKS"
    )

    display(
        failed_checks_df
    )

    raise AssertionError(
        "Modelling Step 2 Part 1 failed. "
        "Do not calculate baseline performance."
    )


# --------------------------------------------------------------
# 15. Save Step 2 Part 1 outputs
# --------------------------------------------------------------

baseline_method_register_path = (
    BASELINE_OUTPUT_DIR
    / "01_baseline_method_register.csv"
)

baseline_source_audit_path = (
    BASELINE_OUTPUT_DIR
    / "01_baseline_source_feature_audit.csv"
)

baseline_generation_audit_path = (
    BASELINE_OUTPUT_DIR
    / "01_baseline_prediction_generation_audit.csv"
)

lag5_alignment_summary_path = (
    BASELINE_OUTPUT_DIR
    / "01_lag5_calendar_alignment_summary.csv"
)

lag5_validation_audit_path = (
    BASELINE_OUTPUT_DIR
    / "01_lag5_validation_reconstruction_audit.csv"
)

part1_validation_path = (
    BASELINE_OUTPUT_DIR
    / "01_baseline_framework_validation_summary.csv"
)

baseline_method_register_df.to_csv(
    baseline_method_register_path,
    index=False,
)

baseline_source_audit_df.to_csv(
    baseline_source_audit_path,
    index=False,
)

baseline_generation_audit_df.to_csv(
    baseline_generation_audit_path,
    index=False,
)

lag5_calendar_alignment_summary_df.to_csv(
    lag5_alignment_summary_path,
    index=False,
)

lag5_validation_audit_df.to_csv(
    lag5_validation_audit_path,
    index=False,
)

part1_validation_df.to_csv(
    part1_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 16. Prepare final display values
# --------------------------------------------------------------

failed_baseline_generators = int(
    (
        ~baseline_generation_audit_df[
            "Passed"
        ]
    ).sum()
)

failed_source_audits = int(
    (
        ~baseline_source_audit_df[
            "Passed"
        ]
    ).sum()
)

passed_check_count = int(
    part1_validation_df[
        "Passed"
    ].sum()
)

total_check_count = int(
    len(part1_validation_df)
)


# --------------------------------------------------------------
# 17. Display final results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 2 PART 1: PASSED")
print("=" * 72)

print(
    "\nBaseline methods registered: "
    f"{len(baseline_method_register_df)}"
)

print(
    "Feature-based baselines: "
    f"{feature_baseline_count}"
)

print(
    "Constant baselines: "
    f"{constant_baseline_count}"
)

print(
    "Validation rows audited: "
    f"{len(validation_df):,}"
)

print(
    "Failed baseline source audits: "
    f"{failed_source_audits}"
)

print(
    "Failed baseline prediction generators: "
    f"{failed_baseline_generators}"
)

print(
    "Lag-five reconstruction mismatches: "
    f"{lag5_reconstruction_mismatch_count}"
)

print(
    "Lag-five same-weekday alignment: "
    f"{lag5_same_weekday_percentage:.2f}%"
)

print(
    "\nValidation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print("\nBaseline method register:")
display(
    baseline_method_register_df
)

print("\nBaseline source-feature audit:")
display(
    baseline_source_audit_df
)

print("\nBaseline prediction-generation audit:")
display(
    baseline_generation_audit_df
)

print("\nLag-five calendar alignment:")
display(
    lag5_calendar_alignment_summary_df
)

print("\nPart 1 validation summary:")
display(
    part1_validation_df
)

print("\nSaved outputs:")
print(
    baseline_method_register_path
)
print(
    baseline_source_audit_path
)
print(
    baseline_generation_audit_path
)
print(
    lag5_alignment_summary_path
)
print(
    lag5_validation_audit_path
)
print(
    part1_validation_path
)

Baseline output folder:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines

MODELLING STEP 2 PART 1: PASSED

Baseline methods registered: 7
Feature-based baselines: 6
Constant baselines: 1
Validation rows audited: 5,080
Failed baseline source audits: 0
Failed baseline prediction generators: 0
Lag-five reconstruction mismatches: 0
Lag-five same-weekday alignment: 60.00%

Validation checks passed: 14 of 14

Baseline method register:


,BaselineName,BaselineFamily,SourceType,SourceFeature,ConstantValue,Description,PrimaryPurpose
0,ZERO_DEMAND,CONSTANT,CONSTANT,None,0.0,Predict zero units for every product-date row.,Conservative reference for sparse demand.
1,LAST_OBSERVED_DEMAND,NAIVE,FEATURE,TotalDemandLag_1,NaN,Use demand from the previous operating day.,Standard one-step naive reference.
2,HISTORICAL_MEAN,HISTORICAL_AVERAGE,FEATURE,ExpandingPastMeanDemand,NaN,Use the mean demand from all previous operatin...,Long-term average-demand reference.
3,MOVING_AVERAGE_5,MOVING_AVERAGE,FEATURE,PastDemandRollingMean_5,NaN,Use average demand over the previous five oper...,Short-term smoothed-demand reference.
4,MOVING_AVERAGE_10,MOVING_AVERAGE,FEATURE,PastDemandRollingMean_10,NaN,Use average demand over the previous ten opera...,Medium-term smoothed-demand reference.
5,MOVING_AVERAGE_20,MOVING_AVERAGE,FEATURE,PastDemandRollingMean_20,NaN,Use average demand over the previous twenty op...,Longer-term smoothed-demand reference.
6,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,FEATURE,TotalDemandLag_5,NaN,Use demand from five Eden operating days earlier.,Comparable operating-cycle reference.



Baseline source-feature audit:


,BaselineName,SourceType,SourceFeature,PresentInTraining,PresentInValidation,ApprovedPredictor,UsesCurrentRowTarget,UsesFutureTarget,TrainingMissingCount,ValidationMissingCount,Passed
0,ZERO_DEMAND,CONSTANT,None,True,True,True,False,False,0,0,True
1,LAST_OBSERVED_DEMAND,FEATURE,TotalDemandLag_1,True,True,True,False,False,254,0,True
2,HISTORICAL_MEAN,FEATURE,ExpandingPastMeanDemand,True,True,True,False,False,254,0,True
3,MOVING_AVERAGE_5,FEATURE,PastDemandRollingMean_5,True,True,True,False,False,254,0,True
4,MOVING_AVERAGE_10,FEATURE,PastDemandRollingMean_10,True,True,True,False,False,254,0,True
5,MOVING_AVERAGE_20,FEATURE,PastDemandRollingMean_20,True,True,True,False,False,254,0,True
6,NAIVE_5_OPERATING_DAYS,FEATURE,TotalDemandLag_5,True,True,True,False,False,1270,0,True



Baseline prediction-generation audit:


...


Lag-five calendar alignment:


,WindowID,ValidationRows,SameWeekdayRows,MinimumCalendarGapDays,MedianCalendarGapDays,MaximumCalendarGapDays,Lag5ReconstructionMismatches,SameWeekdayPercentage
0,STANDARD_BACKTEST_FOLD_1,2540,1143,5,7.0,19,0,45.0
1,STANDARD_BACKTEST_FOLD_2,2540,1905,7,7.0,10,0,75.0



Part 1 validation summary:


,Check,Expected,Actual,Passed
0,Registered baseline methods,7,7,True
1,Constant baseline methods,1,1,True
2,Feature-based baseline methods,6,6,True
3,Baseline sources missing from training,0,0,True
4,Baseline sources missing from validation,0,0,True
5,Baseline sources outside approved predictors,0,0,True
6,Baseline source-contract failures,0,0,True
7,Validation missing cells in baseline sources,0,0,True
8,Failed baseline prediction generators,0,0,True
9,Total generated baseline prediction rows,35560,35560,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/01_baseline_method_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/01_baseline_source_feature_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/01_baseline_prediction_generation_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/01_lag5_calendar_alignment_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/01_lag5_validation_reconstruction_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/01_baseline_framework_validation_summary.csv


In [17]:
# ==============================================================
# MODELLING STEP 2, PART 2
# Generate baseline predictions and overall validation metrics
# ==============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------
# 1. Confirm the required Step 2 Part 1 objects
# --------------------------------------------------------------

required_objects = [
    "validation_df",
    "baseline_method_register_df",
    "generate_simple_baseline_predictions",
    "build_prediction_output",
    "calculate_forecast_metrics",
    "BASELINE_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 2 Part 1 before this cell."
    )


# --------------------------------------------------------------
# 2. Confirm the validation structure
# --------------------------------------------------------------

TARGET_COLUMN = "TotalDemand"

required_validation_columns = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "WindowID",
    "SplitContextID",
    TARGET_COLUMN,
]

missing_validation_columns = [
    column
    for column in required_validation_columns
    if column not in validation_df.columns
]

if missing_validation_columns:
    raise KeyError(
        "The validation dataset is missing required columns:\n"
        f"{missing_validation_columns}"
    )

validation_df["Date"] = pd.to_datetime(
    validation_df["Date"],
    errors="raise",
)

baseline_names = (
    baseline_method_register_df["BaselineName"]
    .dropna()
    .astype(str)
    .tolist()
)

validation_windows = sorted(
    validation_df["WindowID"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if len(baseline_names) != 7:
    raise AssertionError(
        "Expected seven registered baselines, but found "
        f"{len(baseline_names)}."
    )

if len(validation_windows) != 2:
    raise AssertionError(
        "Expected two validation windows, but found "
        f"{len(validation_windows)}."
    )


# --------------------------------------------------------------
# 3. Generate row-level predictions for every baseline
# --------------------------------------------------------------

baseline_definition_lookup = (
    baseline_method_register_df
    .drop_duplicates(subset=["BaselineName"])
    .set_index("BaselineName")
)

prediction_frames = []

for baseline_name in baseline_names:

    baseline_definition = baseline_definition_lookup.loc[
        baseline_name
    ]

    baseline_family = str(
        baseline_definition["BaselineFamily"]
    )

    predictions = generate_simple_baseline_predictions(
        scoring_df=validation_df,
        baseline_name=baseline_name,
    )

    prediction_output = build_prediction_output(
        validation_metadata=validation_df,
        raw_predictions=predictions,
        model_name=baseline_name,
        model_family=baseline_family,
    )

    prediction_output["SourceType"] = (
        baseline_definition["SourceType"]
    )

    prediction_output["SourceFeature"] = (
        baseline_definition["SourceFeature"]
    )

    prediction_frames.append(
        prediction_output
    )

baseline_validation_predictions_df = pd.concat(
    prediction_frames,
    ignore_index=True,
)


# --------------------------------------------------------------
# 4. Standardise the prediction-table order
# --------------------------------------------------------------

prediction_column_order = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "WindowID",
    "SplitContextID",
    "ActualDemand",
    "RawPrediction",
    "PredictedDemand",
    "PredictionClippedAtZero",
    "ModelName",
    "ModelFamily",
    "SourceType",
    "SourceFeature",
    "ForecastError",
    "AbsoluteError",
    "SquaredError",
]

baseline_validation_predictions_df = (
    baseline_validation_predictions_df.loc[
        :,
        prediction_column_order,
    ]
    .sort_values(
        [
            "ModelName",
            "WindowID",
            "CanonicalProductID",
            "Date",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 5. Calculate metrics separately for each validation fold
# --------------------------------------------------------------

fold_metric_records = []

for (
    baseline_name,
    baseline_family,
    window_id,
), group_df in baseline_validation_predictions_df.groupby(
    [
        "ModelName",
        "ModelFamily",
        "WindowID",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    metric_record = {
        "BaselineName": baseline_name,
        "BaselineFamily": baseline_family,
        "WindowID": window_id,
    }

    metric_record.update(
        metric_values
    )

    fold_metric_records.append(
        metric_record
    )

baseline_fold_metrics_df = pd.DataFrame(
    fold_metric_records
)

baseline_fold_metrics_df = (
    baseline_fold_metrics_df
    .sort_values(
        [
            "WindowID",
            "MAE",
            "RMSE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

baseline_fold_metrics_df["MAE_RankWithinFold"] = (
    baseline_fold_metrics_df
    .groupby("WindowID")["MAE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)


# --------------------------------------------------------------
# 6. Calculate combined metrics across both validation folds
# --------------------------------------------------------------

combined_metric_records = []

for (
    baseline_name,
    baseline_family,
), group_df in baseline_validation_predictions_df.groupby(
    [
        "ModelName",
        "ModelFamily",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    metric_record = {
        "BaselineName": baseline_name,
        "BaselineFamily": baseline_family,
    }

    metric_record.update(
        metric_values
    )

    combined_metric_records.append(
        metric_record
    )

baseline_combined_metrics_df = pd.DataFrame(
    combined_metric_records
)


# --------------------------------------------------------------
# 7. Summarise fold-to-fold stability
# --------------------------------------------------------------

baseline_fold_stability_df = (
    baseline_fold_metrics_df
    .groupby(
        [
            "BaselineName",
            "BaselineFamily",
        ],
        as_index=False,
    )
    .agg(
        FoldCount=(
            "WindowID",
            "nunique",
        ),
        MeanFoldMAE=(
            "MAE",
            "mean",
        ),
        MinimumFoldMAE=(
            "MAE",
            "min",
        ),
        MaximumFoldMAE=(
            "MAE",
            "max",
        ),
        FoldMAEStandardDeviation=(
            "MAE",
            "std",
        ),
        MeanFoldRMSE=(
            "RMSE",
            "mean",
        ),
        MeanFoldWAPE_Percent=(
            "WAPE_Percent",
            "mean",
        ),
    )
)

baseline_fold_stability_df[
    "FoldMAEStandardDeviation"
] = baseline_fold_stability_df[
    "FoldMAEStandardDeviation"
].fillna(0.0)


# --------------------------------------------------------------
# 8. Create the overall baseline ranking
# --------------------------------------------------------------

baseline_ranking_df = baseline_combined_metrics_df.merge(
    baseline_fold_stability_df,
    on=[
        "BaselineName",
        "BaselineFamily",
    ],
    how="left",
    validate="one_to_one",
)

baseline_ranking_df = (
    baseline_ranking_df
    .sort_values(
        [
            "MAE",
            "RMSE",
            "WAPE_Percent",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

baseline_ranking_df.insert(
    0,
    "MAE_Rank",
    np.arange(
        1,
        len(baseline_ranking_df) + 1,
    ),
)

baseline_ranking_df["RMSE_Rank"] = (
    baseline_ranking_df["RMSE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_ranking_df["WAPE_Rank"] = (
    baseline_ranking_df["WAPE_Percent"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_ranking_df["IsBestOverallMAE"] = (
    baseline_ranking_df["MAE_Rank"] == 1
)


# --------------------------------------------------------------
# 9. Create prediction-coverage audits
# --------------------------------------------------------------

prediction_coverage_df = (
    baseline_validation_predictions_df
    .groupby(
        [
            "ModelName",
            "WindowID",
        ],
        as_index=False,
    )
    .agg(
        PredictionRows=(
            "PredictedDemand",
            "size",
        ),
        Products=(
            "CanonicalProductID",
            "nunique",
        ),
        SplitContexts=(
            "SplitContextID",
            "nunique",
        ),
        MinimumDate=(
            "Date",
            "min",
        ),
        MaximumDate=(
            "Date",
            "max",
        ),
        ActualDemandTotal=(
            "ActualDemand",
            "sum",
        ),
        PredictedDemandTotal=(
            "PredictedDemand",
            "sum",
        ),
        MissingPredictions=(
            "PredictedDemand",
            lambda values: int(
                values.isna().sum()
            ),
        ),
        NegativePredictions=(
            "PredictedDemand",
            lambda values: int(
                (values < 0).sum()
            ),
        ),
        ClippedPredictions=(
            "PredictionClippedAtZero",
            "sum",
        ),
    )
)

prediction_coverage_df["MinimumDate"] = pd.to_datetime(
    prediction_coverage_df["MinimumDate"]
).dt.date

prediction_coverage_df["MaximumDate"] = pd.to_datetime(
    prediction_coverage_df["MaximumDate"]
).dt.date


# --------------------------------------------------------------
# 10. Calculate structural validation values
# --------------------------------------------------------------

expected_total_prediction_rows = int(
    len(validation_df)
    * len(baseline_names)
)

expected_model_window_contexts = int(
    len(baseline_names)
    * len(validation_windows)
)

prediction_duplicate_count = int(
    baseline_validation_predictions_df.duplicated(
        subset=[
            "ModelName",
            "SplitContextID",
            "Date",
        ]
    ).sum()
)

missing_prediction_count = int(
    baseline_validation_predictions_df[
        "PredictedDemand"
    ].isna().sum()
)

infinite_prediction_count = int(
    np.isinf(
        baseline_validation_predictions_df[
            "PredictedDemand"
        ].to_numpy(dtype=float)
    ).sum()
)

negative_prediction_count = int(
    (
        baseline_validation_predictions_df[
            "PredictedDemand"
        ] < 0
    ).sum()
)

clipped_prediction_count = int(
    baseline_validation_predictions_df[
        "PredictionClippedAtZero"
    ].sum()
)

expected_actual_total_per_baseline = float(
    validation_df[TARGET_COLUMN].sum()
)

actual_total_mismatch_count = int(
    (
        baseline_combined_metrics_df[
            "ActualDemandTotal"
        ]
        != expected_actual_total_per_baseline
    ).sum()
)

fold_wape_missing_count = int(
    baseline_fold_metrics_df[
        "WAPE_Percent"
    ].isna().sum()
)

combined_wape_missing_count = int(
    baseline_combined_metrics_df[
        "WAPE_Percent"
    ].isna().sum()
)

rows_per_validation_window = int(
    len(validation_df)
    // len(validation_windows)
)

coverage_row_mismatch_count = int(
    (
        prediction_coverage_df[
            "PredictionRows"
        ]
        != rows_per_validation_window
    ).sum()
)


# --------------------------------------------------------------
# 11. Build Step 2 Part 2 validation checks
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one validation result.
    """

    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Registered baseline methods",
    len(baseline_names),
    7,
)

add_check(
    "Validation windows",
    len(validation_windows),
    2,
)

add_check(
    "Total baseline prediction rows",
    len(baseline_validation_predictions_df),
    expected_total_prediction_rows,
)

add_check(
    "Duplicate model-context-date predictions",
    prediction_duplicate_count,
    0,
)

add_check(
    "Missing predictions",
    missing_prediction_count,
    0,
)

add_check(
    "Infinite predictions",
    infinite_prediction_count,
    0,
)

add_check(
    "Negative predictions",
    negative_prediction_count,
    0,
)

add_check(
    "Clipped baseline predictions",
    clipped_prediction_count,
    0,
)

add_check(
    "Model-window coverage rows",
    len(prediction_coverage_df),
    expected_model_window_contexts,
)

add_check(
    "Model-window prediction-row mismatches",
    coverage_row_mismatch_count,
    0,
)

add_check(
    "Fold metric rows",
    len(baseline_fold_metrics_df),
    expected_model_window_contexts,
)

add_check(
    "Combined metric rows",
    len(baseline_combined_metrics_df),
    len(baseline_names),
)

add_check(
    "Ranking rows",
    len(baseline_ranking_df),
    len(baseline_names),
)

add_check(
    "Actual-demand total mismatches",
    actual_total_mismatch_count,
    0,
)

add_check(
    "Missing fold WAPE values",
    fold_wape_missing_count,
    0,
)

add_check(
    "Missing combined WAPE values",
    combined_wape_missing_count,
    0,
)

add_check(
    "Best-MAE baseline count",
    int(
        baseline_ranking_df[
            "IsBestOverallMAE"
        ].sum()
    ),
    1,
)

part2_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 12. Stop if any required validation fails
# --------------------------------------------------------------

failed_checks_df = part2_validation_df.loc[
    ~part2_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print(
        "\nFAILED MODELLING STEP 2 PART 2 CHECKS"
    )

    display(
        failed_checks_df
    )

    raise AssertionError(
        "Modelling Step 2 Part 2 failed. "
        "Do not proceed to product-level baseline analysis."
    )


# --------------------------------------------------------------
# 13. Save Step 2 Part 2 outputs
# --------------------------------------------------------------

baseline_predictions_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "02_baseline_validation_predictions.csv"
)

baseline_fold_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "02_baseline_fold_metrics.csv"
)

baseline_combined_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "02_baseline_combined_metrics.csv"
)

baseline_fold_stability_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "02_baseline_fold_stability_summary.csv"
)

baseline_ranking_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "02_baseline_overall_ranking.csv"
)

prediction_coverage_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "02_baseline_prediction_coverage_audit.csv"
)

part2_validation_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "02_baseline_performance_validation_summary.csv"
)

baseline_validation_predictions_df.to_csv(
    baseline_predictions_path,
    index=False,
)

baseline_fold_metrics_df.to_csv(
    baseline_fold_metrics_path,
    index=False,
)

baseline_combined_metrics_df.to_csv(
    baseline_combined_metrics_path,
    index=False,
)

baseline_fold_stability_df.to_csv(
    baseline_fold_stability_path,
    index=False,
)

baseline_ranking_df.to_csv(
    baseline_ranking_path,
    index=False,
)

prediction_coverage_df.to_csv(
    prediction_coverage_path,
    index=False,
)

part2_validation_df.to_csv(
    part2_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 14. Prepare final display values
# --------------------------------------------------------------

best_baseline_name = str(
    baseline_ranking_df.iloc[0][
        "BaselineName"
    ]
)

best_baseline_mae = float(
    baseline_ranking_df.iloc[0][
        "MAE"
    ]
)

best_baseline_rmse = float(
    baseline_ranking_df.iloc[0][
        "RMSE"
    ]
)

best_baseline_wape = float(
    baseline_ranking_df.iloc[0][
        "WAPE_Percent"
    ]
)

passed_check_count = int(
    part2_validation_df["Passed"].sum()
)

total_check_count = int(
    len(part2_validation_df)
)


# --------------------------------------------------------------
# 15. Display final results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 2 PART 2: PASSED")
print("=" * 72)

print(
    "\nBaseline prediction rows: "
    f"{len(baseline_validation_predictions_df):,}"
)

print(
    "Fold-level metric rows: "
    f"{len(baseline_fold_metrics_df)}"
)

print(
    "Combined baseline metric rows: "
    f"{len(baseline_combined_metrics_df)}"
)

print(
    "Best overall baseline by MAE: "
    f"{best_baseline_name}"
)

print(
    "Best baseline MAE: "
    f"{best_baseline_mae:.4f} units"
)

print(
    "Best baseline RMSE: "
    f"{best_baseline_rmse:.4f} units"
)

print(
    "Best baseline WAPE: "
    f"{best_baseline_wape:.2f}%"
)

print(
    "\nValidation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print("\nOverall baseline ranking:")
display(
    baseline_ranking_df
)

print("\nFold-level baseline metrics:")
display(
    baseline_fold_metrics_df
)

print("\nPrediction-coverage audit:")
display(
    prediction_coverage_df
)

print("\nPart 2 validation summary:")
display(
    part2_validation_df
)

print("\nSaved outputs:")
print(baseline_predictions_path)
print(baseline_fold_metrics_path)
print(baseline_combined_metrics_path)
print(baseline_fold_stability_path)
print(baseline_ranking_path)
print(prediction_coverage_path)
print(part2_validation_path)


MODELLING STEP 2 PART 2: PASSED

Baseline prediction rows: 35,560
Fold-level metric rows: 14
Combined baseline metric rows: 7
Best overall baseline by MAE: MOVING_AVERAGE_5
Best baseline MAE: 0.6434 units
Best baseline RMSE: 2.4391 units
Best baseline WAPE: 40.27%

Validation checks passed: 17 of 17

Overall baseline ranking:


,MAE_Rank,BaselineName,BaselineFamily,ObservationCount,ActualDemandTotal,PredictedDemandTotal,MAE,RMSE,WAPE_Percent,MeanBias,...,FoldCount,MeanFoldMAE,MinimumFoldMAE,MaximumFoldMAE,FoldMAEStandardDeviation,MeanFoldRMSE,MeanFoldWAPE_Percent,RMSE_Rank,WAPE_Rank,IsBestOverallMAE
0,1,MOVING_AVERAGE_5,MOVING_AVERAGE,5080,8117.0,7585.200000,0.643425,2.439141,40.268572,-0.104685,...,2,0.643425,0.620866,0.665984,0.031903,2.438685,41.420465,1,1,True
1,2,MOVING_AVERAGE_10,MOVING_AVERAGE,5080,8117.0,7338.800000,0.647087,2.449349,40.497721,-0.153189,...,2,0.647087,0.632874,0.661299,0.020100,2.449015,41.757578,2,2,False
2,3,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,5080,8117.0,7279.000000,0.680315,2.570885,42.577307,-0.164961,...,2,0.680315,0.670866,0.689764,0.013363,2.569739,44.193721,4,3,False
3,4,MOVING_AVERAGE_20,MOVING_AVERAGE,5080,8117.0,6968.050000,0.686526,2.555544,42.965997,-0.226171,...,2,0.686526,0.649508,0.723543,0.052351,2.554124,44.040123,3,4,False
4,5,LAST_OBSERVED_DEMAND,NAIVE,5080,8117.0,7983.000000,0.793701,2.953311,49.673525,-0.026378,...,2,0.793701,0.694094,0.893307,0.140865,2.943608,50.235558,5,5,False
5,6,ZERO_DEMAND,CONSTANT,5080,8117.0,0.000000,1.597835,5.736178,100.000000,-1.597835,...,2,1.597835,1.302756,1.892913,0.417304,5.694152,100.000000,7,6,False
6,7,HISTORICAL_MEAN,HISTORICAL_AVERAGE,5080,8117.0,11788.510115,2.189040,4.974717,137.000420,0.722738,...,2,2.189040,2.141977,2.236103,0.066557,4.972156,141.274581,6,7,False



Fold-level baseline metrics:


,BaselineName,BaselineFamily,WindowID,ObservationCount,ActualDemandTotal,PredictedDemandTotal,MAE,RMSE,WAPE_Percent,MeanBias,TotalBias,OverforecastUnits,UnderforecastUnits,ZeroActualObservationCount,MAE_RankWithinFold
0,MOVING_AVERAGE_5,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_1,2540,3309.0,2791.600000,0.620866,2.485850,47.657903,-0.203701,-517.400000,529.80000,1047.200000,2179,1
1,MOVING_AVERAGE_10,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_1,2540,3309.0,2720.100000,0.632874,2.489494,48.579631,-0.231850,-588.900000,509.30000,1098.200000,2179,2
2,MOVING_AVERAGE_20,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_1,2540,3309.0,2824.650000,0.649508,2.468934,49.856452,-0.190689,-484.350000,582.70000,1067.050000,2179,3
3,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2540,3309.0,2665.000000,0.689764,2.646495,52.946510,-0.253543,-644.000000,554.00000,1198.000000,2179,4
4,LAST_OBSERVED_DEMAND,NAIVE,STANDARD_BACKTEST_FOLD_1,2540,3309.0,3128.000000,0.694094,2.704400,53.278936,-0.071260,-181.000000,791.00000,972.000000,2179,5
5,ZERO_DEMAND,CONSTANT,STANDARD_BACKTEST_FOLD_1,2540,3309.0,0.000000,1.302756,5.001063,100.000000,-1.302756,-3309.000000,0.00000,3309.000000,2179,6
6,HISTORICAL_MEAN,HISTORICAL_AVERAGE,STANDARD_BACKTEST_FOLD_1,2540,3309.0,5982.643082,2.141977,4.812560,164.418920,1.052615,2673.643082,4057.13258,1383.489498,2179,7
7,MOVING_AVERAGE_10,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_2,2540,4808.0,4618.700000,0.661299,2.408536,34.935524,-0.074528,-189.300000,745.20000,934.500000,2143,1
8,MOVING_AVERAGE_5,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_2,2540,4808.0,4793.600000,0.665984,2.391520,35.183028,-0.005669,-14.400000,838.60000,853.000000,2143,2
9,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_2,2540,4808.0,4614.000000,0.670866,2.492982,35.440932,-0.076378,-194.000000,755.00000,949.000000,2143,3



Prediction-coverage audit:


,ModelName,WindowID,PredictionRows,Products,SplitContexts,MinimumDate,MaximumDate,ActualDemandTotal,PredictedDemandTotal,MissingPredictions,NegativePredictions,ClippedPredictions
0,HISTORICAL_MEAN,STANDARD_BACKTEST_FOLD_1,2540,127,127,2026-01-05,2026-01-29,3309,5982.643082,0,0,0
1,HISTORICAL_MEAN,STANDARD_BACKTEST_FOLD_2,2540,127,127,2026-01-30,2026-02-27,4808,5805.867033,0,0,0
2,LAST_OBSERVED_DEMAND,STANDARD_BACKTEST_FOLD_1,2540,127,127,2026-01-05,2026-01-29,3309,3128.000000,0,0,0
3,LAST_OBSERVED_DEMAND,STANDARD_BACKTEST_FOLD_2,2540,127,127,2026-01-30,2026-02-27,4808,4855.000000,0,0,0
4,MOVING_AVERAGE_10,STANDARD_BACKTEST_FOLD_1,2540,127,127,2026-01-05,2026-01-29,3309,2720.100000,0,0,0
5,MOVING_AVERAGE_10,STANDARD_BACKTEST_FOLD_2,2540,127,127,2026-01-30,2026-02-27,4808,4618.700000,0,0,0
6,MOVING_AVERAGE_20,STANDARD_BACKTEST_FOLD_1,2540,127,127,2026-01-05,2026-01-29,3309,2824.650000,0,0,0
7,MOVING_AVERAGE_20,STANDARD_BACKTEST_FOLD_2,2540,127,127,2026-01-30,2026-02-27,4808,4143.400000,0,0,0
8,MOVING_AVERAGE_5,STANDARD_BACKTEST_FOLD_1,2540,127,127,2026-01-05,2026-01-29,3309,2791.600000,0,0,0
9,MOVING_AVERAGE_5,STANDARD_BACKTEST_FOLD_2,2540,127,127,2026-01-30,2026-02-27,4808,4793.600000,0,0,0



Part 2 validation summary:


,Check,Expected,Actual,Passed
0,Registered baseline methods,7,7,True
1,Validation windows,2,2,True
2,Total baseline prediction rows,35560,35560,True
3,Duplicate model-context-date predictions,0,0,True
4,Missing predictions,0,0,True
5,Infinite predictions,0,0,True
6,Negative predictions,0,0,True
7,Clipped baseline predictions,0,0,True
8,Model-window coverage rows,14,14,True
9,Model-window prediction-row mismatches,0,0,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/02_baseline_validation_predictions.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/02_baseline_fold_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/02_baseline_combined_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/02_baseline_fold_stability_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/02_baseline_overall_ranking.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/02_baseline_prediction_coverage_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/02_baseline_performance_validation_summary.csv


In [18]:
# ==============================================================
# MODELLING STEP 2, PART 3
# Product-window, demand-pattern and product-level baseline analysis
# ==============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------
# 1. Confirm required objects
# --------------------------------------------------------------

required_objects = [
    "validation_df",
    "baseline_validation_predictions_df",
    "baseline_ranking_df",
    "calculate_forecast_metrics",
    "BASELINE_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 2 Parts 1 and 2 first."
    )


# --------------------------------------------------------------
# 2. Validate required columns
# --------------------------------------------------------------

required_validation_columns = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "WindowID",
    "SplitContextID",
    "DemandPatternClass",
    "TotalDemand",
]

required_prediction_columns = [
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",
    "WindowID",
    "SplitContextID",
    "ActualDemand",
    "PredictedDemand",
    "ModelName",
    "ModelFamily",
]

missing_validation_columns = [
    column
    for column in required_validation_columns
    if column not in validation_df.columns
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in baseline_validation_predictions_df.columns
]

if missing_validation_columns:
    raise KeyError(
        "The validation dataset is missing required columns:\n"
        f"{missing_validation_columns}"
    )

if missing_prediction_columns:
    raise KeyError(
        "The baseline prediction table is missing required columns:\n"
        f"{missing_prediction_columns}"
    )

validation_df["Date"] = pd.to_datetime(
    validation_df["Date"],
    errors="raise",
)

baseline_validation_predictions_df["Date"] = pd.to_datetime(
    baseline_validation_predictions_df["Date"],
    errors="raise",
)

baseline_names = sorted(
    baseline_validation_predictions_df["ModelName"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

validation_windows = sorted(
    validation_df["WindowID"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

validation_context_ids = sorted(
    validation_df["SplitContextID"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

validation_product_ids = sorted(
    validation_df["CanonicalProductID"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


# --------------------------------------------------------------
# 3. Create one metadata row per product-window context
# --------------------------------------------------------------

context_metadata_columns = [
    "SplitContextID",
    "WindowID",
    "CanonicalProductID",
    "CanonicalProductName",
    "DemandPatternClass",
]

context_metadata_df = (
    validation_df.loc[
        :,
        context_metadata_columns,
    ]
    .drop_duplicates()
    .sort_values(
        [
            "WindowID",
            "CanonicalProductID",
        ]
    )
    .reset_index(drop=True)
)

context_metadata_duplicate_count = int(
    context_metadata_df.duplicated(
        subset=["SplitContextID"],
        keep=False,
    ).sum()
)

if context_metadata_duplicate_count > 0:

    duplicate_contexts_df = context_metadata_df.loc[
        context_metadata_df.duplicated(
            subset=["SplitContextID"],
            keep=False,
        )
    ].copy()

    print("\nDUPLICATE CONTEXT METADATA")
    display(duplicate_contexts_df)

    raise AssertionError(
        "Each SplitContextID must map to exactly one product, "
        "window and demand-pattern class."
    )

context_actual_summary_df = (
    validation_df
    .groupby(
        "SplitContextID",
        as_index=False,
    )
    .agg(
        ContextActualDemandTotal=(
            "TotalDemand",
            "sum",
        ),
        ContextPositiveDemandDays=(
            "TotalDemand",
            lambda values: int(
                (values > 0).sum()
            ),
        ),
        ContextZeroDemandDays=(
            "TotalDemand",
            lambda values: int(
                (values == 0).sum()
            ),
        ),
        ContextObservationCount=(
            "TotalDemand",
            "size",
        ),
    )
)

context_actual_summary_df[
    "AllZeroActualWindow"
] = (
    context_actual_summary_df[
        "ContextActualDemandTotal"
    ] == 0
)

context_actual_summary_df[
    "ContextZeroDemandRate"
] = (
    context_actual_summary_df[
        "ContextZeroDemandDays"
    ]
    / context_actual_summary_df[
        "ContextObservationCount"
    ]
)

context_metadata_df = context_metadata_df.merge(
    context_actual_summary_df,
    on="SplitContextID",
    how="left",
    validate="one_to_one",
)

context_lookup_df = context_metadata_df.set_index(
    "SplitContextID"
)

all_zero_context_count = int(
    context_metadata_df[
        "AllZeroActualWindow"
    ].sum()
)

positive_demand_context_count = int(
    (
        ~context_metadata_df[
            "AllZeroActualWindow"
        ]
    ).sum()
)


# --------------------------------------------------------------
# 4. Calculate metrics for every baseline and context
# --------------------------------------------------------------

context_metric_records = []

for (
    model_name,
    model_family,
    split_context_id,
), group_df in baseline_validation_predictions_df.groupby(
    [
        "ModelName",
        "ModelFamily",
        "SplitContextID",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    context_metadata = context_lookup_df.loc[
        split_context_id
    ]

    metric_record = {
        "BaselineName": model_name,
        "BaselineFamily": model_family,
        "SplitContextID": split_context_id,
        "WindowID": context_metadata["WindowID"],
        "CanonicalProductID":
            context_metadata["CanonicalProductID"],
        "CanonicalProductName":
            context_metadata["CanonicalProductName"],
        "DemandPatternClass":
            context_metadata["DemandPatternClass"],
        "AllZeroActualWindow": bool(
            context_metadata["AllZeroActualWindow"]
        ),
        "PositiveDemandDays": int(
            context_metadata["ContextPositiveDemandDays"]
        ),
        "ZeroDemandDays": int(
            context_metadata["ContextZeroDemandDays"]
        ),
        "ContextZeroDemandRate": float(
            context_metadata["ContextZeroDemandRate"]
        ),
    }

    metric_record.update(
        metric_values
    )

    context_metric_records.append(
        metric_record
    )

baseline_product_window_metrics_df = pd.DataFrame(
    context_metric_records
)


# --------------------------------------------------------------
# 5. Add context rankings and tie-aware wins
# --------------------------------------------------------------

baseline_product_window_metrics_df[
    "BestMAEInContext"
] = (
    baseline_product_window_metrics_df
    .groupby("SplitContextID")["MAE"]
    .transform("min")
)

baseline_product_window_metrics_df[
    "IsContextBestMAE"
] = np.isclose(
    baseline_product_window_metrics_df["MAE"],
    baseline_product_window_metrics_df[
        "BestMAEInContext"
    ],
    rtol=1e-12,
    atol=1e-12,
)

baseline_product_window_metrics_df[
    "ContextBestTieCount"
] = (
    baseline_product_window_metrics_df
    .groupby("SplitContextID")[
        "IsContextBestMAE"
    ]
    .transform("sum")
    .astype(int)
)

baseline_product_window_metrics_df[
    "FractionalContextWin"
] = np.where(
    baseline_product_window_metrics_df[
        "IsContextBestMAE"
    ],
    1.0
    / baseline_product_window_metrics_df[
        "ContextBestTieCount"
    ],
    0.0,
)

baseline_product_window_metrics_df[
    "MAERankWithinContext"
] = (
    baseline_product_window_metrics_df
    .groupby("SplitContextID")["MAE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_product_window_metrics_df = (
    baseline_product_window_metrics_df
    .sort_values(
        [
            "WindowID",
            "CanonicalProductID",
            "MAERankWithinContext",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 6. Summarise context-level performance
# --------------------------------------------------------------

context_summary_records = []

for (
    baseline_name,
    baseline_family,
), group_df in baseline_product_window_metrics_df.groupby(
    [
        "BaselineName",
        "BaselineFamily",
    ],
    sort=True,
):

    all_zero_df = group_df.loc[
        group_df["AllZeroActualWindow"]
    ]

    positive_demand_df = group_df.loc[
        ~group_df["AllZeroActualWindow"]
    ]

    context_summary_records.append(
        {
            "BaselineName": baseline_name,
            "BaselineFamily": baseline_family,
            "ContextCount": int(
                len(group_df)
            ),
            "AllZeroContextCount": int(
                len(all_zero_df)
            ),
            "PositiveDemandContextCount": int(
                len(positive_demand_df)
            ),
            "MeanContextMAE": float(
                group_df["MAE"].mean()
            ),
            "MedianContextMAE": float(
                group_df["MAE"].median()
            ),
            "MeanAllZeroContextMAE": float(
                all_zero_df["MAE"].mean()
            ),
            "MeanPositiveDemandContextMAE": float(
                positive_demand_df["MAE"].mean()
            ),
            "MedianPositiveDemandContextMAE": float(
                positive_demand_df["MAE"].median()
            ),
            "MeanPositiveDemandContextWAPE_Percent": float(
                positive_demand_df[
                    "WAPE_Percent"
                ].mean()
            ),
            "ExactBestContextCount": int(
                group_df["IsContextBestMAE"].sum()
            ),
            "FractionalContextWins": float(
                group_df["FractionalContextWin"].sum()
            ),
            "FirstPlaceRankCount": int(
                (
                    group_df["MAERankWithinContext"]
                    == 1
                ).sum()
            ),
        }
    )

baseline_context_summary_df = pd.DataFrame(
    context_summary_records
)

baseline_context_summary_df = (
    baseline_context_summary_df
    .sort_values(
        [
            "MeanContextMAE",
            "MeanPositiveDemandContextMAE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

baseline_context_summary_df.insert(
    0,
    "MeanContextMAE_Rank",
    np.arange(
        1,
        len(baseline_context_summary_df) + 1,
    ),
)


# --------------------------------------------------------------
# 7. Attach demand-pattern metadata to predictions
# --------------------------------------------------------------

prediction_analysis_df = (
    baseline_validation_predictions_df
    .merge(
        context_metadata_df[
            [
                "SplitContextID",
                "DemandPatternClass",
                "AllZeroActualWindow",
            ]
        ],
        on="SplitContextID",
        how="left",
        validate="many_to_one",
    )
)

if prediction_analysis_df[
    "DemandPatternClass"
].isna().any():

    raise AssertionError(
        "At least one prediction could not be mapped "
        "to a demand-pattern class."
    )

prediction_analysis_df[
    "WindowDemandStatus"
] = np.where(
    prediction_analysis_df[
        "AllZeroActualWindow"
    ],
    "ALL_ZERO_ACTUAL",
    "POSITIVE_ACTUAL_PRESENT",
)


# --------------------------------------------------------------
# 8. Calculate pooled metrics by demand pattern
# --------------------------------------------------------------

demand_pattern_metric_records = []

for (
    model_name,
    model_family,
    demand_pattern_class,
), group_df in prediction_analysis_df.groupby(
    [
        "ModelName",
        "ModelFamily",
        "DemandPatternClass",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    context_ids = group_df[
        "SplitContextID"
    ].unique()

    pattern_context_df = context_metadata_df.loc[
        context_metadata_df["SplitContextID"].isin(
            context_ids
        )
    ]

    metric_record = {
        "BaselineName": model_name,
        "BaselineFamily": model_family,
        "DemandPatternClass": demand_pattern_class,
        "ProductWindowContexts": int(
            group_df["SplitContextID"].nunique()
        ),
        "Products": int(
            group_df["CanonicalProductID"].nunique()
        ),
        "AllZeroContextCount": int(
            pattern_context_df[
                "AllZeroActualWindow"
            ].sum()
        ),
        "PositiveDemandContextCount": int(
            (
                ~pattern_context_df[
                    "AllZeroActualWindow"
                ]
            ).sum()
        ),
    }

    metric_record.update(
        metric_values
    )

    demand_pattern_metric_records.append(
        metric_record
    )

baseline_demand_pattern_metrics_df = pd.DataFrame(
    demand_pattern_metric_records
)

baseline_demand_pattern_metrics_df[
    "MAERankWithinDemandPattern"
] = (
    baseline_demand_pattern_metrics_df
    .groupby("DemandPatternClass")["MAE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_demand_pattern_metrics_df[
    "IsBestMAEForDemandPattern"
] = (
    baseline_demand_pattern_metrics_df[
        "MAERankWithinDemandPattern"
    ] == 1
)

baseline_demand_pattern_metrics_df = (
    baseline_demand_pattern_metrics_df
    .sort_values(
        [
            "DemandPatternClass",
            "MAERankWithinDemandPattern",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 9. Compare all-zero and positive-demand windows
# --------------------------------------------------------------

window_status_metric_records = []

for (
    model_name,
    model_family,
    window_demand_status,
), group_df in prediction_analysis_df.groupby(
    [
        "ModelName",
        "ModelFamily",
        "WindowDemandStatus",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    metric_record = {
        "BaselineName": model_name,
        "BaselineFamily": model_family,
        "WindowDemandStatus": window_demand_status,
        "ProductWindowContexts": int(
            group_df["SplitContextID"].nunique()
        ),
        "Products": int(
            group_df["CanonicalProductID"].nunique()
        ),
    }

    metric_record.update(
        metric_values
    )

    window_status_metric_records.append(
        metric_record
    )

baseline_window_status_metrics_df = pd.DataFrame(
    window_status_metric_records
)

baseline_window_status_metrics_df[
    "MAERankWithinStatus"
] = (
    baseline_window_status_metrics_df
    .groupby("WindowDemandStatus")["MAE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_window_status_metrics_df = (
    baseline_window_status_metrics_df
    .sort_values(
        [
            "WindowDemandStatus",
            "MAERankWithinStatus",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 10. Create one metadata row per product
# --------------------------------------------------------------

product_metadata_df = (
    validation_df[
        [
            "CanonicalProductID",
            "CanonicalProductName",
            "DemandPatternClass",
        ]
    ]
    .drop_duplicates()
    .sort_values("CanonicalProductID")
    .reset_index(drop=True)
)

product_metadata_duplicate_count = int(
    product_metadata_df.duplicated(
        subset=["CanonicalProductID"],
        keep=False,
    ).sum()
)

if product_metadata_duplicate_count > 0:

    duplicate_product_metadata_df = (
        product_metadata_df.loc[
            product_metadata_df.duplicated(
                subset=["CanonicalProductID"],
                keep=False,
            )
        ]
        .copy()
    )

    print("\nDUPLICATE PRODUCT METADATA")
    display(duplicate_product_metadata_df)

    raise AssertionError(
        "Each CanonicalProductID must map to exactly one "
        "product name and demand-pattern class."
    )

product_lookup_df = product_metadata_df.set_index(
    "CanonicalProductID"
)


# --------------------------------------------------------------
# 11. Calculate combined two-fold product metrics
# --------------------------------------------------------------

product_metric_records = []

for (
    model_name,
    model_family,
    canonical_product_id,
), group_df in prediction_analysis_df.groupby(
    [
        "ModelName",
        "ModelFamily",
        "CanonicalProductID",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    product_metadata = product_lookup_df.loc[
        canonical_product_id
    ]

    metric_record = {
        "BaselineName": model_name,
        "BaselineFamily": model_family,
        "CanonicalProductID": canonical_product_id,
        "CanonicalProductName":
            product_metadata["CanonicalProductName"],
        "DemandPatternClass":
            product_metadata["DemandPatternClass"],
        "ValidationFoldCount": int(
            group_df["WindowID"].nunique()
        ),
        "AllZeroActualProduct": bool(
            group_df["ActualDemand"].sum() == 0
        ),
    }

    metric_record.update(
        metric_values
    )

    product_metric_records.append(
        metric_record
    )

baseline_product_combined_metrics_df = pd.DataFrame(
    product_metric_records
)


# --------------------------------------------------------------
# 12. Add product rankings and tie-aware wins
# --------------------------------------------------------------

baseline_product_combined_metrics_df[
    "BestMAEForProduct"
] = (
    baseline_product_combined_metrics_df
    .groupby("CanonicalProductID")["MAE"]
    .transform("min")
)

baseline_product_combined_metrics_df[
    "IsProductBestMAE"
] = np.isclose(
    baseline_product_combined_metrics_df["MAE"],
    baseline_product_combined_metrics_df[
        "BestMAEForProduct"
    ],
    rtol=1e-12,
    atol=1e-12,
)

baseline_product_combined_metrics_df[
    "ProductBestTieCount"
] = (
    baseline_product_combined_metrics_df
    .groupby("CanonicalProductID")[
        "IsProductBestMAE"
    ]
    .transform("sum")
    .astype(int)
)

baseline_product_combined_metrics_df[
    "FractionalProductWin"
] = np.where(
    baseline_product_combined_metrics_df[
        "IsProductBestMAE"
    ],
    1.0
    / baseline_product_combined_metrics_df[
        "ProductBestTieCount"
    ],
    0.0,
)

baseline_product_combined_metrics_df[
    "MAERankWithinProduct"
] = (
    baseline_product_combined_metrics_df
    .groupby("CanonicalProductID")["MAE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_product_combined_metrics_df = (
    baseline_product_combined_metrics_df
    .sort_values(
        [
            "CanonicalProductID",
            "MAERankWithinProduct",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 13. Summarise product-level performance
# --------------------------------------------------------------

product_summary_records = []

for (
    baseline_name,
    baseline_family,
), group_df in baseline_product_combined_metrics_df.groupby(
    [
        "BaselineName",
        "BaselineFamily",
    ],
    sort=True,
):

    positive_product_df = group_df.loc[
        ~group_df["AllZeroActualProduct"]
    ]

    all_zero_product_df = group_df.loc[
        group_df["AllZeroActualProduct"]
    ]

    product_summary_records.append(
        {
            "BaselineName": baseline_name,
            "BaselineFamily": baseline_family,
            "ProductCount": int(
                len(group_df)
            ),
            "AllZeroProductCount": int(
                len(all_zero_product_df)
            ),
            "PositiveDemandProductCount": int(
                len(positive_product_df)
            ),
            "MeanProductMAE": float(
                group_df["MAE"].mean()
            ),
            "MedianProductMAE": float(
                group_df["MAE"].median()
            ),
            "MeanPositiveProductMAE": float(
                positive_product_df["MAE"].mean()
            ),
            "MedianPositiveProductMAE": float(
                positive_product_df["MAE"].median()
            ),
            "MeanPositiveProductWAPE_Percent": float(
                positive_product_df[
                    "WAPE_Percent"
                ].mean()
            ),
            "ExactBestProductCount": int(
                group_df["IsProductBestMAE"].sum()
            ),
            "FractionalProductWins": float(
                group_df["FractionalProductWin"].sum()
            ),
            "FirstPlaceRankCount": int(
                (
                    group_df["MAERankWithinProduct"]
                    == 1
                ).sum()
            ),
        }
    )

baseline_product_summary_df = pd.DataFrame(
    product_summary_records
)

baseline_product_summary_df = (
    baseline_product_summary_df
    .sort_values(
        [
            "MeanProductMAE",
            "MeanPositiveProductMAE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

baseline_product_summary_df.insert(
    0,
    "MeanProductMAE_Rank",
    np.arange(
        1,
        len(baseline_product_summary_df) + 1,
    ),
)


# --------------------------------------------------------------
# 14. Create diagnostics for the best baseline
# --------------------------------------------------------------

best_overall_baseline_name = str(
    baseline_ranking_df
    .sort_values("MAE_Rank")
    .iloc[0]["BaselineName"]
)

best_baseline_product_diagnostics_df = (
    baseline_product_combined_metrics_df.loc[
        baseline_product_combined_metrics_df[
            "BaselineName"
        ] == best_overall_baseline_name
    ]
    .sort_values(
        [
            "ActualDemandTotal",
            "MAE",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 15. Calculate validation values
# --------------------------------------------------------------

expected_context_metric_rows = int(
    len(baseline_names)
    * len(validation_context_ids)
)

expected_demand_pattern_metric_rows = int(
    len(baseline_names)
    * validation_df["DemandPatternClass"].nunique()
)

expected_window_status_metric_rows = int(
    len(baseline_names)
    * context_metadata_df[
        "AllZeroActualWindow"
    ].nunique()
)

expected_product_metric_rows = int(
    len(baseline_names)
    * len(validation_product_ids)
)

context_metric_duplicate_count = int(
    baseline_product_window_metrics_df.duplicated(
        subset=[
            "BaselineName",
            "SplitContextID",
        ]
    ).sum()
)

context_metric_missing_error_count = int(
    baseline_product_window_metrics_df[
        [
            "MAE",
            "RMSE",
        ]
    ].isna().sum().sum()
)

all_zero_context_wape_defined_count = int(
    baseline_product_window_metrics_df.loc[
        baseline_product_window_metrics_df[
            "AllZeroActualWindow"
        ],
        "WAPE_Percent",
    ].notna().sum()
)

positive_context_wape_missing_count = int(
    baseline_product_window_metrics_df.loc[
        ~baseline_product_window_metrics_df[
            "AllZeroActualWindow"
        ],
        "WAPE_Percent",
    ].isna().sum()
)

fractional_context_win_total = float(
    baseline_product_window_metrics_df[
        "FractionalContextWin"
    ].sum()
)

demand_pattern_wape_missing_count = int(
    baseline_demand_pattern_metrics_df[
        "WAPE_Percent"
    ].isna().sum()
)

all_zero_status_wape_defined_count = int(
    baseline_window_status_metrics_df.loc[
        baseline_window_status_metrics_df[
            "WindowDemandStatus"
        ] == "ALL_ZERO_ACTUAL",
        "WAPE_Percent",
    ].notna().sum()
)

positive_status_wape_missing_count = int(
    baseline_window_status_metrics_df.loc[
        baseline_window_status_metrics_df[
            "WindowDemandStatus"
        ] == "POSITIVE_ACTUAL_PRESENT",
        "WAPE_Percent",
    ].isna().sum()
)

product_metric_duplicate_count = int(
    baseline_product_combined_metrics_df.duplicated(
        subset=[
            "BaselineName",
            "CanonicalProductID",
        ]
    ).sum()
)

all_zero_product_wape_defined_count = int(
    baseline_product_combined_metrics_df.loc[
        baseline_product_combined_metrics_df[
            "AllZeroActualProduct"
        ],
        "WAPE_Percent",
    ].notna().sum()
)

positive_product_wape_missing_count = int(
    baseline_product_combined_metrics_df.loc[
        ~baseline_product_combined_metrics_df[
            "AllZeroActualProduct"
        ],
        "WAPE_Percent",
    ].isna().sum()
)

fractional_product_win_total = float(
    baseline_product_combined_metrics_df[
        "FractionalProductWin"
    ].sum()
)


# --------------------------------------------------------------
# 16. Build validation checks
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one validation result.
    """

    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Baseline methods",
    len(baseline_names),
    7,
)

add_check(
    "Validation windows",
    len(validation_windows),
    2,
)

add_check(
    "Product-window contexts",
    len(context_metadata_df),
    254,
)

add_check(
    "Duplicate context metadata rows",
    context_metadata_duplicate_count,
    0,
)

add_check(
    "All-zero plus positive-demand contexts",
    all_zero_context_count
    + positive_demand_context_count,
    len(context_metadata_df),
)

add_check(
    "Product-window metric rows",
    len(baseline_product_window_metrics_df),
    expected_context_metric_rows,
)

add_check(
    "Duplicate baseline-context metrics",
    context_metric_duplicate_count,
    0,
)

add_check(
    "Missing context MAE or RMSE values",
    context_metric_missing_error_count,
    0,
)

add_check(
    "All-zero contexts with defined WAPE",
    all_zero_context_wape_defined_count,
    0,
)

add_check(
    "Positive-demand contexts with missing WAPE",
    positive_context_wape_missing_count,
    0,
)

add_check(
    "Fractional context wins",
    fractional_context_win_total,
    float(len(context_metadata_df)),
    passed=np.isclose(
        fractional_context_win_total,
        float(len(context_metadata_df)),
    ),
)

add_check(
    "Demand-pattern classes",
    validation_df[
        "DemandPatternClass"
    ].nunique(),
    4,
)

add_check(
    "Demand-pattern metric rows",
    len(baseline_demand_pattern_metrics_df),
    expected_demand_pattern_metric_rows,
)

add_check(
    "Demand-pattern metrics with missing WAPE",
    demand_pattern_wape_missing_count,
    0,
)

add_check(
    "Window-status metric rows",
    len(baseline_window_status_metrics_df),
    expected_window_status_metric_rows,
)

add_check(
    "All-zero status rows with defined WAPE",
    all_zero_status_wape_defined_count,
    0,
)

add_check(
    "Positive status rows with missing WAPE",
    positive_status_wape_missing_count,
    0,
)

add_check(
    "Product metadata rows",
    len(product_metadata_df),
    127,
)

add_check(
    "Duplicate product metadata rows",
    product_metadata_duplicate_count,
    0,
)

add_check(
    "Combined product metric rows",
    len(baseline_product_combined_metrics_df),
    expected_product_metric_rows,
)

add_check(
    "Duplicate baseline-product metrics",
    product_metric_duplicate_count,
    0,
)

add_check(
    "All-zero products with defined WAPE",
    all_zero_product_wape_defined_count,
    0,
)

add_check(
    "Positive-demand products with missing WAPE",
    positive_product_wape_missing_count,
    0,
)

add_check(
    "Fractional product wins",
    fractional_product_win_total,
    float(len(product_metadata_df)),
    passed=np.isclose(
        fractional_product_win_total,
        float(len(product_metadata_df)),
    ),
)

add_check(
    "Best-baseline product diagnostic rows",
    len(best_baseline_product_diagnostics_df),
    len(product_metadata_df),
)

part3_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 17. Stop if validation fails
# --------------------------------------------------------------

failed_checks_df = part3_validation_df.loc[
    ~part3_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print(
        "\nFAILED MODELLING STEP 2 PART 3 CHECKS"
    )

    display(
        failed_checks_df
    )

    raise AssertionError(
        "Modelling Step 2 Part 3 failed. "
        "Do not proceed to aggregate baseline analysis."
    )


# --------------------------------------------------------------
# 18. Save outputs
# --------------------------------------------------------------

product_window_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_baseline_product_window_metrics.csv"
)

context_summary_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_baseline_context_summary.csv"
)

demand_pattern_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_baseline_demand_pattern_metrics.csv"
)

window_status_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_baseline_window_status_metrics.csv"
)

product_combined_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_baseline_product_combined_metrics.csv"
)

product_summary_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_baseline_product_summary.csv"
)

best_baseline_product_diagnostics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_best_baseline_product_diagnostics.csv"
)

part3_validation_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "03_baseline_diagnostic_validation_summary.csv"
)

baseline_product_window_metrics_df.to_csv(
    product_window_metrics_path,
    index=False,
)

baseline_context_summary_df.to_csv(
    context_summary_path,
    index=False,
)

baseline_demand_pattern_metrics_df.to_csv(
    demand_pattern_metrics_path,
    index=False,
)

baseline_window_status_metrics_df.to_csv(
    window_status_metrics_path,
    index=False,
)

baseline_product_combined_metrics_df.to_csv(
    product_combined_metrics_path,
    index=False,
)

baseline_product_summary_df.to_csv(
    product_summary_path,
    index=False,
)

best_baseline_product_diagnostics_df.to_csv(
    best_baseline_product_diagnostics_path,
    index=False,
)

part3_validation_df.to_csv(
    part3_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 19. Prepare final display values
# --------------------------------------------------------------

best_context_baseline_name = str(
    baseline_context_summary_df.iloc[0][
        "BaselineName"
    ]
)

best_product_baseline_name = str(
    baseline_product_summary_df.iloc[0][
        "BaselineName"
    ]
)

passed_check_count = int(
    part3_validation_df["Passed"].sum()
)

total_check_count = int(
    len(part3_validation_df)
)

pattern_winners_df = (
    baseline_demand_pattern_metrics_df.loc[
        baseline_demand_pattern_metrics_df[
            "IsBestMAEForDemandPattern"
        ]
    ]
    .copy()
)


# --------------------------------------------------------------
# 20. Display final results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 2 PART 3: PASSED")
print("=" * 72)

print(
    "\nProduct-window contexts: "
    f"{len(context_metadata_df)}"
)

print(
    "All-zero product-window contexts: "
    f"{all_zero_context_count}"
)

print(
    "Positive-demand product-window contexts: "
    f"{positive_demand_context_count}"
)

print(
    "Product-window metric rows: "
    f"{len(baseline_product_window_metrics_df):,}"
)

print(
    "Combined product metric rows: "
    f"{len(baseline_product_combined_metrics_df):,}"
)

print(
    "Best baseline by mean context MAE: "
    f"{best_context_baseline_name}"
)

print(
    "Best baseline by mean product MAE: "
    f"{best_product_baseline_name}"
)

print(
    "Overall best baseline from Part 2: "
    f"{best_overall_baseline_name}"
)

print(
    "\nValidation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print("\nContext-level baseline summary:")
display(
    baseline_context_summary_df
)

print("\nDemand-pattern winners by MAE:")
display(
    pattern_winners_df[
        [
            "DemandPatternClass",
            "BaselineName",
            "MAE",
            "RMSE",
            "WAPE_Percent",
            "MeanBias",
            "ProductWindowContexts",
            "AllZeroContextCount",
            "PositiveDemandContextCount",
        ]
    ]
)

print("\nAll-zero versus positive-demand window metrics:")
display(
    baseline_window_status_metrics_df
)

print("\nProduct-level baseline summary:")
display(
    baseline_product_summary_df
)

print("\nPart 3 validation summary:")
display(
    part3_validation_df
)

print("\nSaved outputs:")
print(product_window_metrics_path)
print(context_summary_path)
print(demand_pattern_metrics_path)
print(window_status_metrics_path)
print(product_combined_metrics_path)
print(product_summary_path)
print(best_baseline_product_diagnostics_path)
print(part3_validation_path)


MODELLING STEP 2 PART 3: PASSED

Product-window contexts: 254
All-zero product-window contexts: 210
Positive-demand product-window contexts: 44
Product-window metric rows: 1,778
Combined product metric rows: 889
Best baseline by mean context MAE: MOVING_AVERAGE_5
Best baseline by mean product MAE: MOVING_AVERAGE_5
Overall best baseline from Part 2: MOVING_AVERAGE_5

Validation checks passed: 25 of 25

Context-level baseline summary:


,MeanContextMAE_Rank,BaselineName,BaselineFamily,ContextCount,AllZeroContextCount,PositiveDemandContextCount,MeanContextMAE,MedianContextMAE,MeanAllZeroContextMAE,MeanPositiveDemandContextMAE,MedianPositiveDemandContextMAE,MeanPositiveDemandContextWAPE_Percent,ExactBestContextCount,FractionalContextWins,FirstPlaceRankCount
0,1,MOVING_AVERAGE_5,MOVING_AVERAGE,254,210,44,0.643425,0.000000,0.000000,3.714318,3.125000,58.789630,218,43.0,218
1,2,MOVING_AVERAGE_10,MOVING_AVERAGE,254,210,44,0.647087,0.000000,0.000000,3.735455,3.202500,58.457772,219,44.0,219
2,3,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,254,210,44,0.680315,0.000000,0.000000,3.927273,3.400000,63.437779,222,47.0,222
3,4,MOVING_AVERAGE_20,MOVING_AVERAGE,254,210,44,0.686526,0.000000,0.000000,3.963125,3.481250,62.106008,217,42.0,217
4,5,LAST_OBSERVED_DEMAND,NAIVE,254,210,44,0.793701,0.000000,0.000000,4.581818,3.600000,71.263477,214,39.0,214
5,6,ZERO_DEMAND,CONSTANT,254,210,44,1.597835,0.000000,0.000000,9.223864,5.650000,100.000000,213,38.0,213
6,7,HISTORICAL_MEAN,HISTORICAL_AVERAGE,254,210,44,2.189040,0.707563,1.540291,5.285345,3.478484,84.232601,1,1.0,1



Demand-pattern winners by MAE:


,DemandPatternClass,BaselineName,MAE,RMSE,WAPE_Percent,MeanBias,ProductWindowContexts,AllZeroContextCount,PositiveDemandContextCount
0,ERRATIC,MOVING_AVERAGE_5,4.899500,6.779712,37.863215,-0.907500,20,0,20
7,INTERMITTENT,MOVING_AVERAGE_20,0.056414,0.356031,71.528302,-0.014301,168,160,8
14,LUMPY,MOVING_AVERAGE_10,0.155714,0.628789,65.074627,-0.030893,56,50,6
21,SMOOTH,MOVING_AVERAGE_20,4.575250,6.915429,38.000415,-1.255250,10,0,10



All-zero versus positive-demand window metrics:


...


Product-level baseline summary:


,MeanProductMAE_Rank,BaselineName,BaselineFamily,ProductCount,AllZeroProductCount,PositiveDemandProductCount,MeanProductMAE,MedianProductMAE,MeanPositiveProductMAE,MedianPositiveProductMAE,MeanPositiveProductWAPE_Percent,ExactBestProductCount,FractionalProductWins,FirstPlaceRankCount
0,1,MOVING_AVERAGE_5,MOVING_AVERAGE,127,105,22,0.643425,0.000000,3.714318,3.125000,55.922453,111,23.0,111
1,2,MOVING_AVERAGE_10,MOVING_AVERAGE,127,105,22,0.647087,0.000000,3.735455,3.336250,55.553929,110,22.0,109
2,3,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,127,105,22,0.680315,0.000000,3.927273,3.425000,59.686539,111,23.5,111
3,4,MOVING_AVERAGE_20,MOVING_AVERAGE,127,105,22,0.686526,0.000000,3.963125,3.552500,58.936334,108,20.5,108
4,5,LAST_OBSERVED_DEMAND,NAIVE,127,105,22,0.793701,0.000000,4.581818,3.712500,68.344675,106,18.5,106
5,6,ZERO_DEMAND,CONSTANT,127,105,22,1.597835,0.000000,9.223864,7.087500,100.000000,107,19.5,107
6,7,HISTORICAL_MEAN,HISTORICAL_AVERAGE,127,105,22,2.189040,0.724697,5.285345,3.978518,79.334919,0,0.0,0



Part 3 validation summary:


,Check,Expected,Actual,Passed
0,Baseline methods,7,7,True
1,Validation windows,2,2,True
2,Product-window contexts,254,254,True
3,Duplicate context metadata rows,0,0,True
4,All-zero plus positive-demand contexts,254,254,True
5,Product-window metric rows,1778,1778,True
6,Duplicate baseline-context metrics,0,0,True
7,Missing context MAE or RMSE values,0,0,True
8,All-zero contexts with defined WAPE,0,0,True
9,Positive-demand contexts with missing WAPE,0,0,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_baseline_product_window_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_baseline_context_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_baseline_demand_pattern_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_baseline_window_status_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_baseline_product_combined_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_baseline_product_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_best_baseline_product_diagnostics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/03_baseline_diagnostic_validation_summary.csv


In [19]:
# ==============================================================
# MODELLING STEP 2, PART 4
# Aggregate daily-demand baseline analysis
# ==============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------
# 1. Confirm required objects
# --------------------------------------------------------------

required_objects = [
    "validation_df",
    "baseline_validation_predictions_df",
    "baseline_ranking_df",
    "calculate_forecast_metrics",
    "BASELINE_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 2 Parts 1 to 3 first."
    )


# --------------------------------------------------------------
# 2. Validate required columns
# --------------------------------------------------------------

required_validation_columns = [
    "Date",
    "WindowID",
    "CanonicalProductID",
    "SplitContextID",
    "TotalDemand",
]

required_prediction_columns = [
    "Date",
    "WindowID",
    "CanonicalProductID",
    "SplitContextID",
    "ActualDemand",
    "PredictedDemand",
    "PredictionClippedAtZero",
    "ModelName",
    "ModelFamily",
]

missing_validation_columns = [
    column
    for column in required_validation_columns
    if column not in validation_df.columns
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in baseline_validation_predictions_df.columns
]

if missing_validation_columns:
    raise KeyError(
        "The validation dataset is missing required columns:\n"
        f"{missing_validation_columns}"
    )

if missing_prediction_columns:
    raise KeyError(
        "The baseline prediction table is missing required columns:\n"
        f"{missing_prediction_columns}"
    )

validation_df["Date"] = pd.to_datetime(
    validation_df["Date"],
    errors="raise",
)

baseline_validation_predictions_df["Date"] = pd.to_datetime(
    baseline_validation_predictions_df["Date"],
    errors="raise",
)

baseline_names = sorted(
    baseline_validation_predictions_df["ModelName"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

validation_windows = sorted(
    validation_df["WindowID"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

validation_dates = (
    validation_df[
        [
            "WindowID",
            "Date",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "WindowID",
            "Date",
        ]
    )
    .reset_index(drop=True)
)

validation_date_count = int(
    len(validation_dates)
)


# --------------------------------------------------------------
# 3. Build the authoritative daily actual-demand reference
# --------------------------------------------------------------

aggregate_actual_reference_df = (
    validation_df
    .groupby(
        [
            "WindowID",
            "Date",
        ],
        as_index=False,
    )
    .agg(
        ReferenceActualDemand=(
            "TotalDemand",
            "sum",
        ),
        ReferenceProductRows=(
            "CanonicalProductID",
            "size",
        ),
        ReferenceProductCount=(
            "CanonicalProductID",
            "nunique",
        ),
        ReferenceSplitContextCount=(
            "SplitContextID",
            "nunique",
        ),
    )
    .sort_values(
        [
            "WindowID",
            "Date",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 4. Aggregate product-level forecasts to daily totals
#
# These are bottom-up total-demand forecasts created by summing
# the 127 product forecasts for each operating date.
# --------------------------------------------------------------

baseline_aggregate_daily_predictions_df = (
    baseline_validation_predictions_df
    .groupby(
        [
            "ModelName",
            "ModelFamily",
            "WindowID",
            "Date",
        ],
        as_index=False,
    )
    .agg(
        ActualDemand=(
            "ActualDemand",
            "sum",
        ),
        PredictedDemand=(
            "PredictedDemand",
            "sum",
        ),
        ProductPredictionRows=(
            "CanonicalProductID",
            "size",
        ),
        ProductCount=(
            "CanonicalProductID",
            "nunique",
        ),
        SplitContextCount=(
            "SplitContextID",
            "nunique",
        ),
        ClippedProductPredictions=(
            "PredictionClippedAtZero",
            "sum",
        ),
    )
)

baseline_aggregate_daily_predictions_df = (
    baseline_aggregate_daily_predictions_df
    .merge(
        aggregate_actual_reference_df,
        on=[
            "WindowID",
            "Date",
        ],
        how="left",
        validate="many_to_one",
    )
)

baseline_aggregate_daily_predictions_df[
    "ActualDemandMatchesReference"
] = np.isclose(
    baseline_aggregate_daily_predictions_df[
        "ActualDemand"
    ],
    baseline_aggregate_daily_predictions_df[
        "ReferenceActualDemand"
    ],
    rtol=1e-12,
    atol=1e-12,
)

baseline_aggregate_daily_predictions_df[
    "ForecastError"
] = (
    baseline_aggregate_daily_predictions_df[
        "PredictedDemand"
    ]
    - baseline_aggregate_daily_predictions_df[
        "ActualDemand"
    ]
)

baseline_aggregate_daily_predictions_df[
    "AbsoluteError"
] = (
    baseline_aggregate_daily_predictions_df[
        "ForecastError"
    ].abs()
)

baseline_aggregate_daily_predictions_df[
    "SquaredError"
] = (
    baseline_aggregate_daily_predictions_df[
        "ForecastError"
    ] ** 2
)


# --------------------------------------------------------------
# 5. Add tie-aware daily winner indicators
# --------------------------------------------------------------

baseline_aggregate_daily_predictions_df[
    "BestAbsoluteErrorForDate"
] = (
    baseline_aggregate_daily_predictions_df
    .groupby(
        [
            "WindowID",
            "Date",
        ]
    )["AbsoluteError"]
    .transform("min")
)

baseline_aggregate_daily_predictions_df[
    "IsDailyBestAbsoluteError"
] = np.isclose(
    baseline_aggregate_daily_predictions_df[
        "AbsoluteError"
    ],
    baseline_aggregate_daily_predictions_df[
        "BestAbsoluteErrorForDate"
    ],
    rtol=1e-12,
    atol=1e-12,
)

baseline_aggregate_daily_predictions_df[
    "DailyBestTieCount"
] = (
    baseline_aggregate_daily_predictions_df
    .groupby(
        [
            "WindowID",
            "Date",
        ]
    )["IsDailyBestAbsoluteError"]
    .transform("sum")
    .astype(int)
)

baseline_aggregate_daily_predictions_df[
    "FractionalDailyWin"
] = np.where(
    baseline_aggregate_daily_predictions_df[
        "IsDailyBestAbsoluteError"
    ],
    1.0
    / baseline_aggregate_daily_predictions_df[
        "DailyBestTieCount"
    ],
    0.0,
)

baseline_aggregate_daily_predictions_df = (
    baseline_aggregate_daily_predictions_df
    .sort_values(
        [
            "ModelName",
            "WindowID",
            "Date",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 6. Calculate aggregate metrics separately for each fold
# --------------------------------------------------------------

aggregate_fold_metric_records = []

for (
    model_name,
    model_family,
    window_id,
), group_df in baseline_aggregate_daily_predictions_df.groupby(
    [
        "ModelName",
        "ModelFamily",
        "WindowID",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    metric_record = {
        "BaselineName": model_name,
        "BaselineFamily": model_family,
        "WindowID": window_id,
    }

    metric_record.update(
        metric_values
    )

    aggregate_fold_metric_records.append(
        metric_record
    )

baseline_aggregate_fold_metrics_df = pd.DataFrame(
    aggregate_fold_metric_records
)

baseline_aggregate_fold_metrics_df[
    "AggregateMAERankWithinFold"
] = (
    baseline_aggregate_fold_metrics_df
    .groupby("WindowID")["MAE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_aggregate_fold_metrics_df = (
    baseline_aggregate_fold_metrics_df
    .sort_values(
        [
            "WindowID",
            "AggregateMAERankWithinFold",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 7. Calculate combined metrics across both folds
# --------------------------------------------------------------

aggregate_combined_metric_records = []

for (
    model_name,
    model_family,
), group_df in baseline_aggregate_daily_predictions_df.groupby(
    [
        "ModelName",
        "ModelFamily",
    ],
    sort=True,
):

    metric_values = calculate_forecast_metrics(
        actual_values=group_df["ActualDemand"],
        predicted_values=group_df["PredictedDemand"],
    )

    metric_record = {
        "BaselineName": model_name,
        "BaselineFamily": model_family,
    }

    metric_record.update(
        metric_values
    )

    aggregate_combined_metric_records.append(
        metric_record
    )

baseline_aggregate_combined_metrics_df = pd.DataFrame(
    aggregate_combined_metric_records
)


# --------------------------------------------------------------
# 8. Summarise daily wins
# --------------------------------------------------------------

baseline_daily_win_summary_df = (
    baseline_aggregate_daily_predictions_df
    .groupby(
        [
            "ModelName",
            "ModelFamily",
        ],
        as_index=False,
    )
    .agg(
        ExactBestDayCount=(
            "IsDailyBestAbsoluteError",
            "sum",
        ),
        FractionalDailyWins=(
            "FractionalDailyWin",
            "sum",
        ),
        MeanDailyAbsoluteError=(
            "AbsoluteError",
            "mean",
        ),
        MedianDailyAbsoluteError=(
            "AbsoluteError",
            "median",
        ),
        MaximumDailyAbsoluteError=(
            "AbsoluteError",
            "max",
        ),
    )
    .rename(
        columns={
            "ModelName": "BaselineName",
            "ModelFamily": "BaselineFamily",
        }
    )
)


# --------------------------------------------------------------
# 9. Create the aggregate baseline ranking
# --------------------------------------------------------------

product_level_reference_df = (
    baseline_ranking_df[
        [
            "BaselineName",
            "MAE_Rank",
            "MAE",
            "RMSE",
            "WAPE_Percent",
        ]
    ]
    .rename(
        columns={
            "MAE_Rank": "ProductLevelMAE_Rank",
            "MAE": "ProductLevelMAE",
            "RMSE": "ProductLevelRMSE",
            "WAPE_Percent": "ProductLevelWAPE_Percent",
        }
    )
)

baseline_aggregate_ranking_df = (
    baseline_aggregate_combined_metrics_df
    .merge(
        baseline_daily_win_summary_df,
        on=[
            "BaselineName",
            "BaselineFamily",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        product_level_reference_df,
        on="BaselineName",
        how="left",
        validate="one_to_one",
    )
)

baseline_aggregate_ranking_df = (
    baseline_aggregate_ranking_df
    .sort_values(
        [
            "MAE",
            "RMSE",
            "WAPE_Percent",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

baseline_aggregate_ranking_df.insert(
    0,
    "AggregateMAE_Rank",
    np.arange(
        1,
        len(baseline_aggregate_ranking_df) + 1,
    ),
)

baseline_aggregate_ranking_df[
    "AggregateRMSE_Rank"
] = (
    baseline_aggregate_ranking_df["RMSE"]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_aggregate_ranking_df[
    "AggregateWAPE_Rank"
] = (
    baseline_aggregate_ranking_df[
        "WAPE_Percent"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

baseline_aggregate_ranking_df[
    "IsBestAggregateMAE"
] = (
    baseline_aggregate_ranking_df[
        "AggregateMAE_Rank"
    ] == 1
)


# --------------------------------------------------------------
# 10. Create diagnostics for the best aggregate baseline
# --------------------------------------------------------------

best_aggregate_baseline_name = str(
    baseline_aggregate_ranking_df.iloc[0][
        "BaselineName"
    ]
)

best_aggregate_baseline_daily_diagnostics_df = (
    baseline_aggregate_daily_predictions_df.loc[
        baseline_aggregate_daily_predictions_df[
            "ModelName"
        ] == best_aggregate_baseline_name
    ]
    .sort_values(
        [
            "AbsoluteError",
            "Date",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 11. Calculate structural validation values
# --------------------------------------------------------------

expected_aggregate_daily_rows = int(
    len(baseline_names)
    * validation_date_count
)

expected_fold_metric_rows = int(
    len(baseline_names)
    * len(validation_windows)
)

aggregate_duplicate_count = int(
    baseline_aggregate_daily_predictions_df.duplicated(
        subset=[
            "ModelName",
            "WindowID",
            "Date",
        ]
    ).sum()
)

actual_reference_mismatch_count = int(
    (
        ~baseline_aggregate_daily_predictions_df[
            "ActualDemandMatchesReference"
        ]
    ).sum()
)

missing_aggregate_prediction_count = int(
    baseline_aggregate_daily_predictions_df[
        "PredictedDemand"
    ].isna().sum()
)

infinite_aggregate_prediction_count = int(
    np.isinf(
        baseline_aggregate_daily_predictions_df[
            "PredictedDemand"
        ].to_numpy(dtype=float)
    ).sum()
)

negative_aggregate_prediction_count = int(
    (
        baseline_aggregate_daily_predictions_df[
            "PredictedDemand"
        ] < 0
    ).sum()
)

product_row_coverage_mismatch_count = int(
    (
        baseline_aggregate_daily_predictions_df[
            "ProductPredictionRows"
        ]
        != baseline_aggregate_daily_predictions_df[
            "ReferenceProductRows"
        ]
    ).sum()
)

product_count_coverage_mismatch_count = int(
    (
        baseline_aggregate_daily_predictions_df[
            "ProductCount"
        ]
        != baseline_aggregate_daily_predictions_df[
            "ReferenceProductCount"
        ]
    ).sum()
)

split_context_coverage_mismatch_count = int(
    (
        baseline_aggregate_daily_predictions_df[
            "SplitContextCount"
        ]
        != baseline_aggregate_daily_predictions_df[
            "ReferenceSplitContextCount"
        ]
    ).sum()
)

fold_wape_missing_count = int(
    baseline_aggregate_fold_metrics_df[
        "WAPE_Percent"
    ].isna().sum()
)

combined_wape_missing_count = int(
    baseline_aggregate_combined_metrics_df[
        "WAPE_Percent"
    ].isna().sum()
)

expected_actual_total = float(
    aggregate_actual_reference_df[
        "ReferenceActualDemand"
    ].sum()
)

combined_actual_total_mismatch_count = int(
    (
        ~np.isclose(
            baseline_aggregate_combined_metrics_df[
                "ActualDemandTotal"
            ],
            expected_actual_total,
            rtol=1e-12,
            atol=1e-12,
        )
    ).sum()
)

fractional_daily_win_total = float(
    baseline_aggregate_daily_predictions_df[
        "FractionalDailyWin"
    ].sum()
)

missing_product_level_rank_count = int(
    baseline_aggregate_ranking_df[
        "ProductLevelMAE_Rank"
    ].isna().sum()
)


# --------------------------------------------------------------
# 12. Build validation checks
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one validation result.
    """

    if passed is None:
        passed = actual_value == expected_value

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Baseline methods",
    len(baseline_names),
    7,
)

add_check(
    "Validation windows",
    len(validation_windows),
    2,
)

add_check(
    "Validation operating dates",
    validation_date_count,
    40,
)

add_check(
    "Actual daily-reference rows",
    len(aggregate_actual_reference_df),
    validation_date_count,
)

add_check(
    "Aggregate daily prediction rows",
    len(baseline_aggregate_daily_predictions_df),
    expected_aggregate_daily_rows,
)

add_check(
    "Duplicate aggregate predictions",
    aggregate_duplicate_count,
    0,
)

add_check(
    "Actual-demand reference mismatches",
    actual_reference_mismatch_count,
    0,
)

add_check(
    "Missing aggregate predictions",
    missing_aggregate_prediction_count,
    0,
)

add_check(
    "Infinite aggregate predictions",
    infinite_aggregate_prediction_count,
    0,
)

add_check(
    "Negative aggregate predictions",
    negative_aggregate_prediction_count,
    0,
)

add_check(
    "Product-row coverage mismatches",
    product_row_coverage_mismatch_count,
    0,
)

add_check(
    "Product-count coverage mismatches",
    product_count_coverage_mismatch_count,
    0,
)

add_check(
    "Split-context coverage mismatches",
    split_context_coverage_mismatch_count,
    0,
)

add_check(
    "Aggregate fold metric rows",
    len(baseline_aggregate_fold_metrics_df),
    expected_fold_metric_rows,
)

add_check(
    "Aggregate combined metric rows",
    len(baseline_aggregate_combined_metrics_df),
    len(baseline_names),
)

add_check(
    "Aggregate ranking rows",
    len(baseline_aggregate_ranking_df),
    len(baseline_names),
)

add_check(
    "Missing aggregate fold WAPE values",
    fold_wape_missing_count,
    0,
)

add_check(
    "Missing aggregate combined WAPE values",
    combined_wape_missing_count,
    0,
)

add_check(
    "Combined actual-demand total mismatches",
    combined_actual_total_mismatch_count,
    0,
)

add_check(
    "Fractional daily wins",
    fractional_daily_win_total,
    float(validation_date_count),
    passed=np.isclose(
        fractional_daily_win_total,
        float(validation_date_count),
    ),
)

add_check(
    "Best aggregate-MAE baseline count",
    int(
        baseline_aggregate_ranking_df[
            "IsBestAggregateMAE"
        ].sum()
    ),
    1,
)

add_check(
    "Missing product-level rank comparisons",
    missing_product_level_rank_count,
    0,
)

add_check(
    "Best aggregate daily diagnostic rows",
    len(
        best_aggregate_baseline_daily_diagnostics_df
    ),
    validation_date_count,
)

part4_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 13. Stop if validation fails
# --------------------------------------------------------------

failed_checks_df = part4_validation_df.loc[
    ~part4_validation_df["Passed"]
].copy()

if not failed_checks_df.empty:

    print(
        "\nFAILED MODELLING STEP 2 PART 4 CHECKS"
    )

    display(
        failed_checks_df
    )

    raise AssertionError(
        "Modelling Step 2 Part 4 failed. "
        "Do not finalise the baseline benchmark."
    )


# --------------------------------------------------------------
# 14. Save outputs
# --------------------------------------------------------------

aggregate_actual_reference_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_aggregate_actual_daily_reference.csv"
)

aggregate_daily_predictions_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_baseline_aggregate_daily_predictions.csv"
)

aggregate_fold_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_baseline_aggregate_fold_metrics.csv"
)

aggregate_combined_metrics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_baseline_aggregate_combined_metrics.csv"
)

daily_win_summary_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_baseline_aggregate_daily_win_summary.csv"
)

aggregate_ranking_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_baseline_aggregate_ranking.csv"
)

best_aggregate_diagnostics_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_best_aggregate_baseline_daily_diagnostics.csv"
)

part4_validation_path = (
    Path(BASELINE_OUTPUT_DIR)
    / "04_baseline_aggregate_validation_summary.csv"
)

aggregate_actual_reference_df.to_csv(
    aggregate_actual_reference_path,
    index=False,
)

baseline_aggregate_daily_predictions_df.to_csv(
    aggregate_daily_predictions_path,
    index=False,
)

baseline_aggregate_fold_metrics_df.to_csv(
    aggregate_fold_metrics_path,
    index=False,
)

baseline_aggregate_combined_metrics_df.to_csv(
    aggregate_combined_metrics_path,
    index=False,
)

baseline_daily_win_summary_df.to_csv(
    daily_win_summary_path,
    index=False,
)

baseline_aggregate_ranking_df.to_csv(
    aggregate_ranking_path,
    index=False,
)

best_aggregate_baseline_daily_diagnostics_df.to_csv(
    best_aggregate_diagnostics_path,
    index=False,
)

part4_validation_df.to_csv(
    part4_validation_path,
    index=False,
)


# --------------------------------------------------------------
# 15. Prepare final display values
# --------------------------------------------------------------

best_aggregate_row = (
    baseline_aggregate_ranking_df.iloc[0]
)

best_aggregate_mae = float(
    best_aggregate_row["MAE"]
)

best_aggregate_rmse = float(
    best_aggregate_row["RMSE"]
)

best_aggregate_wape = float(
    best_aggregate_row["WAPE_Percent"]
)

best_aggregate_mean_bias = float(
    best_aggregate_row["MeanBias"]
)

product_level_best_baseline = str(
    baseline_ranking_df
    .sort_values("MAE_Rank")
    .iloc[0]["BaselineName"]
)

passed_check_count = int(
    part4_validation_df["Passed"].sum()
)

total_check_count = int(
    len(part4_validation_df)
)


# --------------------------------------------------------------
# 16. Display final results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 2 PART 4: PASSED")
print("=" * 72)

print(
    "\nValidation operating dates: "
    f"{validation_date_count}"
)

print(
    "Aggregate daily prediction rows: "
    f"{len(baseline_aggregate_daily_predictions_df):,}"
)

print(
    "Aggregate fold metric rows: "
    f"{len(baseline_aggregate_fold_metrics_df)}"
)

print(
    "Aggregate combined metric rows: "
    f"{len(baseline_aggregate_combined_metrics_df)}"
)

print(
    "Best product-level baseline: "
    f"{product_level_best_baseline}"
)

print(
    "Best aggregate baseline: "
    f"{best_aggregate_baseline_name}"
)

print(
    "Best aggregate MAE: "
    f"{best_aggregate_mae:.4f} units per operating day"
)

print(
    "Best aggregate RMSE: "
    f"{best_aggregate_rmse:.4f} units per operating day"
)

print(
    "Best aggregate WAPE: "
    f"{best_aggregate_wape:.2f}%"
)

print(
    "Best aggregate mean bias: "
    f"{best_aggregate_mean_bias:.4f} units per operating day"
)

print(
    "\nValidation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print("\nAggregate baseline ranking:")
display(
    baseline_aggregate_ranking_df
)

print("\nAggregate fold metrics:")
display(
    baseline_aggregate_fold_metrics_df
)

print("\nDaily-win summary:")
display(
    baseline_daily_win_summary_df
)

print("\nBest aggregate baseline daily diagnostics:")
display(
    best_aggregate_baseline_daily_diagnostics_df.head(15)
)

print("\nPart 4 validation summary:")
display(
    part4_validation_df
)

print("\nSaved outputs:")
print(aggregate_actual_reference_path)
print(aggregate_daily_predictions_path)
print(aggregate_fold_metrics_path)
print(aggregate_combined_metrics_path)
print(daily_win_summary_path)
print(aggregate_ranking_path)
print(best_aggregate_diagnostics_path)
print(part4_validation_path)


MODELLING STEP 2 PART 4: PASSED

Validation operating dates: 40
Aggregate daily prediction rows: 280
Aggregate fold metric rows: 14
Aggregate combined metric rows: 7
Best product-level baseline: MOVING_AVERAGE_5
Best aggregate baseline: NAIVE_5_OPERATING_DAYS
Best aggregate MAE: 42.5000 units per operating day
Best aggregate RMSE: 54.3388 units per operating day
Best aggregate WAPE: 20.94%
Best aggregate mean bias: -20.9500 units per operating day

Validation checks passed: 23 of 23

Aggregate baseline ranking:


,AggregateMAE_Rank,BaselineName,BaselineFamily,ObservationCount,ActualDemandTotal,PredictedDemandTotal,MAE,RMSE,WAPE_Percent,MeanBias,...,MeanDailyAbsoluteError,MedianDailyAbsoluteError,MaximumDailyAbsoluteError,ProductLevelMAE_Rank,ProductLevelMAE,ProductLevelRMSE,ProductLevelWAPE_Percent,AggregateRMSE_Rank,AggregateWAPE_Rank,IsBestAggregateMAE
0,1,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,40,8117.0,7279.000000,42.500000,54.338752,20.943698,-20.950000,...,42.500000,35.000000,143.000000,3,0.680315,2.570885,42.577307,1,1,True
1,2,MOVING_AVERAGE_10,MOVING_AVERAGE,40,8117.0,7338.800000,53.480000,65.719316,26.354564,-19.455000,...,53.480000,49.200000,128.700000,2,0.647087,2.449349,40.497721,2,2,False
2,3,MOVING_AVERAGE_5,MOVING_AVERAGE,40,8117.0,7585.200000,54.105000,65.894150,26.662560,-13.295000,...,54.105000,48.100000,122.200000,1,0.643425,2.439141,40.268572,3,3,False
3,4,MOVING_AVERAGE_20,MOVING_AVERAGE,40,8117.0,6968.050000,59.761250,72.023130,29.449920,-28.723750,...,59.761250,50.875000,154.500000,4,0.686526,2.555544,42.965997,4,4,False
4,5,LAST_OBSERVED_DEMAND,NAIVE,40,8117.0,7983.000000,66.550000,84.241914,32.795368,-3.350000,...,66.550000,48.500000,201.000000,5,0.793701,2.953311,49.673525,5,5,False
5,6,HISTORICAL_MEAN,HISTORICAL_AVERAGE,40,8117.0,11788.510115,97.699077,119.008279,48.145412,91.787753,...,97.699077,106.973839,227.800245,7,2.189040,4.974717,137.000420,6,6,False
6,7,ZERO_DEMAND,CONSTANT,40,8117.0,0.000000,202.925000,215.514211,100.000000,-202.925000,...,202.925000,196.000000,349.000000,6,1.597835,5.736178,100.000000,7,7,False



Aggregate fold metrics:


,BaselineName,BaselineFamily,WindowID,ObservationCount,ActualDemandTotal,PredictedDemandTotal,MAE,RMSE,WAPE_Percent,MeanBias,TotalBias,OverforecastUnits,UnderforecastUnits,ZeroActualObservationCount,AggregateMAERankWithinFold
0,MOVING_AVERAGE_10,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_1,20,3309.0,2720.100000,49.265000,61.575486,29.776367,-29.445000,-588.900000,198.200000,787.100000,0,1
1,MOVING_AVERAGE_20,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_1,20,3309.0,2824.650000,50.242500,63.641134,30.367180,-24.217500,-484.350000,260.250000,744.600000,0,2
2,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,20,3309.0,2665.000000,50.700000,60.067462,30.643699,-32.200000,-644.000000,185.000000,829.000000,0,3
3,MOVING_AVERAGE_5,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_1,20,3309.0,2791.600000,52.310000,64.157540,31.616803,-25.870000,-517.400000,264.400000,781.800000,0,4
4,LAST_OBSERVED_DEMAND,NAIVE,STANDARD_BACKTEST_FOLD_1,20,3309.0,3128.000000,52.950000,70.085305,32.003626,-9.050000,-181.000000,439.000000,620.000000,0,5
5,HISTORICAL_MEAN,HISTORICAL_AVERAGE,STANDARD_BACKTEST_FOLD_1,20,3309.0,5982.643082,133.682154,145.868428,80.799126,133.682154,2673.643082,2673.643082,0.000000,0,6
6,ZERO_DEMAND,CONSTANT,STANDARD_BACKTEST_FOLD_1,20,3309.0,0.000000,165.450000,174.838640,100.000000,-165.450000,-3309.000000,0.000000,3309.000000,0,7
7,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_2,20,4808.0,4614.000000,34.300000,47.930158,14.267887,-9.700000,-194.000000,246.000000,440.000000,0,1
8,MOVING_AVERAGE_5,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_2,20,4808.0,4793.600000,55.900000,67.586152,23.252912,-0.720000,-14.400000,551.800000,566.200000,0,2
9,MOVING_AVERAGE_10,MOVING_AVERAGE,STANDARD_BACKTEST_FOLD_2,20,4808.0,4618.700000,57.695000,69.616927,23.999584,-9.465000,-189.300000,482.300000,671.600000,0,3



Daily-win summary:


...


Best aggregate baseline daily diagnostics:


,ModelName,ModelFamily,WindowID,Date,ActualDemand,PredictedDemand,ProductPredictionRows,ProductCount,SplitContextCount,ClippedProductPredictions,...,ReferenceProductCount,ReferenceSplitContextCount,ActualDemandMatchesReference,ForecastError,AbsoluteError,SquaredError,BestAbsoluteErrorForDate,IsDailyBestAbsoluteError,DailyBestTieCount,FractionalDailyWin
0,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_2,2026-02-09,264,121.0,127,127,127,0,...,127,127,True,-143.0,143.0,20449.0,26.868166,False,1,0.0
1,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2026-01-26,268,152.0,127,127,127,0,...,127,127,True,-116.0,116.0,13456.0,25.656464,False,1,0.0
2,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2026-01-09,104,3.0,127,127,127,0,...,127,127,True,-101.0,101.0,10201.0,14.300000,False,1,0.0
3,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2026-01-08,122,28.0,127,127,127,0,...,127,127,True,-94.0,94.0,8836.0,0.900000,False,1,0.0
4,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2026-01-07,164,74.0,127,127,127,0,...,127,127,True,-90.0,90.0,8100.0,16.350000,False,1,0.0
5,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2026-01-23,67,143.0,127,127,127,0,...,127,127,True,76.0,76.0,5776.0,43.000000,False,1,0.0
6,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_2,2026-02-11,349,274.0,127,127,127,0,...,127,127,True,-75.0,75.0,5625.0,58.548129,False,1,0.0
7,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2026-01-29,184,110.0,127,127,127,0,...,127,127,True,-74.0,74.0,5476.0,8.500000,False,1,0.0
8,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_1,2026-01-28,257,188.0,127,127,127,0,...,127,127,True,-69.0,69.0,4761.0,17.000000,False,1,0.0
9,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,STANDARD_BACKTEST_FOLD_2,2026-02-18,282,349.0,127,127,127,0,...,127,127,True,67.0,67.0,4489.0,7.781326,False,1,0.0



Part 4 validation summary:


,Check,Expected,Actual,Passed
0,Baseline methods,7,7,True
1,Validation windows,2,2,True
2,Validation operating dates,40,40,True
3,Actual daily-reference rows,40,40,True
4,Aggregate daily prediction rows,280,280,True
5,Duplicate aggregate predictions,0,0,True
6,Actual-demand reference mismatches,0,0,True
7,Missing aggregate predictions,0,0,True
8,Infinite aggregate predictions,0,0,True
9,Negative aggregate predictions,0,0,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_aggregate_actual_daily_reference.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_baseline_aggregate_daily_predictions.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_baseline_aggregate_fold_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_baseline_aggregate_combined_metrics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_baseline_aggregate_daily_win_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_baseline_aggregate_ranking.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_best_aggregate_baseline_daily_diagnostics.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/04_baseline_aggregate_validation_summary.csv


In [20]:
# ==============================================================
# MODELLING STEP 2, PART 5
# Lock and validate the baseline benchmark
# ==============================================================

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------
# 1. Confirm required objects from Step 2 Parts 1 to 4
# --------------------------------------------------------------

required_objects = [
    "validation_df",
    "baseline_method_register_df",
    "baseline_validation_predictions_df",
    "baseline_fold_metrics_df",
    "baseline_ranking_df",
    "baseline_product_window_metrics_df",
    "baseline_context_summary_df",
    "baseline_demand_pattern_metrics_df",
    "baseline_product_combined_metrics_df",
    "baseline_product_summary_df",
    "baseline_aggregate_daily_predictions_df",
    "baseline_aggregate_fold_metrics_df",
    "baseline_aggregate_ranking_df",
    "BASELINE_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "The following required Step 2 objects are unavailable:\n"
        f"{missing_objects}\n\n"
        "Run Modelling Step 2 Parts 1 to 4 first."
    )


# --------------------------------------------------------------
# 2. Confirm the baseline directory
# --------------------------------------------------------------

BASELINE_OUTPUT_DIR = Path(
    BASELINE_OUTPUT_DIR
)

if not BASELINE_OUTPUT_DIR.exists():
    raise FileNotFoundError(
        "The baseline output directory does not exist:\n"
        f"{BASELINE_OUTPUT_DIR}"
    )


# --------------------------------------------------------------
# 3. Define the expected outputs from Parts 1 to 4
# --------------------------------------------------------------

expected_baseline_output_files = [
    # Part 1
    "01_baseline_method_register.csv",
    "01_baseline_source_feature_audit.csv",
    "01_baseline_prediction_generation_audit.csv",
    "01_lag5_calendar_alignment_summary.csv",
    "01_lag5_validation_reconstruction_audit.csv",
    "01_baseline_framework_validation_summary.csv",

    # Part 2
    "02_baseline_validation_predictions.csv",
    "02_baseline_fold_metrics.csv",
    "02_baseline_combined_metrics.csv",
    "02_baseline_fold_stability_summary.csv",
    "02_baseline_overall_ranking.csv",
    "02_baseline_prediction_coverage_audit.csv",
    "02_baseline_performance_validation_summary.csv",

    # Part 3
    "03_baseline_product_window_metrics.csv",
    "03_baseline_context_summary.csv",
    "03_baseline_demand_pattern_metrics.csv",
    "03_baseline_window_status_metrics.csv",
    "03_baseline_product_combined_metrics.csv",
    "03_baseline_product_summary.csv",
    "03_best_baseline_product_diagnostics.csv",
    "03_baseline_diagnostic_validation_summary.csv",

    # Part 4
    "04_aggregate_actual_daily_reference.csv",
    "04_baseline_aggregate_daily_predictions.csv",
    "04_baseline_aggregate_fold_metrics.csv",
    "04_baseline_aggregate_combined_metrics.csv",
    "04_baseline_aggregate_daily_win_summary.csv",
    "04_baseline_aggregate_ranking.csv",
    "04_best_aggregate_baseline_daily_diagnostics.csv",
    "04_baseline_aggregate_validation_summary.csv",
]


# --------------------------------------------------------------
# 4. Audit the saved output files
# --------------------------------------------------------------

output_file_audit_records = []

for file_name in expected_baseline_output_files:

    file_path = (
        BASELINE_OUTPUT_DIR
        / file_name
    )

    file_exists = file_path.exists()

    file_size_bytes = (
        file_path.stat().st_size
        if file_exists
        else 0
    )

    output_file_audit_records.append(
        {
            "FileName": file_name,
            "FilePath": str(file_path),
            "Exists": file_exists,
            "FileSizeBytes": file_size_bytes,
            "NonEmpty": bool(
                file_exists
                and file_size_bytes > 0
            ),
        }
    )

baseline_output_file_audit_df = pd.DataFrame(
    output_file_audit_records
)

missing_output_file_count = int(
    (
        ~baseline_output_file_audit_df[
            "Exists"
        ]
    ).sum()
)

empty_output_file_count = int(
    (
        baseline_output_file_audit_df["Exists"]
        & ~baseline_output_file_audit_df["NonEmpty"]
    ).sum()
)


# --------------------------------------------------------------
# 5. Define a robust saved-boolean parser
# --------------------------------------------------------------

def parse_saved_boolean_series(series):
    """
    Convert common CSV boolean representations into booleans.
    """

    converted_series = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
    )

    if converted_series.isna().any():
        invalid_values = (
            series.loc[
                converted_series.isna()
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        raise ValueError(
            "Unexpected saved boolean values:\n"
            f"{invalid_values}"
        )

    return converted_series.astype(bool)


# --------------------------------------------------------------
# 6. Load and validate each saved Part summary
# --------------------------------------------------------------

part_validation_files = {
    "PART_1":
        "01_baseline_framework_validation_summary.csv",

    "PART_2":
        "02_baseline_performance_validation_summary.csv",

    "PART_3":
        "03_baseline_diagnostic_validation_summary.csv",

    "PART_4":
        "04_baseline_aggregate_validation_summary.csv",
}

part_validation_records = []

for part_name, file_name in part_validation_files.items():

    file_path = (
        BASELINE_OUTPUT_DIR
        / file_name
    )

    if not file_path.exists():

        part_validation_records.append(
            {
                "Part": part_name,
                "ValidationFile": file_name,
                "CheckCount": 0,
                "PassedCheckCount": 0,
                "FailedCheckCount": 1,
                "PartPassed": False,
            }
        )

        continue

    part_summary_df = pd.read_csv(
        file_path
    )

    if "Passed" not in part_summary_df.columns:
        raise KeyError(
            f"'Passed' column missing from {file_name}."
        )

    passed_values = parse_saved_boolean_series(
        part_summary_df["Passed"]
    )

    check_count = int(
        len(passed_values)
    )

    passed_check_count = int(
        passed_values.sum()
    )

    failed_check_count = int(
        check_count - passed_check_count
    )

    part_validation_records.append(
        {
            "Part": part_name,
            "ValidationFile": file_name,
            "CheckCount": check_count,
            "PassedCheckCount": passed_check_count,
            "FailedCheckCount": failed_check_count,
            "PartPassed": bool(
                check_count > 0
                and failed_check_count == 0
            ),
        }
    )

baseline_part_validation_register_df = pd.DataFrame(
    part_validation_records
)


# --------------------------------------------------------------
# 7. Select the product-level benchmark
# --------------------------------------------------------------

product_day_ranking_df = (
    baseline_ranking_df
    .sort_values(
        [
            "MAE_Rank",
            "RMSE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

product_day_winner = (
    product_day_ranking_df.iloc[0]
)

product_day_baseline_name = str(
    product_day_winner[
        "BaselineName"
    ]
)

product_day_baseline_family = str(
    product_day_winner[
        "BaselineFamily"
    ]
)

product_day_mae = float(
    product_day_winner["MAE"]
)

product_day_rmse = float(
    product_day_winner["RMSE"]
)

product_day_wape = float(
    product_day_winner[
        "WAPE_Percent"
    ]
)

product_day_mean_bias = float(
    product_day_winner[
        "MeanBias"
    ]
)

product_day_total_bias = float(
    product_day_winner[
        "TotalBias"
    ]
)

product_day_observation_count = int(
    product_day_winner[
        "ObservationCount"
    ]
)


# --------------------------------------------------------------
# 8. Select the context-level benchmark
# --------------------------------------------------------------

context_ranking_df = (
    baseline_context_summary_df
    .sort_values(
        [
            "MeanContextMAE_Rank",
            "MeanPositiveDemandContextMAE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

context_winner = context_ranking_df.iloc[0]

context_baseline_name = str(
    context_winner[
        "BaselineName"
    ]
)

context_mean_mae = float(
    context_winner[
        "MeanContextMAE"
    ]
)

context_positive_mean_mae = float(
    context_winner[
        "MeanPositiveDemandContextMAE"
    ]
)


# --------------------------------------------------------------
# 9. Select the combined product-level benchmark
# --------------------------------------------------------------

product_ranking_df = (
    baseline_product_summary_df
    .sort_values(
        [
            "MeanProductMAE_Rank",
            "MeanPositiveProductMAE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

product_winner = product_ranking_df.iloc[0]

product_baseline_name = str(
    product_winner[
        "BaselineName"
    ]
)

product_mean_mae = float(
    product_winner[
        "MeanProductMAE"
    ]
)

positive_product_mean_mae = float(
    product_winner[
        "MeanPositiveProductMAE"
    ]
)


# --------------------------------------------------------------
# 10. Select the aggregate daily-demand benchmark
# --------------------------------------------------------------

aggregate_ranking_df = (
    baseline_aggregate_ranking_df
    .sort_values(
        [
            "AggregateMAE_Rank",
            "RMSE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

aggregate_winner = aggregate_ranking_df.iloc[0]

aggregate_baseline_name = str(
    aggregate_winner[
        "BaselineName"
    ]
)

aggregate_baseline_family = str(
    aggregate_winner[
        "BaselineFamily"
    ]
)

aggregate_mae = float(
    aggregate_winner["MAE"]
)

aggregate_rmse = float(
    aggregate_winner["RMSE"]
)

aggregate_wape = float(
    aggregate_winner[
        "WAPE_Percent"
    ]
)

aggregate_mean_bias = float(
    aggregate_winner[
        "MeanBias"
    ]
)

aggregate_total_bias = float(
    aggregate_winner[
        "TotalBias"
    ]
)

aggregate_observation_count = int(
    aggregate_winner[
        "ObservationCount"
    ]
)


# --------------------------------------------------------------
# 11. Create the baseline-selection evidence table
# --------------------------------------------------------------

baseline_selection_evidence_df = pd.DataFrame(
    [
        {
            "EvaluationLevel":
                "PRODUCT_DAY_POOLED",
            "SelectionCriterion":
                "LOWEST_COMBINED_MAE",
            "SelectedBaseline":
                product_day_baseline_name,
            "SelectedMetricValue":
                product_day_mae,
            "SupportingMetric":
                "RMSE",
            "SupportingMetricValue":
                product_day_rmse,
        },
        {
            "EvaluationLevel":
                "PRODUCT_WINDOW_CONTEXT",
            "SelectionCriterion":
                "LOWEST_MEAN_CONTEXT_MAE",
            "SelectedBaseline":
                context_baseline_name,
            "SelectedMetricValue":
                context_mean_mae,
            "SupportingMetric":
                "MEAN_POSITIVE_CONTEXT_MAE",
            "SupportingMetricValue":
                context_positive_mean_mae,
        },
        {
            "EvaluationLevel":
                "PRODUCT_COMBINED_TWO_FOLDS",
            "SelectionCriterion":
                "LOWEST_MEAN_PRODUCT_MAE",
            "SelectedBaseline":
                product_baseline_name,
            "SelectedMetricValue":
                product_mean_mae,
            "SupportingMetric":
                "MEAN_POSITIVE_PRODUCT_MAE",
            "SupportingMetricValue":
                positive_product_mean_mae,
        },
        {
            "EvaluationLevel":
                "AGGREGATE_DAILY_BOTTOM_UP",
            "SelectionCriterion":
                "LOWEST_COMBINED_DAILY_MAE",
            "SelectedBaseline":
                aggregate_baseline_name,
            "SelectedMetricValue":
                aggregate_mae,
            "SupportingMetric":
                "RMSE",
            "SupportingMetricValue":
                aggregate_rmse,
        },
    ]
)


# --------------------------------------------------------------
# 12. Create the locked benchmark contract
# --------------------------------------------------------------

baseline_benchmark_contract_df = pd.DataFrame(
    [
        {
            "ForecastLevel":
                "PRODUCT_DAY",
            "BenchmarkStatus":
                "LOCKED_FOR_MODEL_SELECTION",
            "SelectedBaseline":
                product_day_baseline_name,
            "BaselineFamily":
                product_day_baseline_family,
            "PrimarySelectionMetric":
                "MAE",
            "MAE":
                product_day_mae,
            "RMSE":
                product_day_rmse,
            "WAPE_Percent":
                product_day_wape,
            "MeanBias":
                product_day_mean_bias,
            "TotalBias":
                product_day_total_bias,
            "EvaluationObservationCount":
                product_day_observation_count,
            "ComparisonRule":
                "ADVANCED_MODEL_MUST_TARGET_LOWER_MAE",
            "WAPEInterpretation":
                "ERROR_PERCENTAGE_NOT_ACCURACY",
        },
        {
            "ForecastLevel":
                "AGGREGATE_DAILY_BOTTOM_UP",
            "BenchmarkStatus":
                "LOCKED_FOR_MODEL_SELECTION",
            "SelectedBaseline":
                aggregate_baseline_name,
            "BaselineFamily":
                aggregate_baseline_family,
            "PrimarySelectionMetric":
                "MAE",
            "MAE":
                aggregate_mae,
            "RMSE":
                aggregate_rmse,
            "WAPE_Percent":
                aggregate_wape,
            "MeanBias":
                aggregate_mean_bias,
            "TotalBias":
                aggregate_total_bias,
            "EvaluationObservationCount":
                aggregate_observation_count,
            "ComparisonRule":
                "ADVANCED_MODEL_MUST_TARGET_LOWER_MAE",
            "WAPEInterpretation":
                "ERROR_PERCENTAGE_NOT_ACCURACY",
        },
    ]
)


# --------------------------------------------------------------
# 13. Preserve demand-pattern winners as diagnostics
# --------------------------------------------------------------

demand_pattern_winners_df = (
    baseline_demand_pattern_metrics_df.loc[
        baseline_demand_pattern_metrics_df[
            "MAERankWithinDemandPattern"
        ] == 1
    ]
    .copy()
    .sort_values(
        [
            "DemandPatternClass",
            "MAE",
            "BaselineName",
        ]
    )
    .reset_index(drop=True)
)

demand_pattern_class_count = int(
    baseline_demand_pattern_metrics_df[
        "DemandPatternClass"
    ].nunique()
)

demand_pattern_winner_coverage_count = int(
    demand_pattern_winners_df[
        "DemandPatternClass"
    ].nunique()
)


# --------------------------------------------------------------
# 14. Calculate structural summary values
# --------------------------------------------------------------

failed_part_count = int(
    (
        ~baseline_part_validation_register_df[
            "PartPassed"
        ]
    ).sum()
)

baseline_method_count = int(
    baseline_method_register_df[
        "BaselineName"
    ].nunique()
)

validation_row_count = int(
    len(validation_df)
)

prediction_row_count = int(
    len(baseline_validation_predictions_df)
)

validation_window_count = int(
    validation_df[
        "WindowID"
    ].nunique()
)

validation_product_count = int(
    validation_df[
        "CanonicalProductID"
    ].nunique()
)

product_window_context_count = int(
    validation_df[
        "SplitContextID"
    ].nunique()
)

all_zero_context_count = int(
    baseline_product_window_metrics_df.loc[
        baseline_product_window_metrics_df[
            "BaselineName"
        ] == product_day_baseline_name,
        "AllZeroActualWindow",
    ].sum()
)

positive_context_count = int(
    product_window_context_count
    - all_zero_context_count
)

product_window_metric_row_count = int(
    len(baseline_product_window_metrics_df)
)

combined_product_metric_row_count = int(
    len(baseline_product_combined_metrics_df)
)

validation_operating_date_count = int(
    validation_df[
        [
            "WindowID",
            "Date",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

aggregate_prediction_row_count = int(
    len(
        baseline_aggregate_daily_predictions_df
    )
)

product_benchmark_metrics_finite = bool(
    np.isfinite(
        [
            product_day_mae,
            product_day_rmse,
            product_day_wape,
            product_day_mean_bias,
            product_day_total_bias,
        ]
    ).all()
)

aggregate_benchmark_metrics_finite = bool(
    np.isfinite(
        [
            aggregate_mae,
            aggregate_rmse,
            aggregate_wape,
            aggregate_mean_bias,
            aggregate_total_bias,
        ]
    ).all()
)

product_level_winner_consistency = bool(
    product_day_baseline_name
    == context_baseline_name
    == product_baseline_name
)


# --------------------------------------------------------------
# 15. Build the final Step 2 validation summary
# --------------------------------------------------------------

validation_checks = []


def add_check(
    check_name,
    actual_value,
    expected_value,
    passed=None,
):
    """
    Add one Step 2 completion check.
    """

    if passed is None:
        passed = (
            actual_value
            == expected_value
        )

    validation_checks.append(
        {
            "Check": check_name,
            "Expected": str(expected_value),
            "Actual": str(actual_value),
            "Passed": bool(passed),
        }
    )


add_check(
    "Completed Step 2 parts",
    len(
        baseline_part_validation_register_df
    ),
    4,
)

add_check(
    "Failed Step 2 parts",
    failed_part_count,
    0,
)

add_check(
    "Expected saved baseline files",
    len(expected_baseline_output_files),
    29,
)

add_check(
    "Missing saved baseline files",
    missing_output_file_count,
    0,
)

add_check(
    "Empty saved baseline files",
    empty_output_file_count,
    0,
)

add_check(
    "Registered baseline methods",
    baseline_method_count,
    7,
)

add_check(
    "Model-selection validation rows",
    validation_row_count,
    5_080,
)

add_check(
    "Baseline prediction rows",
    prediction_row_count,
    35_560,
)

add_check(
    "Validation windows",
    validation_window_count,
    2,
)

add_check(
    "Validation products",
    validation_product_count,
    127,
)

add_check(
    "Product-window contexts",
    product_window_context_count,
    254,
)

add_check(
    "All-zero product-window contexts",
    all_zero_context_count,
    210,
)

add_check(
    "Positive-demand product-window contexts",
    positive_context_count,
    44,
)

add_check(
    "Product-window metric rows",
    product_window_metric_row_count,
    1_778,
)

add_check(
    "Combined product metric rows",
    combined_product_metric_row_count,
    889,
)

add_check(
    "Validation operating dates",
    validation_operating_date_count,
    40,
)

add_check(
    "Aggregate prediction rows",
    aggregate_prediction_row_count,
    280,
)

add_check(
    "Product-day benchmark",
    product_day_baseline_name,
    "MOVING_AVERAGE_5",
)

add_check(
    "Context-level benchmark",
    context_baseline_name,
    "MOVING_AVERAGE_5",
)

add_check(
    "Combined product benchmark",
    product_baseline_name,
    "MOVING_AVERAGE_5",
)

add_check(
    "Consistent product-level benchmark",
    product_level_winner_consistency,
    True,
)

add_check(
    "Aggregate daily benchmark",
    aggregate_baseline_name,
    "NAIVE_5_OPERATING_DAYS",
)

add_check(
    "Finite product benchmark metrics",
    product_benchmark_metrics_finite,
    True,
)

add_check(
    "Finite aggregate benchmark metrics",
    aggregate_benchmark_metrics_finite,
    True,
)

add_check(
    "Demand-pattern classes",
    demand_pattern_class_count,
    4,
)

add_check(
    "Demand-pattern winner coverage",
    demand_pattern_winner_coverage_count,
    demand_pattern_class_count,
)

step2_final_validation_df = pd.DataFrame(
    validation_checks
)


# --------------------------------------------------------------
# 16. Stop if any final validation check fails
# --------------------------------------------------------------

failed_checks_df = (
    step2_final_validation_df.loc[
        ~step2_final_validation_df[
            "Passed"
        ]
    ]
    .copy()
)

if not failed_checks_df.empty:

    print(
        "\nFAILED MODELLING STEP 2 COMPLETION CHECKS"
    )

    display(
        failed_checks_df
    )

    raise AssertionError(
        "Modelling Step 2 is not complete. "
        "Do not begin advanced model selection."
    )


# --------------------------------------------------------------
# 17. Create the Step 2 completion summary
# --------------------------------------------------------------

step2_completion_summary_df = pd.DataFrame(
    [
        {
            "Phase":
                "MODELLING_STEP_2",
            "Status":
                "COMPLETED_AND_VALIDATED",
            "BaselineMethods":
                baseline_method_count,
            "ValidationRows":
                validation_row_count,
            "ValidationWindows":
                validation_window_count,
            "ValidationProducts":
                validation_product_count,
            "ProductWindowContexts":
                product_window_context_count,
            "ValidationOperatingDates":
                validation_operating_date_count,
            "ProductLevelBenchmark":
                product_day_baseline_name,
            "ProductLevelBenchmarkMAE":
                product_day_mae,
            "ProductLevelBenchmarkRMSE":
                product_day_rmse,
            "ProductLevelBenchmarkWAPE_Percent":
                product_day_wape,
            "AggregateBenchmark":
                aggregate_baseline_name,
            "AggregateBenchmarkMAE":
                aggregate_mae,
            "AggregateBenchmarkRMSE":
                aggregate_rmse,
            "AggregateBenchmarkWAPE_Percent":
                aggregate_wape,
            "AggregateBenchmarkMeanBias":
                aggregate_mean_bias,
            "FinalTestAccessedInBaselinePhase":
                False,
            "NextPhase":
                "MODELLING_STEP_3_ADVANCED_MODEL_SELECTION",
        }
    ]
)


# --------------------------------------------------------------
# 18. Save the final Step 2 outputs
# --------------------------------------------------------------

output_file_audit_path = (
    BASELINE_OUTPUT_DIR
    / "05_baseline_output_file_audit.csv"
)

part_validation_register_path = (
    BASELINE_OUTPUT_DIR
    / "05_baseline_part_validation_register.csv"
)

selection_evidence_path = (
    BASELINE_OUTPUT_DIR
    / "05_baseline_selection_evidence.csv"
)

benchmark_contract_path = (
    BASELINE_OUTPUT_DIR
    / "05_locked_baseline_benchmark_contract.csv"
)

demand_pattern_winners_path = (
    BASELINE_OUTPUT_DIR
    / "05_demand_pattern_baseline_winners.csv"
)

step2_final_validation_path = (
    BASELINE_OUTPUT_DIR
    / "05_modelling_step2_final_validation_summary.csv"
)

step2_completion_summary_path = (
    BASELINE_OUTPUT_DIR
    / "05_modelling_step2_completion_summary.csv"
)

step2_handoff_path = (
    BASELINE_OUTPUT_DIR
    / "MODELLING_STEP2_HANDOFF.md"
)

baseline_output_file_audit_df.to_csv(
    output_file_audit_path,
    index=False,
)

baseline_part_validation_register_df.to_csv(
    part_validation_register_path,
    index=False,
)

baseline_selection_evidence_df.to_csv(
    selection_evidence_path,
    index=False,
)

baseline_benchmark_contract_df.to_csv(
    benchmark_contract_path,
    index=False,
)

demand_pattern_winners_df.to_csv(
    demand_pattern_winners_path,
    index=False,
)

step2_final_validation_df.to_csv(
    step2_final_validation_path,
    index=False,
)

step2_completion_summary_df.to_csv(
    step2_completion_summary_path,
    index=False,
)


# --------------------------------------------------------------
# 19. Create the Step 2 handover document
# --------------------------------------------------------------

handoff_text = f"""
# Modelling Step 2 Handoff

## Status

Completed and validated.

## Model-selection coverage

- Baseline methods: {baseline_method_count}
- Validation rows: {validation_row_count:,}
- Validation windows: {validation_window_count}
- Validation products: {validation_product_count}
- Product-window contexts: {product_window_context_count}
- Validation operating dates: {validation_operating_date_count}

## Locked product-level benchmark

- Baseline: {product_day_baseline_name}
- MAE: {product_day_mae:.6f} units per product-day
- RMSE: {product_day_rmse:.6f} units per product-day
- WAPE: {product_day_wape:.6f}%
- Mean bias: {product_day_mean_bias:.6f} units per product-day

The same method ranked first for pooled product-day MAE,
mean product-window MAE and mean combined product MAE.

## Locked aggregate daily-demand benchmark

- Baseline: {aggregate_baseline_name}
- MAE: {aggregate_mae:.6f} units per operating day
- RMSE: {aggregate_rmse:.6f} units per operating day
- WAPE: {aggregate_wape:.6f}%
- Mean bias: {aggregate_mean_bias:.6f} units per operating day

A negative bias indicates aggregate underforecasting.

## Interpretation rule

WAPE is an error percentage and must not be reported as
forecast accuracy.

## Final-test protection

No reserved final-test data was accessed during baseline
development or baseline evaluation.

## Next phase

Modelling Step 3: advanced model selection using the same two
chronological validation folds.

Advanced product-level models will be compared primarily against
the locked {product_day_baseline_name} product-day MAE benchmark.

Bottom-up aggregate forecasts will also be compared against the
locked {aggregate_baseline_name} aggregate daily benchmark.
""".strip()

step2_handoff_path.write_text(
    handoff_text,
    encoding="utf-8",
)


# --------------------------------------------------------------
# 20. Prepare final display values
# --------------------------------------------------------------

passed_part_count = int(
    baseline_part_validation_register_df[
        "PartPassed"
    ].sum()
)

total_part_count = int(
    len(
        baseline_part_validation_register_df
    )
)

passed_check_count = int(
    step2_final_validation_df[
        "Passed"
    ].sum()
)

total_check_count = int(
    len(
        step2_final_validation_df
    )
)


# --------------------------------------------------------------
# 21. Display completion results
# --------------------------------------------------------------

print("\n" + "=" * 72)
print("MODELLING STEP 2: COMPLETED AND VALIDATED")
print("=" * 72)

print(
    "\nStep 2 parts passed: "
    f"{passed_part_count} of {total_part_count}"
)

print(
    "Final validation checks passed: "
    f"{passed_check_count} of {total_check_count}"
)

print(
    "Baseline methods evaluated: "
    f"{baseline_method_count}"
)

print(
    "Product-level benchmark: "
    f"{product_day_baseline_name}"
)

print(
    "Product-level benchmark MAE: "
    f"{product_day_mae:.4f} units"
)

print(
    "Product-level benchmark RMSE: "
    f"{product_day_rmse:.4f} units"
)

print(
    "Product-level benchmark WAPE: "
    f"{product_day_wape:.2f}%"
)

print(
    "Aggregate daily benchmark: "
    f"{aggregate_baseline_name}"
)

print(
    "Aggregate benchmark MAE: "
    f"{aggregate_mae:.4f} units per operating day"
)

print(
    "Aggregate benchmark RMSE: "
    f"{aggregate_rmse:.4f} units per operating day"
)

print(
    "Aggregate benchmark WAPE: "
    f"{aggregate_wape:.2f}%"
)

print(
    "Aggregate benchmark mean bias: "
    f"{aggregate_mean_bias:.4f} units per operating day"
)

print(
    "\nNext phase: "
    "MODELLING STEP 3 — ADVANCED MODEL SELECTION"
)

print("\nLocked benchmark contract:")
display(
    baseline_benchmark_contract_df
)

print("\nBaseline selection evidence:")
display(
    baseline_selection_evidence_df
)

print("\nDemand-pattern baseline winners:")
display(
    demand_pattern_winners_df[
        [
            "DemandPatternClass",
            "BaselineName",
            "MAE",
            "RMSE",
            "WAPE_Percent",
            "MeanBias",
        ]
    ]
)

print("\nStep 2 part-validation register:")
display(
    baseline_part_validation_register_df
)

print("\nFinal Step 2 validation summary:")
display(
    step2_final_validation_df
)

print("\nSaved outputs:")
print(output_file_audit_path)
print(part_validation_register_path)
print(selection_evidence_path)
print(benchmark_contract_path)
print(demand_pattern_winners_path)
print(step2_final_validation_path)
print(step2_completion_summary_path)
print(step2_handoff_path)


MODELLING STEP 2: COMPLETED AND VALIDATED

Step 2 parts passed: 4 of 4
Final validation checks passed: 26 of 26
Baseline methods evaluated: 7
Product-level benchmark: MOVING_AVERAGE_5
Product-level benchmark MAE: 0.6434 units
Product-level benchmark RMSE: 2.4391 units
Product-level benchmark WAPE: 40.27%
Aggregate daily benchmark: NAIVE_5_OPERATING_DAYS
Aggregate benchmark MAE: 42.5000 units per operating day
Aggregate benchmark RMSE: 54.3388 units per operating day
Aggregate benchmark WAPE: 20.94%
Aggregate benchmark mean bias: -20.9500 units per operating day

Next phase: MODELLING STEP 3 — ADVANCED MODEL SELECTION

Locked benchmark contract:


,ForecastLevel,BenchmarkStatus,SelectedBaseline,BaselineFamily,PrimarySelectionMetric,MAE,RMSE,WAPE_Percent,MeanBias,TotalBias,EvaluationObservationCount,ComparisonRule,WAPEInterpretation
0,PRODUCT_DAY,LOCKED_FOR_MODEL_SELECTION,MOVING_AVERAGE_5,MOVING_AVERAGE,MAE,0.643425,2.439141,40.268572,-0.104685,-531.8,5080,ADVANCED_MODEL_MUST_TARGET_LOWER_MAE,ERROR_PERCENTAGE_NOT_ACCURACY
1,AGGREGATE_DAILY_BOTTOM_UP,LOCKED_FOR_MODEL_SELECTION,NAIVE_5_OPERATING_DAYS,OPERATING_DAY_NAIVE,MAE,42.500000,54.338752,20.943698,-20.950000,-838.0,40,ADVANCED_MODEL_MUST_TARGET_LOWER_MAE,ERROR_PERCENTAGE_NOT_ACCURACY



Baseline selection evidence:


,EvaluationLevel,SelectionCriterion,SelectedBaseline,SelectedMetricValue,SupportingMetric,SupportingMetricValue
0,PRODUCT_DAY_POOLED,LOWEST_COMBINED_MAE,MOVING_AVERAGE_5,0.643425,RMSE,2.439141
1,PRODUCT_WINDOW_CONTEXT,LOWEST_MEAN_CONTEXT_MAE,MOVING_AVERAGE_5,0.643425,MEAN_POSITIVE_CONTEXT_MAE,3.714318
2,PRODUCT_COMBINED_TWO_FOLDS,LOWEST_MEAN_PRODUCT_MAE,MOVING_AVERAGE_5,0.643425,MEAN_POSITIVE_PRODUCT_MAE,3.714318
3,AGGREGATE_DAILY_BOTTOM_UP,LOWEST_COMBINED_DAILY_MAE,NAIVE_5_OPERATING_DAYS,42.500000,RMSE,54.338752



Demand-pattern baseline winners:


...


Step 2 part-validation register:


,Part,ValidationFile,CheckCount,PassedCheckCount,FailedCheckCount,PartPassed
0,PART_1,01_baseline_framework_validation_summary.csv,14,14,0,True
1,PART_2,02_baseline_performance_validation_summary.csv,17,17,0,True
2,PART_3,03_baseline_diagnostic_validation_summary.csv,25,25,0,True
3,PART_4,04_baseline_aggregate_validation_summary.csv,23,23,0,True



Final Step 2 validation summary:


,Check,Expected,Actual,Passed
0,Completed Step 2 parts,4,4,True
1,Failed Step 2 parts,0,0,True
2,Expected saved baseline files,29,29,True
3,Missing saved baseline files,0,0,True
4,Empty saved baseline files,0,0,True
5,Registered baseline methods,7,7,True
6,Model-selection validation rows,5080,5080,True
7,Baseline prediction rows,35560,35560,True
8,Validation windows,2,2,True
9,Validation products,127,127,True



Saved outputs:
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/05_baseline_output_file_audit.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/05_baseline_part_validation_register.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/05_baseline_selection_evidence.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/05_locked_baseline_benchmark_contract.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/05_demand_pattern_baseline_winners.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/05_modelling_step2_final_validation_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/05_modelling_step2_completion_summary.csv
/Users/ryansmac/Desktop/Meng Project/eden_datasets/modelling/02_baselines/MODELLING_STEP2_HANDOFF.md
